# LLMFA Final v4: fixed parser + Llama EXTRACTOR + bank EBIT + frozen v4 integration rule + override

**Goal.** Raise accuracy over Hybrid A (v2: 90.5% tolerance / 94.5% within 1%, 178 exact on 199 cells) and over the v3 run (94.5% tolerance, 93.5% exact; excluding HDFC 98.4%). Aims (never tuned to): all 199 cells exact >= 95%, tolerance >= 97%; excluding HDFC exact >= 98%. These are aims, not guarantees; the notebook reports PASS/FAIL honestly and falls back to Hybrid A or v3 as the recommended final if v4 is lower. The ground-truth key is used ONLY to score, never to choose a page, unit, value, row or source. Runs top to bottom in Colab with no input. Scoring thresholds are unchanged. The 16 cells a company does not have (HDFC current liabilities and inventories, Infosys and Paytm inventories) stay unscored, as before.

**Part I** verbatim except `OUT_DIR` = `results_final_v4`. **Part II** verbatim except the changes below; loops are wrapped in functions so the OLD parser can be re-run as the Hybrid A reference.

**Parser fixes carried over from v3**
- **A1 value-column rule.** K = number of value columns per statement (modal count of trailing numeric tokens on `total ...` rows, else distinct header dates, else 2). Current year = FIRST of the LAST K numeric tokens; leading note numbers and page references are ignored.
- **A2 merged lines.** numbers, then words, then numbers: only the FIRST numeric group is parsed. Such rows are flagged.
- **A3** before/after counts of changed cells (no scoring) and the identity check re-run (cell 8c).

**New in v4 (all made AFTER seeing the v3 results, so dev/test is NOT a clean held-out split)**
1. **Footnote marker after the value columns.** A trailing token like `01` after the K value columns is dropped (`Profit before tax 383,985 126,790 01` read the PREVIOUS year; this also made Llama's correct answer fail validation).
2. **PBT row choice.** With several PBT-like rows: exact `profit before tax` first, then a row that starts with it (`Profit before tax expense`), else the LAST candidate, never the first. Hypothesis for the UltraTech near cells (the first candidate was probably a pre-exceptional/pre-associates line); the P matched label is printed in the non-agreeing table so you can check it.
3. **Bank P&L.** The page finder accepts a bank layout (`interest earned` + `interest expended` + `operating expenses`) which has no `profit before tax` row. Bank EBIT = Total income - Interest expended - Operating expenses (PDF route and Llama, which gets its own bank prompt). Bank detection comes from the statement itself, never from a company name.
4. **Total equity completeness (Python, key-free).** If a `Total equity` row is among Llama's rows keep ONLY it (no minority-interest double count); bank layout: if `Reserves and surplus` is listed but the `Capital` row above it is not, add that row. Every row is still validated against the raw line.
5. **Rule 3.** Only one of P, L exists and O differs by 1% to 10%: take O (as before). If O differs by MORE than 10% it is implausible: keep the document value and put the cell on the REVIEW list.
6. **HDFC checks (Part IV).** Page headings and basis for BS and P&L, the reason a P&L page is not saved (page finder verdict per candidate page), equity components including `Employees stock options` (printed, NOT added) and the scoring-only gap, so you can decide whether your key includes that line.

**Llama extractor (Part L).** `LLM_MODEL` (config cell): `NousResearch/Meta-Llama-3.1-8B-Instruct` in 4-bit NF4, fallbacks Hermes-2-Pro-Llama-3-8B then Llama-2-7b-chat; optional `HF_TOKEN` from Colab secrets; greedy decoding; answer prefilled with `{`. One call per statement per report on the saved numbered RAW rows with worked examples. Python validates every number (token of the cited row, current-year column, sign from brackets), applies the unit from the PDF-route detector and does all arithmetic. Checkpointed to Drive per model; rerun resumes. Old v3 checkpoints are reused for statements whose prompt did not change; bank P&L statements get a new prompt and a new checkpoint.

**Frozen v4 integration rule** (JSON with SHA-256 written before any v4 result; Part III verifies it). P = fixed PDF route, L = validated Llama extraction, O = override. (1) P and L agree within 0.1%: use P. (2) They disagree: if O supports exactly one within 1%, use it; otherwise use L when P came from a flagged row, else keep P and add the cell to REVIEW. (3) Only one of P, L: accept it if O is missing or within 1%; O differs 1-10%: take O; more than 10%: keep the document value and REVIEW. (4) Only O: O. None: not covered. Class of the final value = class of the method used.

**Assumptions.** (a) "O supports neither" also covers O supports both or O missing. (b) "Parser flagged" = multi-candidate, column ambiguity or merged-line split. (c) Prompt budget 8k tokens (4k for Llama-2); longer statements are shortened and flagged. (d) Dash-only values are not accepted from Llama. (e) Employees stock options are NOT added to bank equity (shown only as a diagnostic).

**Honest caveat.** Several v4 changes are hypotheses drawn from the v3 outputs (UltraTech PBT row, HDFC P&L page, bank EBIT layout). They were tested only on mock data. If a change does not help on your reports, Part IV says so.


In [ ]:
# ===== v3 CONFIG (the only place to change the model) =====
LLM_MODEL     = "NousResearch/Meta-Llama-3.1-8B-Instruct"                                   # tried first, 4-bit NF4
LLM_FALLBACKS = ["NousResearch/Hermes-2-Pro-Llama-3-8B", "NousResearch/Llama-2-7b-chat-hf"]  # tried in order if the load fails
# Optional: add HF_TOKEN in Colab (left key icon > Secrets) for gated repos; Part L1 reads it automatically.


# PART I - Override pipeline (cells 1-15 of the v1.1 notebook, verbatim except OUT_DIR)

In [ ]:
# ===== 1. Setup =====
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance", "requests", "beautifulsoup4", "lxml", "pandas", "openpyxl"], check=False)
import re, io, json, time, math, datetime, platform, itertools, hashlib
from decimal import Decimal, InvalidOperation
from pathlib import Path
from urllib.parse import urljoin
import xml.etree.ElementTree as ET
import requests
import pandas as pd
import yfinance as yf
from bs4 import BeautifulSoup
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("(not in Colab, or Drive already mounted)")
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 500)
print("pandas", pd.__version__, "| yfinance", yf.__version__, "| requests", requests.__version__)

In [ ]:
# ===== 2. Configuration. The selection rules below are FROZEN before any ground-truth value is read. =====
BASE_DIR = Path("/content/drive/MyDrive/LLMFA/Prototype")
GT_DIR   = BASE_DIR / "Ground_Truth"
CORE_DIR = BASE_DIR / "core_comparison"
OUT_DIR  = CORE_DIR / "results_final_v4"             # new results folder: the first run's files in core_comparison/ stay untouched
RAW_DIR  = CORE_DIR / "raw"                      # every download is cached here (the first run's downloads are reused, so this is fast)
for d in (OUT_DIR, RAW_DIR):
    d.mkdir(parents=True, exist_ok=True)

SOURCE_ORDER = ["yfinance", "screener", "nse"]  # fetch order
COMPANIES = {   # company key (as in the GT file names) -> identifiers. CONFIRM in a browser if unsure.
    "AIRTEL":  {"name": "Bharti Airtel",        "yf": ["BHARTIARTL.NS"],           "nse": "BHARTIARTL", "screener": ["BHARTIARTL"]},
    "HDFC":    {"name": "HDFC Bank",            "yf": ["HDFCBANK.NS"],             "nse": "HDFCBANK",   "screener": ["HDFCBANK"]},
    "INFOSYS": {"name": "Infosys",              "yf": ["INFY.NS"],                 "nse": "INFY",       "screener": ["INFY"]},
    "MSIL":    {"name": "Maruti Suzuki India",  "yf": ["MARUTI.NS"],               "nse": "MARUTI",     "screener": ["MARUTI"]},
    "PAYTM":   {"name": "One 97 Communications","yf": ["PAYTM.NS"],                "nse": "PAYTM",      "screener": ["PAYTM"]},
    "RIL":     {"name": "Reliance Industries",  "yf": ["RELIANCE.NS"],             "nse": "RELIANCE",   "screener": ["RELIANCE"]},
    "SPIL":    {"name": "Sun Pharma",           "yf": ["SUNPHARMA.NS"],            "nse": "SUNPHARMA",  "screener": ["SUNPHARMA"]},
    "SWIGGY":  {"name": "Swiggy",               "yf": ["SWIGGY.NS"],               "nse": "SWIGGY",     "screener": ["SWIGGY"]},
    "TCS":     {"name": "Tata Consultancy",     "yf": ["TCS.NS"],                  "nse": "TCS",        "screener": ["TCS"]},
    "TITAN":   {"name": "Titan Company",        "yf": ["TITAN.NS"],                "nse": "TITAN",      "screener": ["TITAN"]},
    "TS":      {"name": "Tata Steel",           "yf": ["TATASTEEL.NS"],            "nse": "TATASTEEL",  "screener": ["TATASTEEL"]},
    "UTCL":    {"name": "UltraTech Cement",     "yf": ["ULTRACEMCO.NS"],           "nse": "ULTRACEMCO", "screener": ["ULTRACEMCO"]},
    "ZOMATO":  {"name": "Eternal (Zomato)",     "yf": ["ETERNAL.NS", "ZOMATO.NS"], "nse": "ETERNAL",    "screener": ["ETERNAL", "ZOMATO"]},
}
FIELDS = ["total_assets", "total_equity", "current_liabilities", "inventories", "ebit"]      # gross_profit excluded (all N/A in GT)
FIELD_STMT = {"total_assets": "balance_sheet", "total_equity": "balance_sheet", "current_liabilities": "balance_sheet",
              "inventories": "balance_sheet", "ebit": "income_stmt"}
BASIS_ORDER = ["consolidated", "standalone"]    # statements preferred when a source offers both
REPORT_YEAR_IS_FY_END_YEAR = True               # report year 2025 = financial year ending March 2025
FY_END_MONTH = 3
FORCE_REFRESH = False                           # True = ignore cached downloads
REQUEST_DELAY_SEC = 1.0
MAX_RETRIES = 1
TIMEOUT = 12
BREAKER = 3                                     # skip a source after this many companies in a row return nothing
UNIT_WORDS = (("crore", 10**7), ("lakh", 10**5), ("lac", 10**5), ("billion", 10**9), ("million", 10**6), ("thousand", 10**3))
CRORE = Decimal(10**7)

# row label per source. primary = None -> the source has no such line. 'alts' are only used to EXPLAIN errors, never to select.
# 'Total Equity (' / 'EBIT (' names are rows this notebook builds from the source's own rows (the build is recorded per source).
_X = {"total_assets": {"primary": "Assets", "alts": []}, "inventories": {"primary": "Inventories", "alts": []},
      "current_liabilities": {"primary": "CurrentLiabilities", "alts": []},
      "total_equity": {"primary": "Equity", "alts": ["EquityAttributableToOwnersOfParent"]},
      "ebit": {"primary": "EBIT (ProfitBeforeTax + FinanceCosts)", "alts": ["ProfitBeforeTax"]}}
LABELS_BY_SOURCE = {
    "yfinance": {"total_assets": {"primary": "Total Assets", "alts": []}, "inventories": {"primary": "Inventory", "alts": []},
                 "current_liabilities": {"primary": "Current Liabilities", "alts": []},
                 "total_equity": {"primary": "Total Equity Gross Minority Interest", "alts": ["Stockholders Equity", "Common Stock Equity"]},
                 "ebit": {"primary": "EBIT", "alts": ["Operating Income"]}},
    "screener": {"total_assets": {"primary": "Total Assets", "alts": []}, "inventories": {"primary": "Inventories", "alts": []},
                 "current_liabilities": {"primary": None, "alts": []},       # Screener has no current-liabilities line
                 "total_equity": {"primary": "Total Equity (Equity Capital + Reserves)", "alts": ["Reserves"]},
                 "ebit": {"primary": "EBIT (Profit before tax + Interest)", "alts": ["Profit before tax"]}},
    "nse": _X,
}
ALIASES = {   # tag / row aliases used when building derived rows (first alias found wins; then a pattern search is tried for NSE)
    "screener": {"pbt": ["Profit before tax"], "fin": ["Interest"], "eq_cap": ["Equity Capital"], "reserves": ["Reserves"]},
    "xbrl": {"pbt": ["ProfitBeforeTax", "ProfitLossBeforeTax", "ProfitBeforeTaxFromContinuingOperations", "ProfitLossBeforeTaxFromContinuingOperations"],
             "fin": ["FinanceCosts", "FinanceCost", "InterestExpense", "InterestExpenses", "BorrowingCosts"]},
}
NSE_PERIODS = ("Annual", "Quarterly")           # listing types queried; Quarterly is used only for the March (Q4) filing, which carries full-year figures

# ---------- FROZEN SELECTION RULES (fixed before evaluation; ground truth is never consulted to choose a value) ----------
ANCHOR = {"default": "yfinance", "ebit": "screener"}                                   # preferred source per field type
FALLBACK_PRIORITY = {"default": ["yfinance", "nse", "screener"], "ebit": ["screener", "nse", "yfinance"]}   # used when the anchor has no value
# EBIT fallback reasoning (a definition argument, not a measured result): NSE EBIT is built as pre-tax profit + finance costs, the same definition as the dataset's EBIT.
TOL_PCT  = Decimal("0.1")      # within_tolerance: gap <= 0.1% ...
NEAR_PCT = Decimal("1.0")      # near: <= 1%; mismatch: more than 1%
# ... OR gap <= rounding floor = 0.5 x (precision of source A) + 0.5 x (precision of source B), in crore. For the ground-truth comparison:
# 0.5 x precision of the selected source + 0.5 x the ground-truth unit. Source precision is MEASURED from the data (cell 10), not assumed.
PRECISION_OVERRIDE = {}        # e.g. {"yfinance": Decimal(1)} to force a precision instead of the measured one
MATERIAL = ("near", "mismatch")                       # pair classes that count as a material disagreement
PAIRS = [("yfinance", "screener"), ("yfinance", "nse"), ("screener", "nse")]

In [ ]:
# ===== 3. Helpers: text/number parsing, HTTP + disk cache, table cleaning, derived rows =====
def norm(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def to_dec(v):
    if v is None or isinstance(v, bool):
        return None
    t = re.sub(r"(?i)(\u20b9|rs\.?|inr)", "", str(v).strip().replace("\u2212", "-"))
    t = re.sub(r"\s+", "", t).replace(",", "")
    if re.fullmatch(r"\(.*\)", t):
        t = "-" + t[1:-1]
    try:
        d = Decimal(t)
    except InvalidOperation:
        return None
    return d if d.is_finite() else None

def unit_scale(txt):
    t = str(txt or "").lower()
    for word, val in UNIT_WORDS:
        if word in t:
            return Decimal(val)
    return None

UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
      "Accept-Language": "en-US,en;q=0.9"}

def http_get(sess, url, params=None, headers=None, kind="text"):
    """returns (payload, error). kind = text | json | bytes. Never raises."""
    err = ""
    for attempt in range(MAX_RETRIES + 1):
        try:
            r = sess.get(url, params=params, headers={**UA, **(headers or {})}, timeout=TIMEOUT)
            if r.status_code == 200:
                if kind == "bytes":
                    return r.content, ""
                if kind == "json":
                    return r.json(), ""
                if re.search(r"Access Denied|Request unsuccessful", r.text[:3000], re.I):
                    err = "blocked page (Access Denied)"
                else:
                    return r.text, ""
            else:
                err = f"HTTP {r.status_code}"
                if r.status_code == 404:
                    break
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:80]}"
        time.sleep(REQUEST_DELAY_SEC * (attempt + 1))
    return None, err

def cached(path, fetch, kind="text"):
    """disk cache. If you saved a page / file by hand under the same name, it is used instead of downloading."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0 and not FORCE_REFRESH:
        try:
            if kind == "bytes":
                return path.read_bytes(), ""
            txt = path.read_text(encoding="utf-8")
            return (json.loads(txt) if kind == "json" else txt), ""
        except Exception:
            pass
    payload, err = fetch()
    if payload is None or (kind == "text" and not str(payload).strip()):
        return None, err or "empty response"
    if kind == "bytes":
        path.write_bytes(payload)
    elif kind == "json":
        path.write_text(json.dumps(payload), encoding="utf-8")
    else:
        path.write_text(payload, encoding="utf-8")
    time.sleep(REQUEST_DELAY_SEC)
    return payload, ""

def new_result(source, company):
    return {"source": source, "company": company, "status": "failed", "error": "", "note": "",
            "balance_sheet": None, "income_stmt": None, "basis_by_year": {}}

def clean_label(s):
    s = str(s).replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return re.sub(r"\s*\+\s*$", "", s).strip()

def parse_period(s):
    s = re.sub(r"['\u2019]", "", str(s)).strip()
    for fmt in ("%b %Y", "%b %y", "%B %Y", "%B %y"):
        try:
            return pd.Timestamp(datetime.datetime.strptime(s, fmt))
        except ValueError:
            pass
    return None

def frame_from_dict(data):
    """{label: {Timestamp: Decimal|None}} -> DataFrame (rows = labels, columns = period starts, sorted)."""
    clean = {lab: {p: v for p, v in cols.items() if p is not None} for lab, cols in data.items()}
    df = pd.DataFrame.from_dict(clean, orient="index")
    if df.shape[1]:
        df = df.reindex(sorted(df.columns), axis=1)
    df.index.name = "label"
    return df.astype(object).where(df.notna(), None)

def find_row(df, label):
    key = norm(label)
    for idx in df.index:
        if norm(idx) == key:
            return df.loc[idx]
    return None

def first_row(df, aliases):
    for a in ([aliases] if isinstance(aliases, str) else aliases):
        r = find_row(df, a)
        if r is not None:
            return a, r
    return None, None

def pick_column(series, year, month=FY_END_MONTH):
    y = year if REPORT_YEAR_IS_FY_END_YEAR else year + 1
    for col in series.index:
        if col.year == y and col.month == month:
            return col
    return None

def as_decimal(x):
    if x is None:
        return None
    if isinstance(x, Decimal):
        return x if x.is_finite() else None
    try:
        if pd.isna(x):
            return None
        return Decimal(int(x)) if float(x).is_integer() else Decimal(repr(float(x)))
    except Exception:
        return None

def add_derived(df, name, required, optional=()):
    """Adds one row = sum(sign * row) per column. required / optional = lists of (aliases, sign).
    A column needs every required row; a missing optional row counts as 0. Nothing is added if a required row does not exist."""
    if df is None or df.empty:
        return df
    req = [(first_row(df, al)[1], s) for al, s in required]
    if any(r is None for r, _ in req):
        return df
    opt = [(first_row(df, al)[1], s) for al, s in optional]
    vals = {}
    for col in df.columns:
        tot, ok = Decimal(0), True
        for r, s in req:
            v = as_decimal(r[col])
            if v is None:
                ok = False; break
            tot += s * v
        if not ok:
            continue
        for r, s in opt:
            v = None if r is None else as_decimal(r[col])
            if v is not None:
                tot += s * v
        vals[col] = tot
    if vals:
        df.loc[name] = pd.Series(vals, dtype=object)
    return df

def merge_by_basis(frames, order):
    """frames: {basis: DataFrame}. For every year column take the first basis in `order` that has data for it."""
    cols = {}
    for basis in order:
        df = frames.get(basis)
        if df is None or df.empty:
            continue
        for c in df.columns:
            if c not in cols and df[c].notna().any():
                cols[c] = (basis, df[c])
    if not cols:
        return None, {}
    out = pd.concat({c: s for c, (b, s) in cols.items()}, axis=1)
    out = out.reindex(sorted(out.columns), axis=1)
    out.index.name = "label"
    return out, {c: b for c, (b, _) in cols.items()}

def finish(res, frames_bs, frames_is, order, extra_note=""):
    bs, bmap = merge_by_basis(frames_bs, order)
    inc, _ = merge_by_basis(frames_is, order)
    res["balance_sheet"], res["income_stmt"], res["basis_by_year"] = bs, inc, bmap
    if bs is not None and inc is not None:
        res["status"] = "ok"
    elif bs is not None or inc is not None:
        res["status"] = "partial"
    if extra_note:
        res["note"] = (res["note"] + " | " + extra_note).strip(" |")
    if res["status"] == "ok" and res["error"]:              # something minor failed (e.g. the standalone page) but data was found
        res["note"] = (res["note"] + " | minor: " + res["error"]).strip(" |")
        res["error"] = ""
    return res
print("helpers ready")

def discover(df, patterns, exclude=()):
    """first row label whose normalised name matches a pattern (and none of the exclusions)"""
    if df is None:
        return None
    for idx in df.index:
        n = norm(idx)
        if any(re.search(p, n) for p in patterns) and not any(re.search(x, n) for x in exclude):
            return idx
    return None

In [ ]:
# ===== 4. Report index from the ground-truth FOLDER (file names only: the ANSWERS are NOT read until the final evaluation) =====
print("GT_DIR:", GT_DIR, "| exists:", Path(GT_DIR).exists())
_jsons = sorted(Path(GT_DIR).rglob("*.json")) if Path(GT_DIR).exists() else []
print("JSON files found under GT_DIR:", len(_jsons))
if not _jsons:
    _par = Path(GT_DIR).parent
    print("Parent folder:", _par, "| exists:", _par.exists())
    if _par.exists():
        print("Contents of the parent folder:", sorted(p.name for p in _par.iterdir())[:40])
    raise FileNotFoundError(
        f"No ground-truth .json files under {GT_DIR}. Check that Drive is mounted (the setup cell must have run in THIS session; "
        "a runtime restart after pip install unmounts it) and that the folder name in the setup cell still matches Drive.")

COMPANY_KEYS = {norm(c): c for c in COMPANIES}
REPORTS, unresolved = {}, []
for p in _jsons:
    years = sorted(set(re.findall(r"(?<!\d)(20\d{2})(?!\d)", p.stem)))
    if len(years) != 1:
        try:
            m = re.search(r"(20\d{2})", str(json.loads(p.read_text(encoding="utf-8")).get("report_year", "")))   # only the year is read
            years = [m.group(1)] if m else years
        except Exception as e:
            unresolved.append((p.name, f"unreadable JSON: {e}")); continue
    toks = [norm(t) for t in re.split(r"[^A-Za-z0-9]+", re.sub(r"(20\d{2})", " ", p.stem)) if t]
    comp = next((COMPANY_KEYS[t] for t in toks if t in COMPANY_KEYS), None)
    if comp is None or len(years) != 1:
        unresolved.append((p.name, f"company={comp}, years={years}")); continue
    rid = f"{comp}_{years[0]}"
    if rid in REPORTS:
        unresolved.append((p.name, f"duplicate of {rid}")); continue
    REPORTS[rid] = {"company": comp, "year": int(years[0]), "path": str(p)}
print(f"{len(REPORTS)} reports found in the ground-truth folder; unresolved files: {len(unresolved)}")
for u in unresolved:
    print("   UNRESOLVED:", u)
if len(REPORTS) == 0:
    raise RuntimeError("0 reports resolved: file names did not match COMPANIES/years (see UNRESOLVED above).")
CELLS = [{"report_id": rid, "company": r["company"], "year": r["year"], "field": f} for rid, r in REPORTS.items() for f in FIELDS]
COMPS = sorted({r["company"] for r in REPORTS.values()})
YEARS_BY_COMP = {c: sorted({r["year"] for r in REPORTS.values() if r["company"] == c}) for c in COMPS}
print(f"{len(CELLS)} cells to extract ({len(REPORTS)} reports x {len(FIELDS)} fields); companies: {COMPS}")


In [ ]:
# ===== 5. Fetcher 1 of 3: yfinance (statements cached on disk) =====
def frame_to_disk(df, path):
    out = df.copy(); out.columns = [str(c)[:10] for c in out.columns]; out.index.name = "label"
    out.to_csv(path)

def frame_from_disk(path):
    df = pd.read_csv(path, index_col="label"); df.columns = pd.to_datetime(df.columns); return df

def yf_download(company):
    res = new_result("yfinance", company)
    res.update(ticker=None, currency=None, currency_source=None)
    for tk in COMPANIES[company]["yf"]:
        for attempt in range(MAX_RETRIES + 1):
            try:
                t = yf.Ticker(tk)
                bs, inc = t.balance_sheet, t.income_stmt
                if bs is None or bs.empty:
                    raise ValueError("balance_sheet empty")
                if inc is None or inc.empty:
                    raise ValueError("income_stmt empty")
                cur = None
                try:
                    cur = (t.info or {}).get("financialCurrency")
                except Exception:
                    pass
                res.update(ticker=tk, balance_sheet=bs, income_stmt=inc, currency=cur, status="ok", error="",
                           currency_source="yfinance info.financialCurrency" if cur else "not reported")
                return res
            except Exception as e:
                res["error"] = f"{tk}: {type(e).__name__}: {str(e)[:100]}"
                time.sleep(REQUEST_DELAY_SEC * (attempt + 1))
    return res

def fetch_yfinance(company):
    bs_p, in_p, meta_p = RAW_DIR / "yfinance" / f"{company}_balance_sheet.csv", RAW_DIR / "yfinance" / f"{company}_income_stmt.csv", RAW_DIR / "yfinance" / f"{company}_meta.json"
    bs_p.parent.mkdir(parents=True, exist_ok=True)
    if not FORCE_REFRESH and bs_p.exists() and in_p.exists() and meta_p.exists():
        meta = json.loads(meta_p.read_text())
        res = new_result("yfinance", company)
        res.update(ticker=meta.get("ticker"), currency=meta.get("currency"), currency_source=meta.get("currency_source"),
                   balance_sheet=frame_from_disk(bs_p), income_stmt=frame_from_disk(in_p), status="ok", note="cached")
        return res
    res = yf_download(company)
    if res["status"] == "ok":
        frame_to_disk(res["balance_sheet"], bs_p); frame_to_disk(res["income_stmt"], in_p)
        meta_p.write_text(json.dumps({k: res[k] for k in ("company", "ticker", "currency", "currency_source")}))
    time.sleep(REQUEST_DELAY_SEC)
    return res
print("yfinance fetcher ready")

In [ ]:
# ===== 6. Fetcher 2 of 3: Screener.in (HTML pages, values in INR crore) =====
# ---------- Screener.in ----------
def screener_table(soup, section_id):
    sec = soup.find("section", id=section_id)
    if sec is None:
        return None
    return sec.find("table", class_="data-table") or sec.find("table")

def screener_rows(tbl):
    if tbl is None or tbl.find("thead") is None or tbl.find("tbody") is None:
        return {}
    heads = [th.get_text(" ", strip=True) for th in tbl.find("thead").find_all("th")]
    data = {}
    for tr in tbl.find("tbody").find_all("tr"):
        tds = tr.find_all("td")
        if not tds:
            continue
        lab = clean_label(tds[0].get_text(" ", strip=True))
        if lab:
            data[lab] = {parse_period(h): to_dec(td.get_text(" ", strip=True)) for h, td in zip(heads[1:], tds[1:])}
    return data

def fetch_screener(company):
    res = new_result("screener", company)
    sess = requests.Session()
    frames_bs, frames_is, notes, errs = {}, {}, [], []
    done_actual = set()
    for slug in COMPANIES[company]["screener"]:
        for basis in BASIS_ORDER:
            url = f"https://www.screener.in/company/{slug}/" + ("consolidated/" if basis == "consolidated" else "")
            html, err = cached(RAW_DIR / "screener" / f"{company}_{slug}_{basis}.html", lambda u=url: http_get(sess, u))
            if html is None:
                errs.append(f"{slug}/{basis}: {err}"); continue
            soup = BeautifulSoup(html, "lxml")
            text = soup.get_text(" ", strip=True)
            actual = "consolidated" if "Consolidated Figures" in text else ("standalone" if "Standalone Figures" in text else basis)
            if actual in done_actual:
                continue                                   # the site served the same statements again
            bs_data = screener_rows(screener_table(soup, "balance-sheet"))
            is_data = screener_rows(screener_table(soup, "profit-loss"))
            if not bs_data and not is_data:
                errs.append(f"{slug}/{basis}: no balance-sheet / profit-loss table on the page"); continue
            done_actual.add(actual)
            # expandable rows (Inventories sits under 'Other Assets') come from a second request
            info = soup.find(id="company-info")
            cid = info.get("data-company-id") if info is not None else None
            if cid:
                for parent in ("Other Assets", "Other Liabilities"):
                    params = {"parent": parent, "section": "balance-sheet"}
                    if actual == "consolidated":
                        params["consolidated"] = ""
                    sch, serr = cached(RAW_DIR / "screener" / f"{company}_{slug}_{actual}_{norm(parent)}.json",
                                       lambda p=params, c=cid, u=url: http_get(sess, f"https://www.screener.in/api/company/{c}/schedules/",
                                                                               params=p, headers={"Referer": u, "X-Requested-With": "XMLHttpRequest"}, kind="json"),
                                       kind="json")
                    if not isinstance(sch, dict):
                        notes.append(f"{actual} '{parent}' detail not loaded ({serr})"); continue
                    for lab, vals in sch.items():
                        lab = clean_label(lab)
                        if lab and lab not in bs_data and isinstance(vals, dict):
                            bs_data[lab] = {parse_period(k): to_dec(v) for k, v in vals.items()}
            else:
                notes.append(f"{actual}: company id not found on page (no Inventories)")
            bs, inc = frame_from_dict(bs_data), frame_from_dict(is_data)
            A = ALIASES["screener"]
            bs = add_derived(bs, LABELS_BY_SOURCE["screener"]["total_equity"]["primary"], [(A["eq_cap"], 1), (A["reserves"], 1)])
            inc = add_derived(inc, LABELS_BY_SOURCE["screener"]["ebit"]["primary"], [(A["pbt"], 1), (A["fin"], 1)])
            frames_bs[actual], frames_is[actual] = bs, inc
        if frames_bs or frames_is:
            break                                          # first slug that works
    res["error"] = "; ".join(errs)[:300]
    return finish(res, frames_bs, frames_is, BASIS_ORDER, "; ".join(notes))

print("Screener fetcher ready")

In [ ]:
# ===== 7. Fetcher 3 of 3: NSE India (XBRL result filings, values converted to INR crore) =====
# Both exchanges publish every company's results as an XBRL file. The files are saved under raw_web/<source>/<company>/.
# You can also drop XBRL .xml files you downloaded by hand into that folder: they are read too.
def _ln(tag):
    return tag.rsplit("}", 1)[-1] if isinstance(tag, str) else ""

def parse_xbrl(raw):
    root = ET.fromstring(raw)
    ctx = {}
    for el in root.iter():
        if _ln(el.tag) != "context":
            continue
        info = {"start": None, "end": None, "inst": None, "dim": False}
        for ch in el.iter():
            n, t = _ln(ch.tag), (ch.text or "").strip()
            if n == "startDate": info["start"] = t
            elif n == "endDate": info["end"] = t
            elif n == "instant": info["inst"] = t
            elif n in ("segment", "scenario") and len(list(ch)) > 0: info["dim"] = True
        ctx[el.get("id")] = info
    out = {"basis": None, "end": None, "rounding": None, "facts": {}, "decimals": {}}
    for el in root.iter():
        name, cref, text = _ln(el.tag), el.get("contextRef"), (el.text or "").strip()
        if cref is None or not text:
            continue
        if name == "NatureOfReportStandaloneConsolidated":
            low = text.lower()
            out["basis"] = "standalone" if ("standalone" in low or "non" in low) else ("consolidated" if "consolidated" in low else None)
            continue
        if name == "DateOfEndOfReportingPeriod":
            out["end"] = text; continue
        if name == "LevelOfRoundingUsedInFinancialStatements":
            out["rounding"] = text; continue
        c = ctx.get(cref)
        if c is None or c["dim"]:
            continue
        v = to_dec(text)
        if v is None:
            continue
        try:
            if c["inst"]:
                period = pd.Timestamp(c["inst"])
            elif c["start"] and c["end"]:
                if not 330 <= (pd.Timestamp(c["end"]) - pd.Timestamp(c["start"])).days <= 400:
                    continue                                 # quarters / half years are not annual figures
                period = pd.Timestamp(c["end"])
            else:
                continue
        except Exception:
            continue
        out["facts"].setdefault(name, {}).setdefault(period, v)
        out["decimals"].setdefault(name, el.get("decimals"))
    return out

def xbrl_scale(parsed):
    """rupees per unit of the numbers in this file, and why"""
    dec = parsed["decimals"].get("Assets") or next(iter(parsed["decimals"].values()), None)
    try:
        d = int(dec)
    except (TypeError, ValueError):
        d = None
    if d is not None and d <= -3:
        return Decimal(1), f"absolute rupees (decimals={d})"
    sc = unit_scale(parsed["rounding"])
    if sc:
        return sc, f"from rounding tag '{parsed['rounding']}'"
    return Decimal(1), "ASSUMED absolute rupees (no usable scale info)"


def build_ebit(df, source_key):
    """adds the EBIT row = pre-tax profit + finance costs. Returns (df, text saying how it was built or why it was not)."""
    A = ALIASES["xbrl"]
    pbt = first_row(df, A["pbt"])[0] or discover(df, [r"^profit(loss)?beforetax"], exclude=[r"exceptional", r"discontinu"])
    fin = first_row(df, A["fin"])[0] or discover(df, [r"^(finance|interest)(costs?|expenses?)$", r"^financecost"])
    if pbt and fin:
        df = add_derived(df, LABELS_BY_SOURCE[source_key]["ebit"]["primary"], [([pbt], 1), ([fin], 1)])
        built = find_row(df, LABELS_BY_SOURCE[source_key]["ebit"]["primary"]) is not None
        return df, (f"EBIT = {pbt} + {fin}" if built else f"EBIT not built: {pbt} / {fin} found but never both in one annual period")
    cands = [str(i) for i in df.index if re.search(r"profit|tax|financ|interest", str(i), re.I)][:12]
    return df, f"EBIT not built: missing {'pre-tax profit tag' if not pbt else ''}{' and ' if (not pbt and not fin) else ''}{'finance-cost tag' if not fin else ''}; similar tags: {cands}"

def xbrl_company(folder, source):
    """read every .xml in folder -> (frames_by_basis, scale reasons, unreadable count, ebit notes)"""
    cand, reasons, bad = {}, set(), 0                       # cand[basis][label][period] = (rank, crore value)
    for p in sorted(Path(folder).glob("*.xml")):
        try:
            P = parse_xbrl(p.read_bytes())
        except Exception:
            bad += 1; continue
        if not P["facts"]:
            continue
        sc, why = xbrl_scale(P); reasons.add(why)
        basis = P["basis"] or "unknown"
        try:
            own = pd.Timestamp(P["end"])
        except Exception:
            own = max((pp for f in P["facts"].values() for pp in f), default=None)
        for label, per in P["facts"].items():
            for period, v in per.items():
                rank = 0 if period == own else 1               # a filing's own year beats a prior-year comparative
                slot = cand.setdefault(basis, {}).setdefault(label, {})
                if period not in slot or rank < slot[period][0]:
                    slot[period] = (rank, v * sc / CRORE)
    frames, ebit_notes = {}, []
    for basis, labels in cand.items():
        df = frame_from_dict({lab: {pp: val for pp, (_, val) in per.items()} for lab, per in labels.items()})
        df, how = build_ebit(df, source)
        frames[basis] = df; ebit_notes.append(f"{basis}: {how}")
    return frames, sorted(reasons), bad, ebit_notes

def _xml_link(row, base):
    for v in row.values():
        if isinstance(v, str) and re.search(r"\.xml\b", v, re.I):
            v = v.strip()
            return v if v.lower().startswith("http") else urljoin(base, v)
    return None

def _is_q4(row):
    return any(isinstance(v, str) and re.search(r"(?i)(31[-/ ]?(mar|03)[-/ ]?20\d\d|20\d\d-03-31)", v) for v in row.values())

NSE_HOME = "https://www.nseindia.com"
def nse_session():
    s = requests.Session()
    s.headers.update({**UA, "Accept": "application/json, text/plain, */*", "Referer": NSE_HOME + "/companies-listing/corporate-filings-financial-results"})
    for u in (NSE_HOME, NSE_HOME + "/companies-listing/corporate-filings-financial-results"):
        try:
            s.get(u, timeout=TIMEOUT)
        except Exception:
            pass
    return s

def fetch_nse(company):
    res = new_result("nse", company)
    sym = COMPANIES[company]["nse"]
    folder = RAW_DIR / "nse" / company
    folder.mkdir(parents=True, exist_ok=True)
    sess = nse_session()
    links, errs, nrows, keys, bad_calls = {}, [], 0, [], 0
    windows = [("default", {})] + [(f"y{y}", {"from_date": f"01-04-{y}", "to_date": f"31-12-{y}"}) for y in YEARS_BY_COMP.get(company, [])]
    for period in NSE_PERIODS:
        for tag, extra in windows:
            if bad_calls >= 3:
                break                                         # the site is not answering: stop wasting time
            data, err = cached(RAW_DIR / "nse" / f"{company}_list_{period}_{tag}.json",
                               lambda e=extra, pr=period: http_get(sess, NSE_HOME + "/api/corporates-financial-results",
                                                                   params={"index": "equities", "symbol": sym, "period": pr, **e}, kind="json"), kind="json")
            if data is None:
                errs.append(f"list/{period}/{tag}: {err}"); bad_calls += 1; continue
            rows = data if isinstance(data, list) else (data.get("data") or []) if isinstance(data, dict) else []
            for r in rows:
                if not isinstance(r, dict):
                    continue
                nrows += 1; keys = keys or sorted(r.keys())
                if period == "Quarterly" and not _is_q4(r):
                    continue
                u = _xml_link(r, "https://nsearchives.nseindia.com/")
                if u:
                    links[u] = r
    for u in links:
        name = re.sub(r"[^A-Za-z0-9._-]+", "_", u.rsplit("/", 1)[-1])
        _, err = cached(folder / name, lambda u=u: http_get(sess, u, headers={"Referer": NSE_HOME + "/"}, kind="bytes"), kind="bytes")
        if err:
            errs.append(f"{name}: {err}")
    frames, reasons, bad, ebit_notes = xbrl_company(folder, "nse")
    nfiles = len(list(folder.glob("*.xml")))
    note = f"{nrows} listing rows, {len(links)} with an XML link, {nfiles} XML files on disk"
    if reasons: note += f"; scale: {', '.join(reasons)}"
    if bad: note += f"; {bad} unreadable XML"
    if nrows and not links: note += f"; listing keys seen: {keys}"
    res["error"] = "; ".join(errs)[:300]
    res["ebit_diag"] = ebit_notes
    return finish(res, frames, frames, BASIS_ORDER + ["unknown"], note)
print("NSE fetcher ready")

In [ ]:
# ===== 8. Run the three fetchers in order: yfinance -> Screener -> NSE (no comparison happens here) =====
FETCHERS = {"yfinance": fetch_yfinance, "screener": fetch_screener, "nse": fetch_nse}
SRC = {s: {} for s in SOURCE_ORDER}
for s in SOURCE_ORDER:
    print(f"--- {s} ---", flush=True)
    fails_in_row = 0
    for comp in COMPS:
        if fails_in_row >= BREAKER:
            r = new_result(s, comp); r["error"] = f"skipped: first {BREAKER} companies returned nothing (site probably blocks Colab)"
            SRC[s][comp] = r; print(f"   {comp:8s} skipped", flush=True); continue
        t0 = time.time()
        try:
            r = FETCHERS[s](comp)
        except Exception as e:
            r = new_result(s, comp); r["error"] = f"crashed: {type(e).__name__}: {str(e)[:150]}"
        SRC[s][comp] = r
        fails_in_row = fails_in_row + 1 if r["status"] == "failed" else 0
        print(f"   {comp:8s} {r['status']:8s} {time.time()-t0:5.0f}s  {r['error'][:80]}", flush=True)

rows = []
for s in SOURCE_ORDER:
    for comp in COMPS:
        r = SRC[s][comp]
        yrs = sorted({c.year for c in r["balance_sheet"].columns}) if r["balance_sheet"] is not None else []
        rows.append({"source": s, "company": comp, "status": r["status"], "bs_years": yrs, "error": r["error"][:70], "note": r["note"][:100]})
STATUS = pd.DataFrame(rows)
print("\n=== FETCH STATUS ===")
print(STATUS.to_string(index=False))
print("\nstatus counts per source:")
print(STATUS.groupby(["source", "status"]).size().unstack(fill_value=0).to_string())
print("\nNSE EBIT build log (how EBIT was assembled from the XBRL tags):")
for comp in COMPS:
    for n in SRC["nse"][comp].get("ebit_diag", []):
        print(f"   {comp:8s} {n[:200]}")

In [ ]:
# ===== 9. Normalise every value into INR crore, one row per (source, company, year, field), with an explicit status =====
TO_CRORE = {"yfinance": Decimal(1) / CRORE, "screener": Decimal(1), "nse": Decimal(1)}     # yfinance reports rupees; Screener / NSE already in crore here

def fail_status(r):
    e = (r.get("error") or "").lower()
    return "parser_error" if any(k in e for k in ("crashed", "no matching table", "no balance-sheet", "unreadable")) else "download_failed"

def source_cell(s, comp, year, field):
    spec = LABELS_BY_SOURCE[s][field]
    rec = {"source": s, "company": comp, "year": year, "field": field, "value": None, "status": None, "label": spec["primary"], "basis": None, "detail": ""}
    r = SRC[s].get(comp)
    if spec["primary"] is None:
        rec["status"] = "unsupported_field"; rec["detail"] = f"{s} has no such line"; return rec
    if r is None or (r["balance_sheet"] is None and r["income_stmt"] is None):
        rec["status"] = fail_status(r or {}); rec["detail"] = ((r or {}).get("error") or (r or {}).get("note") or "")[:120]; return rec
    if s == "yfinance":
        cur = r.get("currency")
        if cur is not None and cur != "INR":
            rec["status"] = "currency_unsupported"; rec["detail"] = f"statements are in {cur}; no INR conversion is attempted"; return rec
        if cur is None:
            rec["detail"] = "INR assumed (yfinance gave no currency)"
    df = r[FIELD_STMT[field]]
    if df is None:
        rec["status"] = "statement_missing"; return rec
    row = find_row(df, spec["primary"])
    if row is None:
        rec["status"] = "label_missing"
        if s == "nse" and field == "ebit":
            rec["detail"] = "; ".join(r.get("ebit_diag", []))[:160]
        return rec
    col = pick_column(row, year)
    if col is None:
        rec["status"] = "year_not_available"; return rec
    v = as_decimal(row[col])
    if v is None:
        rec["status"] = "value_empty"; return rec
    rec.update(value=v * TO_CRORE[s], status="ok", basis=r["basis_by_year"].get(col) or ("unknown (assumed consolidated)" if s == "yfinance" else None))
    return rec

LONG = pd.DataFrame([source_cell(s, c["company"], c["year"], c["field"]) for s in SOURCE_ORDER for c in CELLS])
print("status per source (every cell is accounted for):")
print(LONG.groupby(["source", "status"]).size().unstack(fill_value=0).to_string())
LONG.astype(str).to_csv(OUT_DIR / "source_values_long.csv", index=False)

# ---- precision actually seen in each source (measured from the data, no ground truth) ----
def observe_precision(vals):
    vals = [v for v in vals if v is not None]
    if len(vals) < 5:
        return None
    for p in (Decimal(1), Decimal("0.1"), Decimal("0.01"), Decimal("0.001")):
        if sum(1 for v in vals if v % p == 0) / len(vals) >= 0.95:
            return p
    return Decimal(0)
DEFAULT_PREC = {"yfinance": Decimal(1), "screener": Decimal(1), "nse": Decimal("0.01")}      # only used if too few values to measure
PREC, rows = {}, []
for s in SOURCE_ORDER:
    vals = list(LONG[(LONG.source == s) & (LONG.status == "ok")].value)
    obs = observe_precision(vals)
    PREC[s] = PRECISION_OVERRIDE.get(s) or obs or DEFAULT_PREC[s]
    rows.append({"source": s, "values": len(vals), "measured_precision_crore": obs, "used_precision_crore": PREC[s],
                 "whole_crore_share_%": round(100 * sum(1 for v in vals if v % 1 == 0) / len(vals), 1) if vals else None})
print("\n=== precision of each source, measured (crore). 1 = whole crores, 0.01 = lakhs, 0 = exact ===")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# ===== 10. Source-to-source comparison (no ground truth). Rounding floor = 0.5 x precision(A) + 0.5 x precision(B), in crore =====
def pair_compare(a, b, pa, pb):
    """symmetric comparison of two crore values. Handles zero and negative values; returns (class, abs_diff_crore, rel_%)"""
    gap = abs(a - b)
    floor = Decimal("0.5") * pa + Decimal("0.5") * pb
    denom = max(abs(a), abs(b))
    rel = None if denom == 0 else gap / denom * 100
    if gap == 0:
        cls = "exact"
    elif gap <= floor or (rel is not None and rel <= TOL_PCT):
        cls = "within_tolerance"
    elif rel is not None and rel <= NEAR_PCT:
        cls = "near"
    else:
        cls = "mismatch"
    return cls, gap, rel

def power_of_ten(a, b):
    if a == 0 or b == 0:
        return False
    ratio = float(a / b)
    return ratio > 0 and any(abs(ratio / 10 ** k - 1) < 0.005 for k in range(-7, 8) if k != 0)

VAL = {(r.source, r.company, r.year, r.field): r for r in LONG.itertuples()}
rows = []
for c in CELLS:
    for a, b in PAIRS:
        ra, rb = VAL[(a, c["company"], c["year"], c["field"])], VAL[(b, c["company"], c["year"], c["field"])]
        rec = {"report_id": c["report_id"], "company": c["company"], "year": c["year"], "field": c["field"], "pair": f"{a} vs {b}",
               "value_a": ra.value, "value_b": rb.value, "status_a": ra.status, "status_b": rb.status, "basis_a": ra.basis, "basis_b": rb.basis,
               "abs_diff_crore": None, "rel_diff_%": None, "class": None, "pair_status": None, "basis_differs": False, "power_of_ten": False}
        if ra.value is None or rb.value is None:
            rec["pair_status"] = "missing_" + "_and_".join(n for n, r in ((a, ra), (b, rb)) if r.value is None)
        else:
            cls, gap, rel = pair_compare(ra.value, rb.value, PREC[a], PREC[b])
            known = lambda x: x is not None and not str(x).startswith("unknown")
            rec.update({"abs_diff_crore": gap, "rel_diff_%": None if rel is None else round(float(rel), 4), "class": cls, "pair_status": "compared",
                        "basis_differs": bool(known(ra.basis) and known(rb.basis) and ra.basis != rb.basis), "power_of_ten": power_of_ten(ra.value, rb.value)})
        rows.append(rec)
PAIRDF = pd.DataFrame(rows)
print("pair status counts:"); print(PAIRDF.groupby(["pair", "pair_status"]).size().unstack(fill_value=0).to_string())
print("\npair class counts (compared cells only):")
print(PAIRDF.groupby(["pair", "class"]).size().unstack(fill_value=0).reindex(columns=["exact", "within_tolerance", "near", "mismatch"], fill_value=0).to_string())
print("\ncompared pairs whose sources report a different basis (consolidated vs standalone):", int(PAIRDF.basis_differs.sum()),
      "| pairs that differ by an exact power of ten (unit problem):", int(PAIRDF.power_of_ten.sum()))
PAIRDF.astype(str).to_csv(OUT_DIR / "pairwise_comparison.csv", index=False)

In [ ]:
# ===== 11. Override selection. Uses ONLY the three source values; ground truth is not loaded yet. =====
RULES = {"version": "v1.1 (two-source disagreement selects the anchor; flag only when the anchor is outvoted by both others)", "anchor": ANCHOR, "fallback_priority": FALLBACK_PRIORITY, "tol_pct": str(TOL_PCT), "near_pct": str(NEAR_PCT), "material_classes": list(MATERIAL),
         "precision_crore": {k: str(v) for k, v in PREC.items()},
         "rules_text": [
             "non-EBIT fields: anchor = yfinance. All three agree -> consensus, select yfinance. Within tolerance -> select yfinance, record disagreement.",
             "yfinance disagrees materially with BOTH other sources (all three values available) -> FLAG for investigation (no automatic selection).",
             "yfinance disagrees with the only other source that has a value (two sources) -> select yfinance anyway, marked unverified / disputed.",
             "EBIT: anchor = Screener, same logic. Screener unavailable -> fallback to the first source in the EBIT priority list that has a value.",
             "anchor unavailable and the two others disagree -> select the first source in the priority list, marked unverified / disputed.",
             "only one source returns a value -> keep it, unverified. no source -> unavailable.",
             "agree = exact or within_tolerance; material disagreement = near or mismatch (> tolerance)"]}
rules_path = OUT_DIR / "rules_frozen.json"
rules_path.write_text(json.dumps({"frozen_at": datetime.datetime.now().isoformat(timespec="seconds"), **RULES}, indent=2))
print("rules frozen and saved BEFORE any ground-truth value is read:", rules_path)

PCLS = {(r["pair"], r["report_id"], r["field"]): r["class"] for r in PAIRDF.to_dict("records")}
def pair_class(report_id, field, a, b):
    return PCLS.get((f"{a} vs {b}", report_id, field)) or PCLS.get((f"{b} vs {a}", report_id, field))
agree = lambda c: c in ("exact", "within_tolerance")

def decide(report_id, field, vals):
    key = "ebit" if field == "ebit" else "default"
    anchor, prio = ANCHOR[key], FALLBACK_PRIORITY[key]
    avail = [s for s in prio if vals.get(s) is not None]
    n = len(avail)
    pcls = lambda x, y: pair_class(report_id, field, x, y)
    d = {"decision": None, "selected_source": None, "selected_value": None, "verification": None, "flagged": False,
         "proposed_source": None, "proposed_value": None, "fallback_used": False, "rationale": ""}
    def pick(src, decision, verification, why, fallback=False):
        d.update(decision=decision, selected_source=src, selected_value=vals[src], verification=verification, fallback_used=fallback, rationale=why)
        return d
    if n == 0:
        d.update(decision="unavailable", verification="unavailable", rationale="no source returned a usable value"); return d
    if n == 1:
        return pick(avail[0], "single_source_unverified", "unverified", f"only {avail[0]} returned a value", fallback=(avail[0] != anchor))
    all_pairs = [pcls(x, y) for x, y in itertools.combinations(avail, 2)]
    if anchor in avail:
        others = [s for s in avail if s != anchor]
        ok = [o for o in others if agree(pcls(anchor, o))]
        bad = [o for o in others if o not in ok]
        if not bad:
            if n == 3 and all(agree(c) for c in all_pairs):
                if all(c == "exact" for c in all_pairs):
                    return pick(anchor, "consensus_exact", "verified", f"all three sources identical; selected {anchor}")
                return pick(anchor, "consensus_within_tolerance", "verified", f"all three agree within tolerance ({', '.join(all_pairs)}); selected {anchor}, disagreement recorded")
            if n == 3:
                return pick(anchor, "anchor_confirmed_by_both", "partially_verified", f"{anchor} agrees with both others, but they differ from each other beyond tolerance")
            return pick(anchor, "two_source_agreement", "partially_verified", f"{anchor} agrees with {others[0]}; third source has no value")
        if ok:
            return pick(anchor, "anchor_confirmed_by_one", "partially_verified", f"{anchor} confirmed by {ok[0]}; {bad[0]} disagrees materially ({pcls(anchor, bad[0])})")
        if n == 2:
            return pick(anchor, "anchor_default_disputed", "unverified", f"{anchor} disagrees with {others[0]} ({pcls(anchor, others[0])}); only two sources, nobody outvotes {anchor}: default {anchor} selected, disputed")
        d.update(decision="flagged_anchor_disagrees", flagged=True, verification="flagged",
                 rationale=f"{anchor} disagrees materially with {', '.join(others)}: flagged, not selected")
        if len(others) == 2 and agree(pcls(others[0], others[1])):
            d.update(proposed_source=others[0], proposed_value=vals[others[0]], rationale=d["rationale"] + f" (the other two agree: majority value would be {others[0]})")
        return d
    if n == 2 and agree(all_pairs[0]):
        return pick(avail[0], "fallback_two_agree", "partially_verified", f"anchor {anchor} unavailable; {avail[0]} and {avail[1]} agree; selected {avail[0]} by priority", fallback=True)
    return pick(avail[0], "fallback_priority_disputed", "unverified", f"anchor {anchor} unavailable and {avail[0]} vs {avail[1]} disagree ({all_pairs[0]}); selected {avail[0]} by priority, disputed", fallback=True)

rows = []
for c in CELLS:
    vals = {s: VAL[(s, c["company"], c["year"], c["field"])].value for s in SOURCE_ORDER}
    d = decide(c["report_id"], c["field"], vals)
    rec = {**c, **{f"{s}_value": vals[s] for s in SOURCE_ORDER}, **{f"{s}_status": VAL[(s, c["company"], c["year"], c["field"])].status for s in SOURCE_ORDER},
           **{f"{s}_basis": VAL[(s, c["company"], c["year"], c["field"])].basis for s in SOURCE_ORDER}, **d,
           **{f"pair_{a}_vs_{b}": pair_class(c["report_id"], c["field"], a, b) for a, b in PAIRS}}
    rows.append(rec)
SEL = pd.DataFrame(rows)
print("\ndecision counts:"); print(SEL.decision.value_counts().to_string())
print("\nselected source:"); print(SEL.selected_source.fillna("(none)").value_counts().to_string())
print("\ndecision x field:"); print(SEL.groupby(["field", "decision"]).size().unstack(fill_value=0).to_string())
SEL.astype(str).to_csv(OUT_DIR / "selection_audit.csv", index=False)

In [ ]:
# ===== 12. FINAL EVALUATION: only now are the ground-truth answers read. Nothing below changes a selected value. =====
assert rules_path.exists(), "rules must be frozen before ground truth is read"
GT_ROWS = []
for rid, rep in REPORTS.items():
    ans = (json.loads(Path(rep["path"]).read_text(encoding="utf-8")).get("answers") or {})
    for f in FIELDS:
        raw = ans.get(f, "<key missing>")
        unit_txt = ans.get("income_statement_currency") if FIELD_STMT[f] == "income_stmt" else ans.get("balance_sheet_currency")
        d = to_dec(raw)
        GT_ROWS.append({"report_id": rid, "field": f, "gt_raw": raw, "gt_value": d, "gt_unit": unit_txt, "gt_scale": unit_scale(unit_txt),
                        "gt_status": "numeric" if d is not None else ("key_missing" if raw == "<key missing>" else "N/A")})
GTDF = pd.DataFrame(GT_ROWS)
print("ground-truth cells:", GTDF.gt_status.value_counts().to_dict(), "| units:", GTDF.gt_unit.value_counts(dropna=False).to_dict())
FULL = SEL.merge(GTDF, on=["report_id", "field"], how="left")
dec = lambda x: x if isinstance(x, Decimal) else None
fmt = lambda v: "-" if v is None or (not isinstance(v, Decimal) and pd.isna(v)) else f"{float(v):,.2f}"      # display only

def classify_gt(conv, gt, gt_scale, prec):
    gap = conv - gt
    gap_crore = gap * gt_scale / CRORE
    floor = Decimal("0.5") * prec + Decimal("0.5") * gt_scale / CRORE
    gap_pct = None if gt == 0 else gap / abs(gt) * 100
    if gap == 0: cls = "exact"
    elif abs(gap_crore) <= floor or (gap_pct is not None and abs(gap_pct) <= TOL_PCT): cls = "within_tolerance"
    elif gap_pct is not None and abs(gap_pct) <= NEAR_PCT: cls = "near"
    else: cls = "mismatch"
    return cls, gap_pct, gap_crore

METHODS = ["override", "override+majority", "yfinance", "screener", "nse"]
def method_value(row, m):
    if m == "override":
        return dec(row.selected_value), row.selected_source
    if m == "override+majority":
        return (dec(row.selected_value), row.selected_source) if dec(row.selected_value) is not None else (dec(row.proposed_value), row.proposed_source)
    return dec(getattr(row, f"{m}_value")), m

ev = []
for row in FULL.itertuples():
    if row.gt_status != "numeric":
        continue
    for m in METHODS:
        v, src = method_value(row, m)
        rec = {"method": m, "report_id": row.report_id, "company": row.company, "year": row.year, "field": row.field, "gt_value": row.gt_value, "gt_unit": row.gt_unit,
               "source_used": src, "value_crore": v, "converted": None, "abs_err_crore": None, "pct_err": None, "class": None, "no_value_reason": None}
        if v is None:
            rec["no_value_reason"] = (row.decision if m.startswith("override") else getattr(row, f"{m}_status"))
        elif row.gt_scale is None:
            rec["no_value_reason"] = "gt_unit_unknown"
        else:
            conv = v * CRORE / row.gt_scale
            cls, gp, gc = classify_gt(conv, row.gt_value, row.gt_scale, PREC[src])
            rec.update(converted=round(float(conv), 4), abs_err_crore=round(float(abs(gc)), 4), pct_err=None if gp is None else round(float(abs(gp)), 4), **{"class": cls})
        ev.append(rec)
EVAL = pd.DataFrame(ev)
print(f"{FULL[FULL.gt_status == 'numeric'].shape[0]} cells have a numeric ground truth and are evaluated; "
      f"{(FULL.gt_status != 'numeric').sum()} are N/A (NOT scored, NOT treated as zero).")
na = FULL[(FULL.gt_status != "numeric") & FULL.selected_value.notna()]
print(f"cells where the ground truth is N/A but the override returned a value: {len(na)} (listed in the audit file, not scored)")
EVAL.astype(str).to_csv(OUT_DIR / "evaluation_long.csv", index=False)
FULL.astype(str).to_csv(OUT_DIR / "audit_all_cells.csv", index=False)

In [ ]:
# ===== 13. Reports 1 to 4: coverage, agreement, override performance, override vs each single source =====
pct = lambda a, b: round(100 * a / b, 1) if b else None
show = lambda t: print(t.to_string(index=False))
CLASSES = ["exact", "within_tolerance", "near", "mismatch"]

print("=" * 100); print("REPORT 1: SOURCE COVERAGE (all", len(CELLS), "cells; every status is explicit)"); print("=" * 100)
cov = LONG.groupby(["source", "field", "status"]).size().unstack(fill_value=0)
cov["cells"] = cov.sum(axis=1)
cov["coverage_%"] = (100 * cov.get("ok", 0) / cov["cells"]).round(1)
print(cov.reset_index().to_string(index=False))
print("\nper source overall:")
t = LONG.groupby("source").status.value_counts().unstack(fill_value=0)
t["cells"] = t.sum(axis=1); t["coverage_%"] = (100 * t["ok"] / t["cells"]).round(1) if "ok" in t else 0
print(t.to_string())
gtn = FULL[FULL.gt_status == "numeric"]
print("\ncoverage on the cells that have a numeric ground truth (these are the evaluation cells):")
print(pd.DataFrame([{"source": s, "eval_cells": len(gtn), "retrieved": int(gtn[f"{s}_value"].notna().sum()), "coverage_%": pct(int(gtn[f"{s}_value"].notna().sum()), len(gtn))} for s in SOURCE_ORDER]
                   + [{"source": "override (selected)", "eval_cells": len(gtn), "retrieved": int(gtn.selected_value.notna().sum()), "coverage_%": pct(int(gtn.selected_value.notna().sum()), len(gtn))}]).to_string(index=False))

print("\n" + "=" * 100); print("REPORT 2: SOURCE AGREEMENT (pairs, no ground truth)"); print("=" * 100)
def pair_summary(g):
    comp = g[g["class"].notna()]; c = comp["class"].value_counts(); n = len(comp)
    ex, wt, nr, mm = (int(c.get(k, 0)) for k in CLASSES)
    return {"cells": len(g), "both_available": n, "exact": ex, "within_tol": wt, "near(<=1%)": nr, "mismatch": mm,
            "exact_%": pct(ex, n), "exact+tol_%": pct(ex + wt, n), "within_1%_%": pct(ex + wt + nr, n)}
print("--- overall per pair ---"); show(pd.DataFrame([{"pair": k, **pair_summary(g)} for k, g in PAIRDF.groupby("pair")]))
print("\n--- per pair and field ---"); show(pd.DataFrame([{"pair": k[0], "field": k[1], **pair_summary(g)} for k, g in PAIRDF.groupby(["pair", "field"])]))

def acc(g):
    n = len(g); cov_ = int(g["class"].notna().sum()); c = g["class"].value_counts()
    ex, wt, nr, mm = (int(c.get(k, 0)) for k in CLASSES)
    return {"gt_cells": n, "covered": cov_, "coverage_%": pct(cov_, n), "exact": ex, "within_tol": wt, "near": nr, "mismatch": mm,
            "exact_%_all": pct(ex, n), "exact+tol_%_all": pct(ex + wt, n), "exact+tol_%_covered": pct(ex + wt, cov_), "within_1%_%_covered": pct(ex + wt + nr, cov_)}

print("\n" + "=" * 100); print("REPORT 3: OVERRIDE PERFORMANCE"); print("=" * 100)
print("decisions (all cells):"); print(SEL.decision.value_counts().to_string())
print("\nverification status (all cells):"); print(SEL.verification.value_counts().to_string())
print("\nselected-source distribution:"); print(SEL.selected_source.fillna("(none: flagged / unavailable)").value_counts().to_string())
print(f"\nfallback selections (anchor had no value): {int(SEL.fallback_used.sum())} | flagged / unresolved disagreements: {int(SEL.flagged.sum())} | unavailable: {int((SEL.decision == 'unavailable').sum())}")
O = EVAL[EVAL.method == "override"]
print("\nFINAL ACCURACY of the override (selected values vs ground truth; flagged / unavailable cells count as not covered):")
show(pd.DataFrame([acc(O)]))
print("\n--- by field ---");  show(pd.DataFrame([{"field": k, **acc(g)} for k, g in O.groupby("field")]))
print("\n--- by company ---"); show(pd.DataFrame([{"company": k, **acc(g)} for k, g in O.groupby("company")]))
print("\n--- by year ---");   show(pd.DataFrame([{"year": k, **acc(g)} for k, g in O.groupby("year")]))
O2 = O.merge(SEL[["report_id", "field", "decision", "verification"]], on=["report_id", "field"])
print("\n--- by verification status (evaluation only: does 'verified' really mean correct?) ---"); show(pd.DataFrame([{"verification": k, **acc(g)} for k, g in O2.groupby("verification")]))
print("\n--- by decision ---"); show(pd.DataFrame([{"decision": k, **acc(g)} for k, g in O2.groupby("decision")]))

print("\n" + "=" * 100); print("REPORT 4: OVERRIDE vs EACH SOURCE ON THE SAME CELLS"); print("=" * 100)
print("(a) every cell with a numeric ground truth (missing = not covered; 'all' columns treat a missing cell as wrong)")
show(pd.DataFrame([{"method": k, **acc(g)} for k, g in EVAL.groupby("method")]).set_index("method").loc[METHODS].reset_index())
common = FULL[(FULL.gt_status == "numeric") & FULL.yfinance_value.notna() & FULL.screener_value.notna() & FULL.nse_value.notna()][["report_id", "field"]]
EC = EVAL.merge(common, on=["report_id", "field"])
print(f"\n(b) IDENTICAL cells: only cells where all three sources returned a value ({len(common)} cells) - same cells for every method")
if len(common):
    show(pd.DataFrame([{"method": k, **acc(g)} for k, g in EC.groupby("method")]).set_index("method").loc[METHODS].reset_index())
    print("\n    by field on the identical cells:")
    show(pd.DataFrame([{"field": f, "method": m, **acc(g)} for (f, m), g in EC.groupby(["field", "method"])]).sort_values(["field", "method"]))
else:
    print("    no cell has a value from all three sources")
both = FULL[(FULL.gt_status == "numeric") & FULL.selected_value.notna()][["report_id", "field"]]
EB = EVAL.merge(both, on=["report_id", "field"])
print(f"\n(c) cells where the override selected a value ({len(both)} cells): each single source on exactly those cells")
show(pd.DataFrame([{"method": k, **acc(g)} for k, g in EB.groupby("method")]).set_index("method").loc[METHODS].reset_index())
print("\n(d) by field, all evaluation cells: exact+tol % of ALL cells (missing counts as wrong)")
print(pd.DataFrame([{"field": f, "method": m, "exact+tol_%_all": acc(g)["exact+tol_%_all"], "coverage_%": acc(g)["coverage_%"]} for (f, m), g in EVAL.groupby(["field", "method"])])
      .pivot(index="field", columns="method", values="exact+tol_%_all")[METHODS].to_string())

In [ ]:
# ===== 14. REPORT 5: error analysis (evaluation only) =====
print("=" * 100); print("REPORT 5: ERROR ANALYSIS"); print("=" * 100)
ctx = FULL.set_index(["report_id", "field"])
def alt_matches(src, company, year, field, gt_conv_crore):
    """does an ALTERNATIVE row of the selected source reproduce the ground truth? (explains definition differences)"""
    r = SRC[src].get(company)
    if r is None:
        return ""
    df = r[FIELD_STMT[field]]
    if df is None:
        return ""
    for alt in LABELS_BY_SOURCE[src][field]["alts"]:
        row = find_row(df, alt)
        if row is not None and pick_column(row, year) is not None:
            v = as_decimal(row[pick_column(row, year)])
            if v is not None and abs(v * TO_CRORE[src] - gt_conv_crore) <= Decimal("0.5") * PREC[src]:
                return alt
    return ""

def causes(e):
    c = ctx.loc[(e.report_id, e.field)]
    gt_crore = e.gt_value * unit_scale(e.gt_unit) / CRORE
    tags = []
    if power_of_ten(e.value_crore, gt_crore):
        tags.append("unit / scale (power of ten)")
    if e.source_used and str(c[f"{e.source_used}_basis"]).startswith("standalone"):
        tags.append("selected value is STANDALONE")
    if any(PAIRDF[(PAIRDF.report_id == e.report_id) & (PAIRDF.field == e.field)].basis_differs):
        tags.append("sources use different bases")
    alt = alt_matches(e.source_used, e.company, e.year, e.field, gt_crore) if e.source_used else ""
    if alt:
        tags.append(f"alt row '{alt}' matches GT (definition)")
    right = [s for s in SOURCE_ORDER if s != e.source_used and dec(c[f"{s}_value"]) is not None and abs(dec(c[f"{s}_value"]) - gt_crore) <= Decimal("0.5") * PREC[s] + Decimal("0.5") * unit_scale(e.gt_unit) / CRORE]
    if right:
        tags.append("another source matches GT: " + ",".join(right))
    return "; ".join(tags) or "unexplained (restatement / definition / data error)"

bad = O[O["class"].isin(["near", "mismatch"])].copy()
bad["cause"] = [causes(e) for e in bad.itertuples()]
bad["src_values(crore)"] = [" | ".join(f"{s[:3]}={fmt(ctx.loc[(r.report_id, r.field)][f'{s}_value'])}" for s in SOURCE_ORDER) for r in bad.itertuples()]
bad["decision"] = [ctx.loc[(r.report_id, r.field)]["decision"] for r in bad.itertuples()]
print(f"\n--- selected values that are near / mismatch against ground truth: {len(bad)} cells (largest relative error first) ---")
if len(bad):
    show(bad.sort_values("pct_err", ascending=False)[["report_id", "field", "source_used", "gt_value", "converted", "pct_err", "decision", "cause", "src_values(crore)"]].head(40))
    print("\ncause counts (a cell can have several tags):")
    print(bad.cause.str.split("; ").explode().str.replace(r"another source matches GT: .*", "another source matches GT", regex=True).str.replace(r"alt row .*", "alt row matches GT (definition)", regex=True).value_counts().to_string())

print("\n--- the largest discrepancies between SOURCES (no ground truth): top 25 relative differences ---")
pc = PAIRDF[PAIRDF.pair_status == "compared"].copy(); pc["rel"] = pc["rel_diff_%"].astype(float)
top = pc.sort_values("rel", ascending=False)[["report_id", "field", "pair", "value_a", "value_b", "rel_diff_%", "class", "basis_a", "basis_b", "power_of_ten"]].head(25).copy()
top["value_a"], top["value_b"] = top.value_a.map(fmt), top.value_b.map(fmt)
show(top)

fl = FULL[(FULL.flagged) & (FULL.gt_status == "numeric")]
print(f"\n--- flagged cells with a numeric ground truth: {len(fl)} (which source was actually closest? evaluation only) ---")
rows = []
for r in fl.itertuples():
    row = {"report_id": r.report_id, "field": r.field, "decision": r.decision, "gt": r.gt_value}
    for s in SOURCE_ORDER:
        v = dec(getattr(r, f"{s}_value"))
        row[s] = None if v is None else classify_gt(v * CRORE / r.gt_scale, r.gt_value, r.gt_scale, PREC[s])[0]
    row["proposed_majority"] = None if dec(r.proposed_value) is None else classify_gt(dec(r.proposed_value) * CRORE / r.gt_scale, r.gt_value, r.gt_scale, PREC[r.proposed_source])[0]
    rows.append(row)
if rows:
    show(pd.DataFrame(rows))

print("\n--- NSE EBIT parser check ---")
nse_e = LONG[(LONG.source == "nse") & (LONG.field == "ebit")]
print("NSE EBIT status counts:", nse_e.status.value_counts().to_dict())
for comp in COMPS:
    r = SRC["nse"][comp]
    bad_e = nse_e[(nse_e.company == comp) & (nse_e.status != "ok")]
    if len(bad_e):
        print(f"   {comp:8s} {len(bad_e)} cells without NSE EBIT | {'; '.join(r.get('ebit_diag', []))[:220] or r['note'][:160]}")

print("\n--- reporting-basis check for NSE (Infosys / Reliance FY2023 were suspected): every cell, values in crore ---")
chk = FULL[(FULL.company.isin(["INFOSYS", "RIL"])) & (FULL.year == 2023)]
chk = chk[["report_id", "field", "yfinance_value", "screener_value", "nse_value", "yfinance_basis", "screener_basis", "nse_basis", "decision", "gt_value", "gt_unit"]].copy()
for c_ in ("yfinance_value", "screener_value", "nse_value"):
    chk[c_] = chk[c_].map(fmt)
show(chk)

print("\n--- cells where the ground truth is N/A but the sources / override returned a value (information only, not scored) ---")
na = FULL[(FULL.gt_status != "numeric") & (FULL[[f"{s}_value" for s in SOURCE_ORDER]].notna().any(axis=1))]
na_t = na[["report_id", "field", "gt_raw", "yfinance_value", "screener_value", "nse_value", "selected_value", "decision"]].head(40).copy()
for c_ in ("yfinance_value", "screener_value", "nse_value", "selected_value"):
    na_t[c_] = na_t[c_].map(fmt)
show(na_t) if len(na) else print("none")

In [ ]:
# ===== 15. REPORT 6: final summary + save everything =====
print("=" * 100); print("REPORT 6: FINAL SUMMARY"); print("=" * 100)
def metric(df):
    a = acc(df); return a
M = {m: acc(EVAL[EVAL.method == m]) for m in METHODS}
Mc = {m: acc(EC[EC.method == m]) for m in METHODS} if len(common) else {}
n_eval = M["override"]["gt_cells"]
print(f"Evaluation cells (numeric ground truth): {n_eval}   |   all cells attempted: {len(CELLS)}")
print("\nAll evaluation cells (missing counts against a method):")
print(pd.DataFrame(M).T[["coverage_%", "exact_%_all", "exact+tol_%_all", "exact+tol_%_covered", "within_1%_%_covered"]].loc[METHODS].to_string())
if Mc:
    print(f"\nIdentical cells where all three sources have a value ({len(common)}):")
    print(pd.DataFrame(Mc).T[["gt_cells", "exact_%_all", "exact+tol_%_all", "within_1%_%_covered"]].loc[METHODS].to_string())

best_single = max(SOURCE_ORDER, key=lambda s: M[s]["exact+tol_%_all"] or 0)
ov = M["override"]
print("\nAnswers to the two questions:")
print(f"1) ACCURACY. Override exact+tolerance = {ov['exact+tol_%_all']}% of all evaluation cells ({ov['exact+tol_%_covered']}% of the cells it answered); "
      f"best single source ({best_single}) = {M[best_single]['exact+tol_%_all']}% of all cells ({M[best_single]['exact+tol_%_covered']}% of its answered cells).")
if Mc:
    bs_c = max(SOURCE_ORDER, key=lambda s: Mc[s]["exact+tol_%_all"] or 0)
    print(f"   On identical cells (all three sources present): override {Mc['override']['exact+tol_%_all']}% vs {bs_c} {Mc[bs_c]['exact+tol_%_all']}%.")
print(f"2) COVERAGE. Override answers {ov['coverage_%']}% of evaluation cells (flagged cells are deliberately left unanswered: {int(SEL.flagged.sum())}); "
      f"best single-source coverage = {max(M[s]['coverage_%'] for s in SOURCE_ORDER)}%.")
print(f"   If flagged cells are resolved by the majority of the other two sources: coverage {M['override+majority']['coverage_%']}%, exact+tol {M['override+majority']['exact+tol_%_all']}% of all cells.")
ver = O2[O2.verification == "verified"]
wrong_cons = int(ver["class"].isin(["near", "mismatch"]).sum())
print(f"3) CONSENSUS IS NOT PROOF. {len(ver)} evaluated cells were three-source 'verified'; {wrong_cons} of them are still near / mismatch against ground truth "
      f"(sources can share upstream data or errors).")
print("\nLimitations to state with these numbers:")
for t in ["Rules were designed after an earlier ground-truth run, so some overfitting risk remains; they were frozen (rules_frozen.json) before ground truth was read here.",
          "yfinance reports Infosys in USD: it is marked currency_unsupported (no FX conversion), so Infosys relies on Screener / NSE.",
          "Screener has no current-liabilities line; NSE EBIT depends on the XBRL tags found (see the NSE EBIT parser check above).",
          "Source precision is measured from the data; yfinance / Screener round to whole crores, which limits exact matches.",
          "Consolidated vs standalone is taken from each source (yfinance basis is assumed consolidated, not reported).",
          "NSE listing endpoints are undocumented and may return limited years; coverage per source is in Report 1."]:
    print("  -", t)

# ---- save ----
with pd.ExcelWriter(OUT_DIR / "core_comparison_summary.xlsx") as xw:
    cov.reset_index().to_excel(xw, sheet_name="1_coverage", index=False)
    pd.DataFrame([{"pair": k, **pair_summary(g)} for k, g in PAIRDF.groupby("pair")]).to_excel(xw, sheet_name="2_agreement", index=False)
    pd.DataFrame([{"pair": k[0], "field": k[1], **pair_summary(g)} for k, g in PAIRDF.groupby(["pair", "field"])]).to_excel(xw, sheet_name="2_agreement_by_field", index=False)
    SEL.decision.value_counts().rename_axis("decision").reset_index(name="cells").to_excel(xw, sheet_name="3_decisions", index=False)
    pd.DataFrame([{"method": m, **M[m]} for m in METHODS]).to_excel(xw, sheet_name="4_methods_all_cells", index=False)
    if Mc:
        pd.DataFrame([{"method": m, **Mc[m]} for m in METHODS]).to_excel(xw, sheet_name="4_methods_identical_cells", index=False)
    bad.astype(str).to_excel(xw, sheet_name="5_selected_errors", index=False)
    FULL.astype(str).to_excel(xw, sheet_name="audit_all_cells", index=False)
    EVAL.astype(str).to_excel(xw, sheet_name="evaluation_long", index=False)
    PAIRDF.astype(str).to_excel(xw, sheet_name="pairwise", index=False)
    LONG.astype(str).to_excel(xw, sheet_name="source_values", index=False)
    STATUS.astype(str).to_excel(xw, sheet_name="fetch_status", index=False)

results = {"run_date": datetime.date.today().isoformat(), "python": platform.python_version(), "pandas": pd.__version__, "yfinance": yf.__version__,
           "cells_attempted": len(CELLS), "evaluation_cells": n_eval, "methods_all_cells": {m: M[m] for m in METHODS},
           "methods_identical_cells": Mc, "identical_cells": int(len(common)), "decision_counts": SEL.decision.value_counts().to_dict(),
           "verification_counts": SEL.verification.value_counts().to_dict(), "flagged": int(SEL.flagged.sum()), "consensus_cells_wrong_vs_gt": wrong_cons}
(OUT_DIR / "results.json").write_text(json.dumps(results, indent=2, default=str))
config = {"sources": SOURCE_ORDER, "fields": FIELDS, "basis_order": BASIS_ORDER, "rules": RULES, "measured_precision_crore": {k: str(v) for k, v in PREC.items()},
          "labels": LABELS_BY_SOURCE, "aliases": ALIASES, "nse_periods": list(NSE_PERIODS), "companies": COMPANIES}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2, default=str))
print("\nSaved to", OUT_DIR, ": audit_all_cells.csv, selection_audit.csv, pairwise_comparison.csv, source_values_long.csv, evaluation_long.csv, core_comparison_summary.xlsx, results.json, config.json, rules_frozen.json")

In [ ]:
# ===== Snapshot of every Part I name, taken BEFORE Part II runs (used by the collision check) =====
_P1_SNAP = {k: v for k, v in list(globals().items()) if not k.startswith("_")}
print("Part I names snapshotted:", len(_P1_SNAP))
for _k in ("EVAL", "FULL", "SEL", "PREC", "OUT_DIR", "RAW_DIR"):
    assert _k in _P1_SNAP, f"Part I did not create {_k}; run Part I first"
print("EVAL rows:", len(EVAL), "| FULL rows:", len(FULL), "| OUT_DIR:", OUT_DIR)

# FROZEN v4 integration rule (written to disk before Part II, Part L or Part III compute anything)

In [ ]:
# ===== FROZEN v4 INTEGRATION RULE. Written to disk BEFORE Part II, Part L or Part III compute anything =====
import json, hashlib, datetime
V4_RULE_TEXT = [
 "P = fixed PDF route (status 'found' AND identity != 'FAIL' AND a value exists). L = validated Llama extraction (every number grounded in the cited raw row, current-year column, unit from the PDF-route detector). O = override.",
 "1. P and L both exist and agree within 0.1% -> use P (same value).",
 "2. Both exist and disagree: if O exists and supports exactly one of them within 1% -> use that one. Otherwise (O supports neither, supports both, or is missing): use L when P came from a row the parser flagged (multi-candidate, column ambiguity or merged-line split); else keep P and add the cell to the REVIEW list.",
 "3. Only one of P, L exists: accept it if O is missing or agrees within 1%. If O differs by 1% to 10%, take O. If O differs by MORE than 10% the override is implausible: keep the document value (P or L) and add the cell to the REVIEW list (v4 change, made after seeing the v3 results).",
 "4. Only O exists -> O. None -> not covered.",
 "Class of the final value = class of the method whose value was used. The ground-truth key is used ONLY to score, never to choose a page, unit, value, row or source."]
V4_RULE_PARAMS = {"agree_pdf_llama_rel": 0.001, "support_override_rel": 0.01, "implausible_override_rel": 0.10,
                  "parser_flagged": "multi_candidate OR column ambiguity (leading tokens that are not note/page-like, or more than 2) OR merged-line split"}
V4_RULE_SHA = hashlib.sha256(json.dumps([V4_RULE_TEXT, V4_RULE_PARAMS], sort_keys=True).encode()).hexdigest()
V4_RULE_PATH = OUT_DIR / "final_v4_rule_frozen.json"
V4_RULE_PATH.write_text(json.dumps({"frozen_at": datetime.datetime.now().isoformat(timespec="seconds"), "sha256": V4_RULE_SHA,
                                    "rule": V4_RULE_TEXT, **V4_RULE_PARAMS}, indent=2), encoding="utf-8")
print("Rule frozen before any v4 result exists:", V4_RULE_PATH, "| sha256", V4_RULE_SHA[:16])
print("\n".join(V4_RULE_TEXT))


# PART II - PDF text route (v1.3 code; Drive not mounted again; `SEL`->`PDFSEL`, `clean_label`->`pdf_clean_label`)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymupdf"])

import re, json, math, datetime as dt
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import fitz  # PyMuPDF
from IPython.display import display

# Paths reused from Part I (BASE_DIR, GT_DIR, OUT_DIR); no second Drive mount
BASE    = BASE_DIR
PDF_DIR = BASE_DIR / "FY_Reports"
RES_DIR = OUT_DIR
OUT     = OUT_DIR / "pdf_route"
OUT.mkdir(parents=True, exist_ok=True)

def out_path(*parts):
    """Every write in this notebook goes through here; refuses anything outside OUT."""
    p = OUT.joinpath(*parts)
    if OUT.resolve() not in p.resolve().parents and p.resolve() != OUT.resolve():
        raise RuntimeError(f"Refusing to write outside OUT: {p}")
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

FIELDS = ["total_assets", "total_equity", "current_liabilities", "inventories", "ebit"]
BS_FIELDS = ["total_assets", "total_equity", "current_liabilities", "inventories"]
EXPECTED_REPORTS = 43
EXPECTED_SCORED = 199
EXPECTED_COMPANIES = ["AIRTEL","HDFC","INFOSYS","MSIL","PAYTM","RIL","SPIL","SWIGGY","TCS","TITAN","TS","UTCL","ZOMATO"]

# Page-finder / parser constants
Y_TOL = 3.0          # pt, row grouping tolerance
HEAD_CHARS = 700     # heading window
MIN_LABELS = 4       # min distinct typical row labels on a statement page
MIN_NUMS = 15        # min number-dense tokens on a statement page
CONT_MIN_NUMS = 10   # for a continuation page

# Unit -> INR crore (1 crore = 10 million = 100 lakh)
UNIT_TO_CRORE = {"crore": 1.0, "million": 0.1, "lakh": 0.01, "billion": 100.0, "thousand": 1e-4}

# Locked v1.1 numbers, used ONLY to flag contradictions (never recomputed or changed)
LOCKED_OVERRIDE = {"exact": 137, "within_tolerance": 33, "near": 11, "mismatch": 14, "not_covered": 4}
LOCKED_OTHERS = {"overridemajority": 85.9, "yfinance": 76.4, "screener": 53.3, "nse": 27.1}

print("BASE exists  :", BASE.exists())
print("GT_DIR exists:", GT_DIR.exists(), "| PDF_DIR exists:", PDF_DIR.exists())
print("RES_DIR exists:", RES_DIR.exists())
print("OUT          :", OUT)

## 2. Inventory: 43 PDFs and 43 ground-truth JSON files

In [ ]:
def parse_name(path):
    stem = Path(path).stem
    years = sorted(set(re.findall(r"(?<!\d)(20\d{2})(?!\d)", stem)))
    toks = [t for t in re.split(r"[^A-Za-z0-9]+", stem) if t]
    if len(years) != 1 or not toks:
        return None
    return f"{toks[0].upper()}_{years[0]}", toks[0].upper(), int(years[0])

pdf_files = sorted(set(PDF_DIR.rglob("*.pdf")) | set(PDF_DIR.rglob("*.PDF")))   # recursive: PDFs may sit in per-company subfolders
gt_files  = sorted(GT_DIR.glob("*.json"))
PDF_PATH, GT_PATH, problems = {}, {}, []
for kind, files, store in (("PDF", pdf_files, PDF_PATH), ("JSON", gt_files, GT_PATH)):
    for p in files:
        r = parse_name(p)
        if r is None:
            problems.append(f"{kind}: cannot build report id from {p.name}")
            continue
        if r[0] in store:
            problems.append(f"{kind}: duplicate report id {r[0]} ({p.name})")
        store[r[0]] = p

REPORT_IDS = sorted(set(PDF_PATH) & set(GT_PATH))
only_pdf, only_gt = sorted(set(PDF_PATH) - set(GT_PATH)), sorted(set(GT_PATH) - set(PDF_PATH))
if only_pdf: problems.append(f"PDF without JSON: {only_pdf}")
if only_gt:  problems.append(f"JSON without PDF: {only_gt}")
COMPANY = {rid: rid.rsplit("_", 1)[0] for rid in REPORT_IDS}
YEAR = {rid: int(rid.rsplit("_", 1)[1]) for rid in REPORT_IDS}
if sorted(set(COMPANY.values())) != sorted(EXPECTED_COMPANIES):
    problems.append(f"Company set differs from expected: found {sorted(set(COMPANY.values()))}")
if len(REPORT_IDS) != EXPECTED_REPORTS:
    problems.append(f"Expected {EXPECTED_REPORTS} matched reports, found {len(REPORT_IDS)}")

print(f"PDFs: {len(pdf_files)} | JSON: {len(gt_files)} | matched report ids: {len(REPORT_IDS)}")
if problems:
    print("PROBLEMS:"); [print(" -", x) for x in problems]
    raise RuntimeError("Inventory check failed - stop and report the lines above.")
print("Inventory OK:", Counter(COMPANY.values()))

## 3. Ground truth: key values, units, and the 199-cell check

In [ ]:
def to_num(v):
    if isinstance(v, bool) or v is None: return None
    if isinstance(v, (int, float)):
        return float(v) if math.isfinite(v) else None
    if isinstance(v, str):
        s = v.strip().replace(",", "")
        neg = s.startswith("(") and s.endswith(")")
        s = s.strip("()")
        try:
            x = float(s)
        except ValueError:
            return None
        return (-x if neg else x) if math.isfinite(x) else None
    return None

def unit_from_currency(s):
    low = str(s).lower()
    for key, name in (("crore","crore"),("million","million"),("lakh","lakh"),("lac","lakh"),("billion","billion"),("thousand","thousand")):
        if key in low:
            return name
    raise RuntimeError(f"Cannot read a unit from currency string {s!r}")

rows, gt_problems = [], []
for rid in REPORT_IDS:
    with open(GT_PATH[rid], encoding="utf-8") as f:
        obj = json.load(f)
    ans = obj.get("answers")
    if not isinstance(ans, dict):
        gt_problems.append(f"{rid}: no 'answers' object"); continue
    bs_cur, is_cur = ans.get("balance_sheet_currency"), ans.get("income_statement_currency")
    for fld in FIELDS:
        raw = ans.get(fld, "<missing key>")
        num = to_num(raw)
        cur = is_cur if fld == "ebit" else bs_cur
        unit, factor, key_crore = None, None, None
        if num is not None:
            try:
                unit = unit_from_currency(cur); factor = UNIT_TO_CRORE[unit]; key_crore = num * factor
            except Exception as e:
                gt_problems.append(f"{rid}/{fld}: {e}")
        rows.append({"report_id": rid, "company": COMPANY[rid], "year": YEAR[rid], "field": fld,
                     "key_raw": raw, "key_value": num, "key_currency": cur, "key_unit": unit,
                     "key_crore": key_crore, "scored": num is not None and key_crore is not None})
GT = pd.DataFrame(rows)
if gt_problems:
    print("GT PROBLEMS:"); [print(" -", x) for x in gt_problems]
    raise RuntimeError("Ground-truth read failed - stop and report.")

n_attempted, n_scored = len(GT), int(GT.scored.sum())
print(f"Cells attempted: {n_attempted} | scored (numeric key): {n_scored} | N/A or non-numeric key: {n_attempted - n_scored}")
na = GT[~GT.scored]
print("Non-scored cells by field  :", dict(na.field.value_counts()))
print("Non-scored cells by company:", dict(na.company.value_counts()))
print("Currency strings seen:", dict(Counter(GT.key_currency.astype(str))))
if n_scored != EXPECTED_SCORED:
    raise RuntimeError(f"Scored-cell count is {n_scored}, expected {EXPECTED_SCORED}. Stop and report (see non-scored lists above).")
print(f"OK: scored-cell count is {n_scored}.")

## 4. Step A - Find the consolidated statement pages by content

In [ ]:
TIE_MARGIN = 5.0         # score points: within this of the best, the earliest page wins
WINDOW_MAX_SPAN = 7      # if BS and P&L are further apart than this, use per-page +-1 instead

SPLIT_COLUMNS = True      # v1.3: split side-by-side statement pages (RIL balance sheet; MSIL balance sheet | profit and loss). Set False to compare.
GUTTER_MIN = 14.0         # minimum width (pt) of the empty vertical band between the two blocks
SPLIT_MIN_ROWS = 8        # EACH side needs this many rows holding BOTH a text label and a comma-grouped number
DATA_NUM = re.compile(r"^\(?[-\u2212\u2013]?\d{1,3}(?:,\d{2,3})+(?:\.\d+)?\)?$")   # a comma-grouped value (not a year, not a note number)

def _group_rows(items):
    """items: (y-centre, x0, x1, text) -> [(y, row text)]; words grouped by y-centre (tolerance Y_TOL), ordered by x."""
    items = sorted(items, key=lambda w: (w[0], w[1], w[3]))
    rows, cur, ref = [], [], None
    for yc, x0, x1, txt in items:
        if cur and abs(yc - ref) <= Y_TOL:
            cur.append((x0, txt)); ref = (ref * (len(cur) - 1) + yc) / len(cur)
        else:
            if cur: rows.append((ref, " ".join(t for _, t in sorted(cur))))
            cur, ref = [(x0, txt)], yc
    if cur: rows.append((ref, " ".join(t for _, t in sorted(cur))))
    return rows

def _labelled_data_row(text):
    return bool(re.search(r"[A-Za-z]{3,}", text)) and any(DATA_NUM.match(t) for t in text.split())

def _split_side_by_side(items):
    """Return (header_rows, left_rows, right_rows) or None.
    Split ONLY when (1) an empty vertical band (>= GUTTER_MIN pt) sits in the middle 30-70% of the page below the header, and
    (2) BOTH sides hold >= SPLIT_MIN_ROWS rows with a text label AND a number. A normal statement (labels left, numbers right) fails (2)
    because its right side has numbers only."""
    if len(items) < 40: return None
    W = max(it[2] for it in items)
    rows = _group_rows(items)
    y0 = next((y for y, t in rows if sum(1 for k in t.split() if DATA_NUM.match(k)) >= 2), None)
    if y0 is None: return None
    body = [it for it in items if it[0] >= y0 - Y_TOL]
    if len(body) < 30: return None
    nbin = int(W) + 2; cover = [0] * nbin
    for _, x0, x1, _t in body:
        for k in range(max(0, int(x0)), min(nbin, int(x1) + 1)): cover[k] += 1
    lo, hi = int(0.30 * W), int(0.70 * W)
    tol = max(1, int(0.01 * len(body)))
    best, start = None, None
    for k in range(lo, hi + 2):
        empty = k <= hi and cover[k] <= tol
        if empty and start is None: start = k
        if not empty and start is not None:
            if best is None or k - start > best[1] - best[0]: best = (start, k)
            start = None
    if best is None or best[1] - best[0] < GUTTER_MIN: return None
    mid = 0.5 * (best[0] + best[1])
    left = [it for it in body if 0.5 * (it[1] + it[2]) < mid]
    right = [it for it in body if 0.5 * (it[1] + it[2]) >= mid]
    L, R = _group_rows(left), _group_rows(right)
    if sum(_labelled_data_row(t) for _, t in L) < SPLIT_MIN_ROWS or sum(_labelled_data_row(t) for _, t in R) < SPLIT_MIN_ROWS:
        return None
    head = _group_rows([it for it in items if it[0] < y0 - Y_TOL])
    return [t for _, t in head], [t for _, t in L], [t for _, t in R]

def page_rows(page, return_flag=False):
    """Rebuild visual rows. Side-by-side pages are returned as: header rows, then the whole left block, then the whole right block."""
    words = page.get_text("words")
    if not words:
        return ([], False) if return_flag else []
    rot = page.rotation_matrix if page.rotation else None
    items = []
    for w in words:
        x0, y0, x1, y1, txt = w[:5]
        if rot is not None:
            r = fitz.Rect(x0, y0, x1, y1) * rot; r.normalize()
            x0, y0, x1, y1 = r.x0, r.y0, r.x1, r.y1
        if txt.strip():
            items.append((0.5 * (y0 + y1), x0, x1, txt))
    split = _split_side_by_side(items) if SPLIT_COLUMNS else None
    if split is not None:
        rows = split[0] + split[1] + split[2]
    else:
        rows = [t for _, t in _group_rows(items)]
    return (rows, split is not None) if return_flag else rows

BS_LABELS = ["total assets", "total current liabilities", "total equity", "equity and liabilities", "inventor",
             "non current assets", "current assets", "trade receivables", "trade payables", "borrowings",
             "other equity", "share capital", "cash and cash equivalents", "total liabilities",
             "capital and liabilities", "property and assets", "deposits", "reserves"]
PL_LABELS = ["revenue from operations", "other income", "total income", "total expenses", "profit before tax",
             "finance cost", "depreciation", "tax expense", "current tax", "deferred tax", "employee benefit",
             "profit for the", "earnings per share", "other comprehensive income", "exceptional",
             "interest earned", "net interest income", "before tax"]
BANK_PL_LABELS = ["interest earned", "interest expended", "operating expenses", "provisions and contingencies", "net profit for the", "other income"]   # counted ONLY on a page that prints both interest earned and interest expended
HEAD_BS = re.compile(r"balance\s*sheet|financial\s+position", re.I)
HEAD_PL = re.compile(r"profit\s*(?:and|&|or)\s*loss|income\s+statement|statement\s+of\s+(?:comprehensive\s+)?income", re.I)
HEAD_EXCL = re.compile(r"cash\s*flows?|changes\s+in\s+equity|independent\s+auditor|auditor['\u2019]?s?\s+report|notes\s+to|table\s+of\s+contents|\bcontents\b|\bindex\b", re.I)
SUBSIDIARY_HEAD = re.compile(r"annexure|statement\s+pursuant\s+to\s+section\s*129|associates?\s+and\s+joint\s+ventures?", re.I)
NUM_DENSE = re.compile(r"^\(?[-\u2212\u2013]?\d[\d,]*(?:\.\d+)?\)?$")
SPACED = re.compile(r"(?:\b[A-Z]{1,2}\b ){3,}\b[A-Z]{1,2}\b")

def squash_spaced_caps(s):
    """'C O N S O L I D AT E D B A L A N C E S H E E T' -> 'CONSOLIDATEDBALANCESHEET' (runs of 4+ tiny caps tokens)."""
    return SPACED.sub(lambda m: m.group(0).replace(" ", ""), s)

def basis_counts(head):
    head = squash_spaced_caps(head)
    n_cons = len(re.findall(r"consolidated", head, re.I))
    n_stand = len(re.findall(r"stand[\s\-]?alone", head, re.I))
    return n_cons, n_stand

def n_dense(text):
    """Number-like tokens with at least 3 digits (so note numbers and 1-2 digit counters are not counted)."""
    return sum(1 for t in text.split() if NUM_DENSE.match(t) and len(re.sub(r"\D", "", t)) >= 3)

def score_page(text, kind, relaxed=False):
    """Return (score, info) or (None, reason). relaxed=True is used ONLY next to an already-found statement:
    'consolidated' may be absent from the heading, but a page that says standalone more than consolidated is still refused."""
    head = squash_spaced_caps(text[:HEAD_CHARS])
    n_cons, n_stand = basis_counts(head)
    if (n_cons < n_stand) if relaxed else (n_cons <= n_stand):
        return None, "basis"
    if SUBSIDIARY_HEAD.search(head):
        return None, "annexure"
    hre = HEAD_BS if kind == "bs" else HEAD_PL
    m = hre.search(head)
    if not m:
        return None, "heading"
    if HEAD_EXCL.search(head):
        return None, "excluded_heading"
    low = re.sub(r"[-\u2013\u2011]", " ", text.lower())
    labels = sum(1 for l in (BS_LABELS if kind == "bs" else PL_LABELS) if l in low)
    bank_pl = kind == "pl" and "interest expended" in low and "interest earned" in low        # v4: bank layout has no "profit before tax" row
    if bank_pl: labels += sum(1 for l in BANK_PL_LABELS if l in low and l not in PL_LABELS)
    nums = n_dense(text)
    if labels < MIN_LABELS: return None, "few_labels"
    if nums < MIN_NUMS: return None, "not_dense"
    if kind == "bs" and not any(k in low for k in ("total assets", "total current liabilities", "equity and liabilities", "capital and liabilities")):
        return None, "no_bs_anchor"
    if kind == "pl" and not (("before" in low and "tax" in low) or (bank_pl and "operating expenses" in low)):
        return None, "no_pl_anchor"
    score = 5 * (n_cons - n_stand) + 4 * labels + min(nums, 150) / 10 + (3 if m.start() < 300 else 0)
    return score, {"labels": labels, "dense": nums, "cons": n_cons, "stand": n_stand}

def is_bs_continuation(text):
    head = text[:HEAD_CHARS]
    n_cons, n_stand = basis_counts(head)
    if n_stand > n_cons: return False
    if HEAD_EXCL.search(squash_spaced_caps(head)): return False
    if HEAD_PL.search(head) and not HEAD_BS.search(head): return False
    low = text.lower()
    if not any(k in low for k in ("total assets", "total current liabilities", "equity and liabilities", "total equity")):
        return False
    return n_dense(text) >= CONT_MIN_NUMS

def is_bs_prev(text):
    """Page BEFORE the chosen balance-sheet page: stricter than a continuation. It must itself be headed as a
    consolidated balance sheet (so a standalone statement, an auditor page or a notes page is never parsed)."""
    head = squash_spaced_caps(text[:HEAD_CHARS])
    n_cons, n_stand = basis_counts(head)
    if n_cons <= n_stand: return False
    if not HEAD_BS.search(head) or HEAD_EXCL.search(head) or SUBSIDIARY_HEAD.search(head): return False
    low = text.lower()
    if not any(k in low for k in ("total assets", "equity and liabilities", "capital and liabilities", "total equity")):
        return False
    return n_dense(text) >= CONT_MIN_NUMS

def pick_best(cs, margin=TIE_MARGIN):
    cs = sorted(cs, key=lambda t: (-t[0], t[1]))
    near_top = [c for c in cs if c[0] >= cs[0][0] - margin]
    return min(near_top, key=lambda c: c[1])          # earliest page among near-ties

def scan_pdf(path):
    doc = fitz.open(str(path))
    n = len(doc); cache = {}
    splits = set()
    def get(i):
        if i not in cache:
            rows, flag = page_rows(doc[i], return_flag=True)
            cache[i] = rows
            if flag: splits.add(i)
        return cache[i]
    def text_of(i): return "\n".join(get(i))
    cands = {"bs": [], "pl": []}; empty = 0; annex = set()
    for i in range(n):
        plain = doc[i].get_text("text")
        if len(plain.strip()) < 30:
            empty += 1; continue
        if "consolidated" not in squash_spaced_caps(plain).lower():
            continue                       # cheap pre-filter: a statement page must say consolidated somewhere
        text = text_of(i)
        head_sq = squash_spaced_caps(text[:HEAD_CHARS])
        if SUBSIDIARY_HEAD.search(head_sq) and (HEAD_BS.search(head_sq) or HEAD_PL.search(head_sq)):
            annex.add(i)                       # a subsidiary / annexure STATEMENT page
        if (i - 1) in annex or (i - 2) in annex:
            continue                           # v1.3: its continuation pages are annexure pages too (RIL_2026 page 145 beat page 100)
        for kind in ("bs", "pl"):
            s, info = score_page(text, kind)
            if s is not None:
                cands[kind].append((s, i, info))
    res = {"n_pages": n, "empty_pages": empty, "top": {}, "pages": {"bs": [], "pl": []}, "rows": {},
           "fallback": {"bs": False, "pl": False}, "window": {}}
    best = {}
    for kind in ("bs", "pl"):
        cs = sorted(cands[kind], key=lambda t: (-t[0], t[1]))
        res["top"][kind] = [(round(s, 1), i + 1) for s, i, _ in cs[:3]]
        if cs:
            best[kind] = pick_best(cands[kind])[1]
    # (6) one statement found, the other not: look next to the found one (relaxed basis rule)
    for kind, other in (("pl", "bs"), ("bs", "pl")):
        if kind not in best and other in best:
            b = best[other]
            order = sorted((i for i in range(max(0, b - 3), min(n, b + 4)) if i != b),
                           key=lambda i: (abs(i - b), 0 if (i > b) == (kind == "pl") else 1))
            for i in order:
                if len(doc[i].get_text("text").strip()) < 30: continue
                s, _ = score_page(text_of(i), kind, relaxed=True)
                if s is not None:
                    best[kind] = i; res["fallback"][kind] = True; break
    # (4)/(5) pages to parse: best page plus real neighbours
    if "bs" in best:
        b = best["bs"]; pages = [b]
        if b - 1 >= 0 and is_bs_prev(text_of(b - 1)): pages.insert(0, b - 1)
        if b + 1 < n and is_bs_continuation(text_of(b + 1)): pages.append(b + 1)
        res["pages"]["bs"] = pages
    if "pl" in best:
        b = best["pl"]; pages = [b]
        for j in (b - 1, b + 1):
            if 0 <= j < n and j not in res["pages"]["bs"] and score_page(text_of(j), "pl", relaxed=True)[0] is not None:
                pages.append(j)
        res["pages"]["pl"] = sorted(pages)
    for kind in ("bs", "pl"):
        for i in res["pages"][kind]:
            res["rows"][i] = get(i)
    # (7) unit-detection window: page before the first statement page .. page after the last one
    sel = sorted(set(res["pages"]["bs"] + res["pages"]["pl"]))
    if sel:
        lo, hi = max(0, sel[0] - 1), min(n - 1, sel[-1] + 1)
        idx = list(range(lo, hi + 1)) if hi - lo <= WINDOW_MAX_SPAN else sorted({k for p in sel for k in (p - 1, p, p + 1) if 0 <= k < n})
        res["window"] = {k: get(k) for k in idx}
    res["split_pages"] = sorted(splits)
    doc.close()
    return res
print("Page finder v1.3 defined.")


## 5. Step B - Run the page finder on all 43 PDFs and save the text

In [ ]:
import time
PDFSEL, TXT_FILE = [], {}
t0 = time.time()
for k, rid in enumerate(REPORT_IDS, 1):
    res = scan_pdf(PDF_PATH[rid])
    row = {"report_id": rid, "company": COMPANY[rid], "year": YEAR[rid], "n_pages": res["n_pages"],
           "empty_pages": res["empty_pages"]}
    for kind, name in (("bs", "consolidated_balance_sheet"), ("pl", "consolidated_profit_loss")):
        pages = [i + 1 for i in res["pages"][kind]]
        row[f"{kind}_pages"] = "-".join(map(str, pages)) if pages else ""
        row[f"{kind}_found"] = bool(pages)
        row[f"{kind}_via_neighbour_fallback"] = res["fallback"][kind]
        row[f"{kind}_top3(score,page)"] = str(res["top"][kind])
        if pages:
            fn = out_path("txt", rid, f"{name}_p{'_'.join(map(str, pages))}.txt")
            lines = [f"# report_id: {rid}", f"# statement: {name.replace('_', ' ')}",
                     f"# pages (1-based): {', '.join(map(str, pages))}", f"# source pdf: {PDF_PATH[rid].name}"]
            for i in res["pages"][kind]:
                lines.append(f"# ----- page {i + 1} -----")
                lines += res["rows"][i]
            fn.write_text("\n".join(lines) + "\n", encoding="utf-8")
            TXT_FILE[(rid, kind)] = fn
    if res["window"]:
        ks = sorted(res["window"])
        fn = out_path("txt", rid, f"unit_window_p{ks[0] + 1}_{ks[-1] + 1}.txt")
        lines = [f"# report_id: {rid}", "# unit-detection window (page before first statement page .. page after last); NOT parsed for values"]
        for kk in ks:
            lines.append(f"# ----- page {kk + 1} -----"); lines += res["window"][kk]
        fn.write_text("\n".join(lines) + "\n", encoding="utf-8")
        TXT_FILE[(rid, "win")] = fn
        row["window_pages"] = f"{ks[0] + 1}-{ks[-1] + 1}"
        row["split_pages"] = ",".join(str(p + 1) for p in res["split_pages"] if p in res["window"])
    PDFSEL.append(row)
    print(f"[{k:2d}/{len(REPORT_IDS)}] {rid:14s} BS: {row['bs_pages'] or 'NOT FOUND':10s} | PL: {row['pl_pages'] or 'NOT FOUND':8s} | "
          f"win: {row.get('window_pages', '-'):7s} | split: {row.get('split_pages', '') or '-':8s} | empty: {res['empty_pages']} | {time.time()-t0:.0f}s")

PDFSEL = pd.DataFrame(PDFSEL)
PDFSEL.to_csv(out_path("pdf_page_selection.csv"), index=False)
nf = PDFSEL[~(PDFSEL.bs_found & PDFSEL.pl_found)]
print(f"\nBalance sheet found: {int(PDFSEL.bs_found.sum())}/{len(PDFSEL)} | profit and loss found: {int(PDFSEL.pl_found.sum())}/{len(PDFSEL)}")
print("Found only via neighbour fallback:", PDFSEL[(PDFSEL.bs_via_neighbour_fallback) | (PDFSEL.pl_via_neighbour_fallback)].report_id.tolist())
print("Reports with a statement page NOT found:" if len(nf) else "All statement pages found.")
if len(nf):
    display(nf[["report_id", "bs_found", "pl_found", "bs_top3(score,page)", "pl_top3(score,page)", "empty_pages"]])


## 6. Step C - Parse values from the saved text

In [ ]:
NUM_RE  = re.compile(r"^\(?[-\u2212\u2013]?\(?\d[\d,]*(?:\.\d+)?\)?$")
DASH_RE = re.compile(r"^[-\u2212\u2013\u2014]+$")
NOTE_ALT = re.compile(r"^\d{1,2}(?:\([a-zA-Z]\)|[A-Za-z])$")

def parse_num(tok):
    if DASH_RE.match(tok): return "DASH"
    neg = "(" in tok or ")" in tok or tok[:1] in "-\u2212\u2013"
    s = re.sub(r"[()\-\u2212\u2013,]", "", tok)
    try:
        return -float(s) if neg else float(s)
    except ValueError:
        return None

def pdf_clean_label(label):
    label = re.split(r"[\(\[]?\s*\b(?:refer\s+)?notes?\b", label, maxsplit=1, flags=re.I)[0]
    return label.strip(" :;.-")

def core_label(label):
    s = re.sub(r"\([^)]*\)", " ", label)
    for _ in range(3):                                   # leading enumerators: (c) / a. / 1)
        s = re.sub(r"^\s*(?:\(\s*[A-Za-z0-9]{1,4}\s*\)|[A-Za-z0-9]{1,3}[.)])\s+", "", s)
    s = re.sub(r"[^a-z ]", " ", s.lower())
    return re.sub(r"\s+", " ", s).strip()

SPLIT_GROUP = re.compile(r"(?<=\d),\s+(?=\d{2,3}(?!\d))")      # "1, 59, 629" -> "1,59,629"  (a space inside the number; years are 4 digits so excluded)
SPLIT_TAIL = re.compile(r"(\d,\d{2},\d{2})\s(\d)(?!\d)")        # "1,82,37 2"  -> "1,82,372"  (last digit split off)
SPLIT_MID = re.compile(r"(?<=\d),\s+(\d)\s(\d{2})(?!\d)")             # "60, 4 56"   -> "60,456"    (a group split in two)
def fix_split_numbers(t):
    t = SPLIT_MID.sub(r",\1\2", t)
    return SPLIT_TAIL.sub(r"\1\2", SPLIT_GROUP.sub(",", t))

def parse_row_v13(line):   # LEGACY (v1.3) row parser, verbatim; used only when PARSER_FIX is False
    t = fix_split_numbers(line).replace("\u20b9", " ").replace("`", " ")
    t = re.sub(r"\(\s+", "(", t); t = re.sub(r"\s+\)", ")", t)
    toks = t.split(); i = len(toks)
    while i > 0 and (NUM_RE.match(toks[i-1]) or DASH_RE.match(toks[i-1])):
        i -= 1
    tail, note_alt = toks[i:], False
    if i > 0 and NOTE_ALT.match(toks[i-1]) and len(tail) >= 2:
        tail = [toks[i-1]] + tail; i -= 1; note_alt = True
    label = " ".join(toks[:i])
    nums, note_dropped = tail, False
    if note_alt:
        nums, note_dropped = nums[1:], True
    elif len(nums) >= 3 and re.fullmatch(r"\d{1,2}", nums[0]):
        nums, note_dropped = nums[1:], True
    vals = [parse_num(x) for x in nums]
    return {"label": pdf_clean_label(label), "vals": vals, "n_numbers": len(nums), "note_dropped": note_dropped}


# ---------------------------------------------------------------------------------------------------------------
# v3 PARSER FIXES (general layout rules, evidence-based, key-free)
# A1 value-column rule: infer K = number of value columns per statement; current year = FIRST of the LAST K numeric tokens.
# A2 merged lines: numbers, then words, then numbers -> parse only the FIRST segment.
# ---------------------------------------------------------------------------------------------------------------
PARSER_FIX = True            # build_all-style runs below switch this; True = A1 + A2 active
STMT_K = {}                  # str(path) -> (K, source of K, K from header)
LEAD_OK = re.compile(r"^\d{1,4}$")      # a plain integer of up to 4 digits: looks like a note number or a page reference
MONTHS = "jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec"
DATE_RE = re.compile(rf"(?:(\d{{1,2}})(?:st|nd|rd|th)?[\s,.\-]*)?({MONTHS})[a-z]*\.?[\s,.\-]*(?:(\d{{1,2}})(?:st|nd|rd|th)?[\s,.\-]+)?((?:19|20)\d{{2}})", re.I)

def _clean_tokens(line):
    t = fix_split_numbers(line).replace("\u20b9", " ").replace("`", " ")
    t = re.sub(r"\(\s+", "(", t); t = re.sub(r"\s+\)", ")", t)
    return t.split()
def _is_num(tok):  return bool(NUM_RE.match(tok) or DASH_RE.match(tok))
def _is_real(tok): return bool(NUM_RE.match(tok))

def split_merged(toks):
    """A2. Return (tokens kept, was_split). Split only when: a numeric group of >= 2 tokens (with a real number) follows a label,
    then a token with >= 3 letters (real words), then more numbers. Only the FIRST group (and its label) is kept."""
    n, i, runs = len(toks), 0, []
    while i < n:
        if _is_num(toks[i]):
            j = i
            while j < n and _is_num(toks[j]): j += 1
            runs.append((i, j)); i = j
        else:
            i += 1
    for k, (a, b) in enumerate(runs):
        if b - a >= 2 and a > 0 and any(_is_real(x) for x in toks[a:b]):
            for (c, d) in runs[k + 1:]:
                if any(_is_real(x) for x in toks[c:d]) and any(len(re.sub(r"[^A-Za-z]", "", w)) >= 3 for w in toks[b:c]):
                    return toks[:b], True
            break
    return toks, False

def _tail_len(line):
    toks, _ = split_merged(_clean_tokens(line)); i = len(toks)
    while i > 0 and _is_num(toks[i - 1]): i -= 1
    return len(toks) - i, " ".join(toks[:i])

def infer_K(lines):
    """A1. K = modal count of trailing numeric tokens on 'total ...' rows (they carry no note or page column);
    needs >= 3 such rows with a >= 60% modal share and K in 2..4. Else K from the header (distinct dates in the first 15 lines);
    else 2. Returns (K, source, K from header)."""
    cnt = Counter()
    for ln in lines:
        n, lab = _tail_len(ln)
        if 2 <= n <= 4 and lab and core_label(pdf_clean_label(lab)).startswith("total"):
            cnt[n] += 1
    dates = {(m.group(1) or m.group(3) or "", m.group(2).lower(), m.group(4)) for ln in lines[:15] for m in DATE_RE.finditer(ln)}
    k_hdr = len(dates) if 2 <= len(dates) <= 4 else None
    tot = sum(cnt.values())
    if tot >= 3:
        k, c = cnt.most_common(1)[0]
        if c / tot >= 0.6:
            return k, ("total_rows_mode" if k_hdr in (None, k) else f"total_rows_mode (header says {k_hdr})"), k_hdr
    if k_hdr: return k_hdr, "header_dates", k_hdr
    return 2, "default", k_hdr

def parse_row_fixed(line, K):
    toks, merged = split_merged(_clean_tokens(line))
    i = len(toks)
    while i > 0 and _is_num(toks[i-1]): i -= 1
    tail, note_alt = toks[i:], False
    if i > 0 and NOTE_ALT.match(toks[i-1]) and len(tail) >= 2:
        tail = [toks[i-1]] + tail; i -= 1; note_alt = True
    label = " ".join(toks[:i])
    ntail = tail[1:] if note_alt else tail
    while len(ntail) > K and re.fullmatch(r"0\d{1,2}", ntail[-1]):      # v4: a footnote marker like '01' AFTER the value columns (AIRTEL: 'Profit before tax 383,985 126,790 01')
        ntail = ntail[:-1]
    flag_col, lead = False, []
    if len(ntail) >= K:                                   # A1: first of the last K numeric tokens is the current year
        nums = ntail[-K:]; extras = ntail[:-K]
        lead = extras + (tail[:1] if note_alt else [])
        note_dropped = bool(lead)
        flag_col = len(extras) > 2 or any(not LEAD_OK.match(x) for x in extras)   # leading tokens that do not look like note / page
    else:                                                 # fewer tokens than K: old behaviour (verbatim)
        nums, note_dropped = tail, False
        if note_alt:
            nums, note_dropped = nums[1:], True
        elif len(nums) >= 3 and re.fullmatch(r"\d{1,2}", nums[0]):
            nums, note_dropped = nums[1:], True
    vals = [parse_num(x) for x in nums]
    return {"label": pdf_clean_label(label), "vals": vals, "n_numbers": len(nums), "note_dropped": note_dropped,
            "vtoks": nums, "lead": lead, "flag_col": flag_col, "flag_split": merged}

def parse_row(line, K=None):
    return parse_row_fixed(line, 2 if K is None else K) if PARSER_FIX else {**parse_row_v13(line), "vtoks": None, "lead": [], "flag_col": False, "flag_split": False}

def read_statement_file(path, fix=None):
    fix = PARSER_FIX if fix is None else fix
    content, page = [], None
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.startswith("# ----- page "):
            page = int(re.search(r"page (\d+)", line).group(1)); content.append((page, None)); continue
        if line.startswith("# ") or not line.strip():
            continue
        content.append((page, line))
    K, ksrc, khdr = infer_K([l for _, l in content if l is not None]) if fix else (None, "legacy", None)
    STMT_K[str(path)] = (K, ksrc, khdr)
    out, prev = [], None
    for page, line in content:
        if line is None:
            prev = None; continue
        if fix: r = parse_row_fixed(line, K)
        else:   r = {**parse_row_v13(line), "vtoks": None, "lead": [], "flag_col": False, "flag_split": False}
        r["page"] = page; r["raw"] = line.strip()
        if not r["label"] and r["vals"] and prev is not None and not prev["vals"] and prev["label"]:
            r["label"] = prev["label"]                       # wrapped label merged from the row above
        r["core"] = core_label(r["label"])
        out.append(r); prev = r
    return out

PBT_EXACT = re.compile(r"^(?:profit|loss)(?: (?:profit|loss))? before (?:income )?taxe?s?$")
def m_total_assets(c): return c == "total assets"
def m_total_equity(c): return c.startswith("total equity") and "liabilit" not in c
def m_tcl(c):          return c.startswith("total current liabilities")
def m_inv(c):          return c.startswith("inventor")
PBT_PREFIX = re.compile(r"^(?:profit|loss)(?: (?:profit|loss))? before (?:income )?taxe?s?\b")
def pick_pbt(cands):
    """v4: with several PBT-like rows prefer (1) an exact 'profit before tax' row, (2) a row that STARTS with it (e.g. 'Profit before tax expense'), (3) the LAST candidate
    (the final pre-tax line, after exceptional items and associates) - never the first. Returns the same tuple as pick()."""
    nums = [c for c in cands if c["val"] != "DASH"]
    n_dash = sum(1 for c in cands if c["val"] == "DASH")
    distinct = {round(c["val"], 6) for c in nums}
    if not nums: return None, 0, n_dash
    ex = [c for c in nums if PBT_EXACT.match(c["core"])]
    pre = [c for c in nums if PBT_PREFIX.match(c["core"])]
    return (ex[0] if ex else (pre[-1] if pre else nums[-1])), len(distinct), n_dash

def m_pbt(c):          return "before" in c and "tax" in c and ("profit" in c or "loss" in c) and "exceptional" not in c and "after" not in c
def m_fc(c):           return c.startswith("finance cost")
MATCHERS = {"total_assets": (m_total_assets, None), "total_equity": (m_total_equity, lambda c: c == "total equity"),
            "current_liabilities": (m_tcl, None), "inventories": (m_inv, None)}

def candidates(rows, matcher):
    out = []
    for r in rows:
        if r["vals"] and matcher(r["core"]) and r["vals"][0] is not None:
            out.append({"val": r["vals"][0], "core": r["core"], "label": r["label"], "page": r["page"],
                        "raw": r["raw"], "note_dropped": r["note_dropped"],
                        "flag_col": r.get("flag_col", False), "flag_split": r.get("flag_split", False)})
    return out

def pick(cands, prefer=None):
    nums = [c for c in cands if c["val"] != "DASH"]
    n_dash = sum(1 for c in cands if c["val"] == "DASH")
    distinct = {round(c["val"], 6) for c in nums}
    chosen = None
    if nums:
        chosen = next((c for c in nums if prefer and prefer(c["core"])), nums[0])
    return chosen, len(distinct), n_dash
print("Parser defined.")
# ---- parser self-test on the exact lines quoted in the diagnosis (no key, no files) ----
def _t(line, K=2): return parse_row_fixed(line, K)
_m = _t("Inventories 10 406 53,181 54,435");                   assert _m["vals"][0] == 53181.0 and _m["lead"] == ["10", "406"] and not _m["flag_col"], _m
_r = _t("Finance Costs 29 19,571 14,584 Diluted (in C) - Before Exceptional Items 33 97.97 84.12"); assert _r["vals"][0] == 19571.0 and _r["flag_split"], _r
_r = _t("Finance Costs 29 23,118 19,571 Basic (in C) 32 - 0.62");  assert _r["vals"][0] == 23118.0 and _r["flag_split"], _r
_h = _t("Capital 1 759.69 557.97");                            assert _h["vals"][0] == 759.69
_h = _t("Reserves and surplus 2 452,982.84 287,762.33");       assert _h["vals"][0] == 452982.84
_n = _t("Total assets 1,00,000 90,000");                       assert _n["vals"][0] == 100000.0 and not _n["lead"]
_n = _t("Other equity 16 (1,250) (900)");                      assert _n["vals"][0] == -1250.0
_n = _t("Inventories 9 - -");                                  assert _n["vals"][0] == "DASH"
_n = _t("Other equity 2(b) 100 90");                           assert _n["vals"][0] == 100.0
_n = _t("Equity share capital (Face value 10 each) 2 100 90"); assert _n["vals"][0] == 100.0 and not _n["flag_split"]
_n = _t("Profit before tax 383,985 126,790 01");               assert _n["vals"][0] == 383985.0 and _n["vtoks"] == ["383,985", "126,790"], _n
_n = _t("Finance costs 18 164 243");                           assert _n["vals"][0] == 164.0
_c = lambda lab, v: {"val": v, "core": core_label(lab), "label": lab}
_p = pick_pbt([_c("Profit before share of profit of associates and tax", 10943.02), _c("Profit before Tax Expense", 10927.19)]);  assert _p[0]["val"] == 10927.19
_p = pick_pbt([_c("Profit before tax", 5.0), _c("Profit before tax (VII - VIII)", 7.0)]);  assert _p[0]["val"] == 5.0
_p = pick_pbt([_c("Profit before share of associates and tax", 1.0), _c("Profit before income tax and associates", 2.0)]); assert _p[0]["val"] == 2.0
_n = _t("Total assets 100 90 80", K=3);                        assert _n["vals"][0] == 100.0
print("Parser v3 self-test passed (MSIL, RIL x2, HDFC x2 and normal rows).")


## 7. Step D - Units

In [ ]:
UNIT_PAT = r"(crores?|millions?|lakhs?|lacs?|billions?|thousands?)"
CUR_PAT = r"(?:\u20b9|`|rs\.?|inr|us\s?\$|usd|\$)"
STRAY = r"[`\u20b9CHIJG]"          # rupee glyph mis-decoded by the font; UPPER-case only (so 'a million' never matches)
UNIT_RES = [re.compile(rf"(?P<cur>{CUR_PAT})\s*(?:in\s+)?{UNIT_PAT}\b", re.I),
            re.compile(rf"\bin\s+(?P<cur>{CUR_PAT})?\s*{UNIT_PAT}\b", re.I),
            re.compile(rf"\(\s*(?P<cur>)\s*{UNIT_PAT}\s*\)", re.I),
            re.compile(rf"(?<![A-Za-z0-9])(?P<cur>{STRAY})\s+(?:in\s+)?(?i:{UNIT_PAT})\b")]
NORM_UNIT = {"crore": "crore", "lakh": "lakh", "lac": "lakh", "million": "million", "billion": "billion", "thousand": "thousand"}

def units_in(text):
    found = set()
    for rx in UNIT_RES:
        for m in rx.finditer(text):
            cur = (m.group("cur") or "").lower().replace(" ", "")
            if "us$" in cur or "usd" in cur or cur == "$":
                continue
            word = m.group(m.lastindex).lower().rstrip("s")
            found.add(NORM_UNIT.get(word, word))
    return found

def detect_unit(text):
    """-> (unit or None, status, source). Heading block first, then the whole text of the statement's own pages."""
    head = text[:1500]
    for src, region in (("head", head), ("page", text)):
        f = units_in(region)
        if len(f) == 1: return next(iter(f)), "ok", src
        if len(f) > 1:  return None, "unit_ambiguous", src
    return None, "unit_unknown", None

def detect_unit_pooled(own_text, window_pages):
    """1) the statement's own pages decide if they say exactly one unit; two units on the own pages = ambiguous (never guessed).
       2) if the own pages say nothing, pool the whole window (page before .. page after):
          exactly one unit -> ok; several -> only if one unit has at least 2 more mentions than the next, else ambiguous."""
    unit, status, src = detect_unit(own_text)
    if status in ("ok", "unit_ambiguous"):
        return unit, status, src
    votes = Counter()
    for t in window_pages:
        for u in units_in(t):
            votes[u] += 1
    if len(votes) == 1:
        return next(iter(votes)), "ok", "window"
    if len(votes) > 1:
        (u1, c1), (_, c2) = votes.most_common(2)
        if c1 >= c2 + 2:
            return u1, "ok", "window_majority"
        return None, "unit_ambiguous", "window"
    return None, "unit_unknown", None

def statement_text(rid, kind):
    p = TXT_FILE.get((rid, kind))
    if p is None: return ""
    return "\n".join(l for l in Path(p).read_text(encoding="utf-8").splitlines() if not l.startswith("# "))

def window_texts(rid):
    p = TXT_FILE.get((rid, "win"))
    if p is None: return []
    pages, cur = [], []
    for l in Path(p).read_text(encoding="utf-8").splitlines():
        if l.startswith("# -----"):
            if cur: pages.append("\n".join(cur)); cur = []
        elif not l.startswith("# "):
            cur.append(l)
    if cur: pages.append("\n".join(cur))
    return pages

print("Unit detector v1.3 self-check:",
      detect_unit("(H crore)"), detect_unit("(All amounts in I million, unless otherwise stated)"),
      detect_unit("(I in crore)"), detect_unit("(C in crore)"), detect_unit("a million reasons"),
      detect_unit("(US$ million) only"), detect_unit("no unit here"))


## 8. Extract all 215 cells (independent of the key)

In [ ]:
def bank_total_assets(rows):
    """Bank layout (HDFC): the balance sheet prints only 'TOTAL' - once under capital and liabilities, once under property and assets.
    Total assets = a value printed on BOTH sides. Returns (value, page, raw row) or None. Used only when no 'Total assets' row exists."""
    tots = [(r["vals"][0], r["page"], r["raw"]) for r in rows if r["core"] == "total" and r["vals"] and isinstance(r["vals"][0], float)]
    for v, pg, raw in tots:
        if sum(1 for t in tots if t[0] == v) >= 2:
            return v, pg, raw
    return None

def is_bank_pl(rows):
    """Bank profit-and-loss layout (key-free, from the statement itself): prints both 'Interest earned' and 'Interest expended'."""
    return bool(rows) and any(r["core"].startswith("interest earned") for r in rows) and any(r["core"].startswith("interest expended") for r in rows)

def bank_ebit(rows):
    """v4 bank EBIT = Total income - Interest expended - Operating expenses. Total income = the last 'TOTAL' (or 'Total income') row printed
    BEFORE the 'Interest expended' row. Returns (value, label text, pages) or (None, reason, '')."""
    num = lambda r: r["vals"][0] if r["vals"] and isinstance(r["vals"][0], float) else None
    ie_i = next((i for i, r in enumerate(rows) if r["core"].startswith("interest expended") and num(r) is not None), None)
    oe = next((r for r in rows if r["core"].startswith("operating expenses") and num(r) is not None), None)
    if ie_i is None or oe is None: return None, f"interest expended {'found' if ie_i is not None else 'missing'}; operating expenses {'found' if oe else 'missing'}", ""
    ti = next((r for r in reversed(rows[:ie_i]) if (r["core"] == "total" or r["core"].startswith("total income")) and num(r) is not None), None)
    if ti is None: return None, "total income row missing", ""
    ie = rows[ie_i]
    return num(ti) - num(ie) - num(oe), f"{ti['label']} - {ie['label']} - {oe['label']}", ",".join(sorted({str(r['page']) for r in (ti, ie, oe)}))

def extract_cell(rid, fld, rows_bs, rows_pl, unit_bs, unit_pl):
    base = {"report_id": rid, "company": COMPANY[rid], "year": YEAR[rid], "field": fld, "derived": fld == "ebit",
            "status": None, "value_printed": None, "unit": None, "unit_source": None, "value_crore": None,
            "pages": "", "matched_label": "", "n_distinct_values": 0, "multi_candidate": False, "n_dash": 0, "detail": "",
            "col_flag": False, "merged_split": False, "parser_flagged": False}
    if fld == "ebit":
        rows, (unit, ustat, usrc), kind = rows_pl, unit_pl, "pl"
    else:
        rows, (unit, ustat, usrc), kind = rows_bs, unit_bs, "bs"
    if rows is None:
        base["status"] = "page_not_found"; return base
    base["unit"], base["unit_source"] = unit, usrc
    if fld == "ebit" and is_bank_pl(rows):
        val, lab, pgs = bank_ebit(rows)
        base["pages"] = pgs
        if val is None:
            base["status"] = "missing_component"; base["detail"] = "bank EBIT: " + lab; return base
        base["value_printed"], base["matched_label"] = val, lab
        base["detail"] = f"bank EBIT = total income - interest expended - operating expenses = {val:,.2f} (printed units)"
        if unit is None:
            base["status"] = ustat; return base
        base["status"] = "found"; base["value_crore"] = val * UNIT_TO_CRORE[unit]; return base
    if fld == "ebit":
        pbt, n1, d1 = pick_pbt(candidates(rows, m_pbt))
        fc, n2, d2 = pick(candidates(rows, m_fc))
        base["n_distinct_values"], base["n_dash"] = max(n1, n2), d1 + d2
        base["multi_candidate"] = n1 > 1 or n2 > 1
        base["col_flag"] = any(c.get("flag_col", False) for c in (pbt, fc) if c)
        base["merged_split"] = any(c.get("flag_split", False) for c in (pbt, fc) if c)
        base["pages"] = ",".join(sorted({str(c["page"]) for c in (pbt, fc) if c}))
        if pbt is None or fc is None:
            base["status"] = "dash_missing" if (d1 or d2) else "missing_component"
            base["detail"] = f"PBT {'found' if pbt else 'missing'}; finance costs {'found' if fc else 'missing'}"
            return base
        base["value_printed"] = pbt["val"] + fc["val"]
        base["matched_label"] = f"{pbt['label']} + {fc['label']}"
        base["detail"] = f"PBT={pbt['val']:,.2f} + finance costs={fc['val']:,.2f} (printed units)"
    else:
        matcher, prefer = MATCHERS[fld]
        c, nd, dash = pick(candidates(rows, matcher), prefer)
        base["n_distinct_values"], base["n_dash"], base["multi_candidate"] = nd, dash, nd > 1
        if c is None and fld == "total_assets":                      # v1.2 minimal bank fallback (see bank_total_assets)
            bt = bank_total_assets(rows)
            if bt is not None:
                c = {"val": bt[0], "label": "TOTAL (bank layout: same value on both sides)", "page": bt[1], "raw": bt[2]}
        if c is None:
            base["status"] = "dash_missing" if dash else "no_row"; return base
        base["col_flag"], base["merged_split"] = bool(c.get("flag_col", False)), bool(c.get("flag_split", False))
        base["value_printed"], base["matched_label"], base["pages"] = c["val"], c["label"], str(c["page"])
        base["detail"] = f"row: {c['raw'][:110]}"
    base["parser_flagged"] = bool(base["multi_candidate"] or base["col_flag"] or base["merged_split"])
    if unit is None:
        base["status"] = ustat; return base
    base["status"] = "found"
    base["value_crore"] = base["value_printed"] * UNIT_TO_CRORE[unit]
    return base

UNIT_BY = {}      # (report_id, 'bs'|'pl') -> (unit, status, source) from the existing PDF-route unit detector (also used by Part L)
def run_extraction(fix):
    """Run the whole extraction with the old parser (fix=False) or the fixed parser (fix=True). Leaves PARSER_FIX = fix."""
    global PARSER_FIX
    PARSER_FIX = fix
    EX = []
    for rid in REPORT_IDS:
        rows_bs = read_statement_file(TXT_FILE[(rid, "bs")]) if (rid, "bs") in TXT_FILE else None
        rows_pl = read_statement_file(TXT_FILE[(rid, "pl")]) if (rid, "pl") in TXT_FILE else None
        win = window_texts(rid)
        ub = detect_unit_pooled(statement_text(rid, "bs"), win) if rows_bs is not None else (None, "unit_unknown", None)
        up = detect_unit_pooled(statement_text(rid, "pl"), win) if rows_pl is not None else (None, "unit_unknown", None)
        UNIT_BY[(rid, "bs")], UNIT_BY[(rid, "pl")] = ub, up
        for fld in FIELDS:
            EX.append(extract_cell(rid, fld, rows_bs, rows_pl, ub, up))
    return pd.DataFrame(EX)

EXTRACT_OLD = run_extraction(False)          # v1.3 parser, kept ONLY as the reference for Hybrid A (reproduction check)
EXTRACT = run_extraction(True)               # v3 parser (A1 + A2): this is P from now on
EXTRACT_OLD.to_csv(out_path("pdf_extraction_oldparser.csv"), index=False)
EXTRACT.to_csv(out_path("pdf_extraction.csv"), index=False)
print("Cells extracted:", len(EXTRACT), "(old parser:", len(EXTRACT_OLD), ")")
display(pd.crosstab(EXTRACT.field, EXTRACT.status, margins=True))
print("Cells with more than one distinct matching value (multi_candidate):", int(EXTRACT.multi_candidate.sum()))
print("Cells whose row the parser flagged (multi-candidate, column ambiguity or merged-line split):", int(EXTRACT.parser_flagged.sum()))
print("Statements by inferred K:", dict(Counter(v[0] for v in STMT_K.values() if v[0] is not None)), "| K sources:", dict(Counter(v[1] for v in STMT_K.values())))


## 8b. Accounting-identity check (no answer key)

In [ ]:
def m_tel(c): return c in ("total equity and liabilities", "total liabilities and equity", "total capital and liabilities")


def run_identity(fix):
    global PARSER_FIX
    PARSER_FIX = fix
    ID = []
    for rid in REPORT_IDS:
        if (rid, "bs") not in TXT_FILE:
            ID.append({"report_id": rid, "identity": "no_bs_page"}); continue
        rows = read_statement_file(TXT_FILE[(rid, "bs")])
        ta, _, _ = pick(candidates(rows, m_total_assets))
        te, _, _ = pick(candidates(rows, m_tel))
        bt = bank_total_assets(rows)
        ta_v = ta["val"] if ta else (bt[0] if bt else None)
        te_v = te["val"] if te else (ta_v if bt and bt[0] == ta_v else None)
        if ta_v is None or te_v is None:
            st = "cannot_check"
        else:
            st = "ok" if abs(ta_v - te_v) <= 1e-6 * max(1, abs(ta_v)) + 1.0 else "FAIL"
        ID.append({"report_id": rid, "total_assets_printed": ta_v, "tot_eq_liab_printed": te_v, "identity": st})
    return pd.DataFrame(ID)

IDENT_OLD = run_identity(False)
IDENT = run_identity(True)                    # PARSER_FIX stays True
IDENT_OLD.to_csv(out_path("pdf_identity_check_oldparser.csv"), index=False)
IDENT.to_csv(out_path("pdf_identity_check.csv"), index=False)
print("Identity check, old parser:", IDENT_OLD.identity.value_counts().to_dict())
print("Identity check, fixed parser:", IDENT.identity.value_counts().to_dict())
display(IDENT[IDENT.identity != "ok"])


## 8c. Parser fixes: before / after counts (no scoring)

In [ ]:
# ===== 8c. Parser fixes: before / after (NO scoring, NO key). Counts of changed cells only =====
_B = EXTRACT_OLD.set_index(["report_id", "field"]); _A = EXTRACT.set_index(["report_id", "field"])
_J = _B[["status", "value_crore", "matched_label"]].join(_A[["status", "value_crore", "matched_label", "parser_flagged"]], lsuffix="_old", rsuffix="_new")
_o, _n = _J.value_crore_old, _J.value_crore_new
_J["change"] = np.select(
    [_o.isna() & _n.isna(), _o.isna() & _n.notna(), _o.notna() & _n.isna(), (_o - _n).abs() > 1e-9 * np.maximum(1, _n.abs())],
    ["both missing", "newly has a value", "lost its value", "value changed"], default="unchanged")
_J = _J.reset_index()
_T = pd.crosstab(_J.field, _J.change, margins=True)
print("Cells by change type, old parser -> fixed parser (199 + 16 unscored = 215 cells):"); display(_T)
_chg = _J[_J.change.isin(["value changed", "newly has a value", "lost its value"])]
display(_chg[["report_id", "field", "change", "value_crore_old", "value_crore_new", "matched_label_old", "matched_label_new"]])
print("Identity transitions (old -> new):")
display(IDENT_OLD[["report_id", "identity"]].merge(IDENT[["report_id", "identity"]], on="report_id", suffixes=("_old", "_new")).query("identity_old != identity_new"))
print("Identity totals old:", IDENT_OLD.identity.value_counts().to_dict(), "| new:", IDENT.identity.value_counts().to_dict())
_J.to_csv(out_path("parser_fix_before_after.csv"), index=False)


## 9. Step E - Score against the key (frozen v1.1 classes)

In [ ]:
def classify(pdf, key, floor):
    gap = abs(pdf - key)
    rel = gap / abs(key) if key != 0 else (0.0 if gap == 0 else math.inf)
    if gap <= 1e-6: return "exact", gap, rel
    if gap <= floor + 1e-9 or rel <= 0.001: return "within_tolerance", gap, rel
    if rel <= 0.01: return "near", gap, rel
    return "mismatch", gap, rel

def score_cells(EXT):
  S = GT[GT.scored].merge(EXT, on=["report_id", "company", "year", "field"], how="left", validate="one_to_one")
  cls, gaps, rels, floors = [], [], [], []
  for _, r in S.iterrows():
      if r["status"] != "found" or pd.isna(r["value_crore"]):
          cls.append("not_covered"); gaps.append(np.nan); rels.append(np.nan); floors.append(np.nan); continue
      floor = UNIT_TO_CRORE[r["unit"]]
      c, g, rl = classify(float(r["value_crore"]), float(r["key_crore"]), floor)
      cls.append(c); gaps.append(g); rels.append(rl); floors.append(floor)
  S["class"], S["gap_crore"], S["rel_gap"], S["floor_crore"] = cls, gaps, rels, floors
  S["agree"] = S["class"].isin(["exact", "within_tolerance"])
  S = S.rename(columns={"value_crore": "pdf_value_crore", "unit": "pdf_unit"})
  return S

SCORED_COLS = ["report_id", "company", "year", "field", "key_raw", "key_unit", "key_crore", "pdf_value_crore", "pdf_unit",
               "unit_source", "gap_crore", "rel_gap", "floor_crore", "class", "agree", "status", "n_distinct_values",
               "multi_candidate", "parser_flagged", "derived", "pages", "matched_label", "detail"]
S = score_cells(EXTRACT)                  # P = fixed PDF route
S_OLD = score_cells(EXTRACT_OLD)          # v1.3 parser: reference only (Hybrid A reproduction)
S_OLD[SCORED_COLS].to_csv(out_path("pdf_scored_cells_oldparser.csv"), index=False)
S[SCORED_COLS].to_csv(out_path("pdf_scored_cells.csv"), index=False)
assert len(S) == EXPECTED_SCORED, f"scored cells = {len(S)}"
print("Scored cells:", len(S), "(must be 199)")
print(S["class"].value_counts().to_dict())
print("Old parser (v1.3):", S_OLD["class"].value_counts().to_dict())
assert len(S_OLD) == EXPECTED_SCORED


## 10. Result tables

In [ ]:
COLS = ["Cells", "Exact", "Tolerance", "Near", "Mismatch", "Not covered", "Exact %", "With tolerance %", "Within 1%"]
def summarize(df):
    n = len(df); c = df["class"].value_counts()
    ex, tol, ne, mm, nc = (int(c.get(k, 0)) for k in ("exact", "within_tolerance", "near", "mismatch", "not_covered"))
    pct = lambda x: round(100 * x / n, 1) if n else np.nan
    return pd.Series([n, ex, tol, ne, mm, nc, pct(ex), pct(ex + tol), pct(ex + tol + ne)], index=COLS)

def breakdown(df, by, order=None):
    keys = order if order is not None else sorted(df[by].unique())
    t = pd.DataFrame({k: summarize(df[df[by] == k]) for k in keys}).T
    t.loc["TOTAL"] = summarize(df)
    t[COLS[:6]] = t[COLS[:6]].astype(int)
    t.index.name = by
    return t

T_HEAD = pd.DataFrame([summarize(S)], index=["PDF text route (all 199 scored cells)"])
T_HEAD.loc["Sensitivity: excluding HDFC cells"] = summarize(S[S.company != "HDFC"])
T_HEAD[COLS[:6]] = T_HEAD[COLS[:6]].astype(int)
T_HEAD.index.name = "Route"
T_FIELD = breakdown(S, "field", FIELDS)
T_YEAR = breakdown(S, "year", sorted(S.year.unique()))
T_COMP = breakdown(S, "company", sorted(S.company.unique()))
for title, t in (("Headline: without vs with tolerance", T_HEAD), ("By field", T_FIELD), ("By year", T_YEAR), ("By company", T_COMP)):
    print("\n" + title); display(t)

NC = S[S["class"] == "not_covered"]
print("\nNot-covered cells by reason (status) and company:")
display(pd.crosstab(NC.company, NC.status, margins=True) if len(NC) else pd.DataFrame({"note": ["none"]}))

In [ ]:
# ===== Name-collision check between Part I and Part II =====
# (a) Build-time AST result: module-level names assigned in BOTH parts (after the two renames), embedded below.
# (b) Run-time proof: every Part I name whose object changed identity while Part II ran.
import ast as _ast
_AST_OVERLAP = ['FIELDS', 'Path', 'ans', 'c', 'cls', 'f', 'json', 'math', 'na', 'p', 'pd', 'r', 'raw', 're', 'rid', 'row', 'rows', 'subprocess', 'sys', 't', 't0', 'time']
_CRITICAL = {"EVAL", "FULL", "SEL", "PREC", "RULES", "REPORTS", "COMPANIES", "CRORE", "OUT_DIR", "RAW_DIR", "BASE_DIR",
             "GT_DIR", "CORE_DIR", "GTDF", "METHODS", "FIELD_STMT", "TOL_PCT", "NEAR_PCT"}
print("AST overlap of module-level names (Part I vs Part II):", _AST_OVERLAP)
_changed = []
for _k, _v in _P1_SNAP.items():
    if _k in globals() and globals()[_k] is not _v:
        try:
            _same = bool(globals()[_k] == _v)
        except Exception:
            _same = False
        _changed.append((_k, "rebound to an equal value" if _same else "REBOUND TO A DIFFERENT VALUE"))
print("Part I names rebound while Part II ran:", _changed)
_bad = [k for k, why in _changed if k in _CRITICAL and why.startswith("REBOUND")]
assert not _bad, f"Part II overwrote critical Part I names: {_bad}"
assert "PDFSEL" in globals() and "pdf_clean_label" in globals(), "renamed PDF names missing"
assert _P1_SNAP["SEL"] is SEL and "decision" in SEL.columns, "Part I SEL (override selection table) was overwritten"
assert _P1_SNAP["clean_label"] is clean_label, "Part I clean_label was overwritten"
print("OK: no critical Part I name was overwritten; the only rebinds are loop temporaries / equal values.")

# PART L - Llama EXTRACTOR (4-bit) on the RAW rows of the saved .txt statements
One call per statement per report. Llama copies a row number and the current-year number; Python validates, applies the unit and does all arithmetic.


In [ ]:
# ===== Part L1. Load the LLM in 4-bit NF4 (needs a GPU runtime). Falls back down LLM_FALLBACKS if a load fails =====
import subprocess, sys, os, gc, time
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "bitsandbytes"], check=False)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Colab: Runtime > Change runtime type > T4 GPU, then run again from the top.")

def _get_hf_token():
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t: return t
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")
HF_TOKEN = _get_hf_token()                       # optional: only needed for gated repos
print("HF_TOKEN:", "found" if HF_TOKEN else "not set (open mirrors do not need it)")

tok = llm = LLM_LOADED = None
for _name in [LLM_MODEL] + [m for m in LLM_FALLBACKS if m != LLM_MODEL]:
    _t0 = time.time()
    try:
        tok = AutoTokenizer.from_pretrained(_name, token=HF_TOKEN)
        llm = AutoModelForCausalLM.from_pretrained(_name, device_map="auto", torch_dtype=torch.float16, token=HF_TOKEN,
            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True))
        llm.eval(); LLM_LOADED = _name
        print(f"Loaded {_name} in 4-bit NF4 on {torch.cuda.get_device_name(0)} in {time.time()-_t0:.0f}s | GPU memory {torch.cuda.memory_allocated()/1e9:.1f} GB")
        break
    except Exception as e:
        print(f"LOAD FAILED for {_name}: {type(e).__name__}: {str(e)[:200]}  -> trying the next model")
        tok = llm = None; gc.collect(); torch.cuda.empty_cache()
if llm is None:
    raise RuntimeError("No model could be loaded (see the messages above). Set HF_TOKEN in Colab secrets or change LLM_MODEL.")
if tok.pad_token is None: tok.pad_token = tok.eos_token
LLM_CTX = 4096 if "llama-2" in LLM_LOADED.lower() else 8192       # prompt budget per call; Llama-2 only has 4k
_EOS = [i for i in {tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>") if "<|eot_id|>" in tok.get_vocab() else None,
                    tok.convert_tokens_to_ids("<|im_end|>") if "<|im_end|>" in tok.get_vocab() else None} if i is not None]

def n_tok(s):
    return len(tok(s, add_special_tokens=False)["input_ids"])

def llm_chat(system, user, prefill="{", max_new_tokens=400):
    """Greedy decoding. The assistant answer is PREFILLED with '{'; returns only the generated continuation (the caller adds '{' back)."""
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + prefill
    ids = tok(text, return_tensors="pt", add_special_tokens=False).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           eos_token_id=_EOS or None, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)


In [ ]:
# ===== Part L2. Llama EXTRACTOR on RAW rows: prompts, JSON parsing, key-free validation (no model needed for this cell) =====
import json, re, time, hashlib
import numpy as np, pandas as pd

EXT_ROOT = OUT_DIR / "llm_route" / "extract"
STMT_VARS = {"bs": ["total_assets", "total_equity", "current_liabilities", "inventories"], "pl": ["profit_before_tax", "finance_costs"],
             "plbank": ["total_income", "interest_expended", "operating_expenses"]}
VARS = {"total_assets": ["total_assets"], "total_equity": ["total_equity"], "current_liabilities": ["current_liabilities"],
        "inventories": ["inventories"], "ebit": ["profit_before_tax", "finance_costs"]}        # ebit = PBT + finance costs, summed in Python
PART_KIND = {v: ("pl" if k == "plbank" else k) for k, vs in STMT_VARS.items() for v in vs}          # statement kind for the unit lookup
BANK_PARTS = STMT_VARS["plbank"]                       # bank EBIT = total_income - interest_expended - operating_expenses (Python)
MULTI_ROW = {"total_equity"}
MAX_LINE_CHARS = 220

_RULES = """You read the raw text rows of ONE consolidated financial statement. Each row is numbered like "12| label note page 1,234 1,100".
Rows are copied from a PDF, so columns can be mixed up. Rules for the number you report:
- "value" is the CURRENT-YEAR number, copied EXACTLY, character for character, from the cited row (keep commas, decimals and brackets).
- A row normally ends with the value columns, current year FIRST, then the previous year. The statement header tells you how many value columns each row has.
- IGNORE a note number or a page reference that comes BEFORE the value columns (e.g. "Inventories 10 406 53,181 54,435": 10 is the note, 406 the page, the current year is 53,181).
- Brackets mean negative: copy them, e.g. "(1,250)".
- If a row holds two statements merged on one line (text between two groups of numbers), use ONLY the FIRST group of numbers.
- Use the row number from the left of the row. If no row fits, answer null. Never invent a number. Never add or subtract numbers yourself.
- Reply with ONE JSON object and nothing else."""

_DEF_BS = """Reply with exactly this JSON shape:
{"total_assets": {"row": <number or null>, "value": "<text or null>"},
 "total_equity": {"rows": [{"row": <number>, "value": "<text>"}]},
 "current_liabilities": {"row": <number or null>, "value": "<text or null>"},
 "inventories": {"row": <number or null>, "value": "<text or null>"}}
Definitions:
- total_assets: the grand total of ALL assets (non-current plus current): the row 'Total assets'. A bank balance sheet prints only 'TOTAL' once on the liabilities side and once on the assets side: use the TOTAL on the ASSETS side. Never a sub-total such as 'Total non-current assets', 'Other financial assets' or 'Right-of-use assets'.
- total_equity: owners' equity PLUS non-controlling (minority) interest as ONE total. If a row 'Total equity' exists, use ONLY that row. If not, list in "rows" every row that must be added: the owners' equity and, if printed, the non-controlling / minority interest. For a bank: 'Capital' + 'Reserves and surplus' (+ 'Minority interest' if printed). 'Equity attributable to owners' alone is NOT enough when a non-controlling interest row exists. Never use 'Total equity and liabilities'.
- current_liabilities: the sub-total row 'Total current liabilities'. null if there is none.
- inventories: the current-asset row 'Inventories' (stock-in-trade). If the company has no inventories row (for example a bank), answer null. Do NOT choose a different row."""

_DEF_PL = """Reply with exactly this JSON shape:
{"profit_before_tax": {"row": <number or null>, "value": "<text or null>"},
 "finance_costs": {"row": <number or null>, "value": "<text or null>"}}
Definitions:
- profit_before_tax: the row 'Profit/(loss) before tax': the final pre-tax result AFTER exceptional items. NOT 'before exceptional items', NOT 'after tax'.
- finance_costs: the single TOTAL finance-cost row in the expenses (interest and borrowing costs). Not a sub-item (interest on one loan, on leases) and not finance income. null if the statement shows no single total row."""

_EX_BS = """WORKED EXAMPLE 1 (company balance sheet; the numbers belong to the example only). Value columns per row: 2.
--- page 1 ---
1| Particulars Note Page As at March 31, 2025 As at March 31, 2024
2| Inventories 9 212 3,250 2,980
3| Total current assets 8,400 7,700
4| Total assets 20,000 18,500
5| Equity share capital 15 240 100 100
6| Other equity 16 241 (1,250) (900)
7| Equity attributable to owners of the parent 8,900 8,100
8| Non-controlling interests 17 242 300 250
9| Total equity 9,200 8,350
10| Total current liabilities 5,100 4,800
11| Total equity and liabilities 20,000 18,500
Answer: {"total_assets": {"row": 4, "value": "20,000"}, "total_equity": {"rows": [{"row": 9, "value": "9,200"}]}, "current_liabilities": {"row": 10, "value": "5,100"}, "inventories": {"row": 2, "value": "3,250"}}

WORKED EXAMPLE 2 (bank balance sheet). Value columns per row: 2.
--- page 1 ---
1| CAPITAL AND LIABILITIES
2| Capital 1 100.50 90.25
3| Reserves and surplus 2 4,500.75 3,900.10
4| Minority interest 3 250.00 200.00
5| Deposits 4 40,000.00 35,000.00
6| TOTAL 50,000.00 44,000.00
7| PROPERTY AND ASSETS
8| Cash and balances with Reserve Bank of India 5 2,000.00 1,800.00
9| TOTAL 50,000.00 44,000.00
Answer: {"total_assets": {"row": 9, "value": "50,000.00"}, "total_equity": {"rows": [{"row": 2, "value": "100.50"}, {"row": 3, "value": "4,500.75"}, {"row": 4, "value": "250.00"}]}, "current_liabilities": {"row": null, "value": null}, "inventories": {"row": null, "value": null}}"""

_EX_PL = """WORKED EXAMPLE (profit and loss; the numbers belong to the example only). Value columns per row: 2.
--- page 1 ---
1| Revenue from operations 22 40,000 35,000
2| Profit before exceptional items and tax 4,100 3,600
3| Exceptional items (net) 28 (250) -
4| Profit/(loss) before tax 3,850 3,600
5| Finance costs 29 1,150 1,010 Basic (in Rs) 32 - 6.62
Answer: {"profit_before_tax": {"row": 4, "value": "3,850"}, "finance_costs": {"row": 5, "value": "1,150"}}
(Row 5 is a merged line: only the first group of numbers counts. A loss would be copied with brackets, e.g. "(250)".)"""

_DEF_PLBANK = """This is a BANK profit and loss account (it prints Interest earned and Interest expended). Reply with exactly this JSON shape:
{"total_income": {"row": <number or null>, "value": "<text or null>"},
 "interest_expended": {"row": <number or null>, "value": "<text or null>"},
 "operating_expenses": {"row": <number or null>, "value": "<text or null>"}}
Definitions:
- total_income: the TOTAL of the INCOME section (interest earned plus other income): the first row labelled TOTAL (or 'Total income'), printed BEFORE the expenditure section. Not the TOTAL of the expenditure section.
- interest_expended: the row 'Interest expended'.
- operating_expenses: the row 'Operating expenses' (the single row, not its (i) and (ii) parts and not 'Provisions and contingencies').
Python computes EBIT = total income - interest expended - operating expenses; do not compute it yourself."""

_EX_PLBANK = """WORKED EXAMPLE (bank profit and loss; the numbers belong to the example only). Value columns per row: 2.
--- page 1 ---
1| I. INCOME
2| Interest earned 13 7,000.50 6,100.25
3| Other income 14 900.00 800.00
4| TOTAL 7,900.50 6,900.25
5| II. EXPENDITURE
6| Interest expended 15 3,200.00 2,700.00
7| Operating expenses 16 1,800.25 1,600.00
8| Provisions and contingencies 17 500.00 450.00
9| TOTAL 5,500.25 4,750.00
Answer: {"total_income": {"row": 4, "value": "7,900.50"}, "interest_expended": {"row": 6, "value": "3,200.00"}, "operating_expenses": {"row": 7, "value": "1,800.25"}}"""

SYS_PROMPT = {"bs": _RULES + "\n\n" + _DEF_BS + "\n\n" + _EX_BS, "pl": _RULES + "\n\n" + _DEF_PL + "\n\n" + _EX_PL,
              "plbank": _RULES + "\n\n" + _DEF_PLBANK + "\n\n" + _EX_PLBANK}
MAX_NEW = {"bs": 420, "pl": 200, "plbank": 260}

def is_bank_text(lines):
    """Bank P&L layout from the numbered raw lines (key-free): both 'interest earned' and 'interest expended' are printed."""
    low = [raw.lower() for _, _, raw in lines]
    return any(l.startswith("interest earned") or " interest earned" in l for l in low) and any(l.startswith("interest expended") or " interest expended" in l for l in low)

def numbered_lines(path):
    """The saved .txt as numbered RAW lines: [(line_no, page, raw)]. Same line filter as read_statement_file."""
    out, page, n = [], None, 0
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.startswith("# ----- page "):
            page = int(re.search(r"page (\d+)", line).group(1)); continue
        if line.startswith("# ") or not line.strip(): continue
        n += 1; out.append((n, page, line.strip()))
    return out

def build_user_prompt(rid, kind, lines, K, budget_tokens):
    """Numbered raw lines under a token budget. Order of shortening: cap each line at MAX_LINE_CHARS; drop rows without any digit
    (pure headings); cut from the end. Returns (prompt, info) with info['truncated'] flagged."""
    head = (f"Statement: consolidated {'balance sheet' if kind == 'bs' else 'profit and loss statement'} of report {rid}.\n"
            f"Value columns per row in this statement: {K} (current year first).\nRows:\n")
    tail = "\nReturn the JSON object now."
    items = [(n, page, f"{n}| {raw[:MAX_LINE_CHARS]}") for n, page, raw in lines]
    cost = lambda it: n_tok(it[2]) + 1
    base = n_tok(head + tail) + 8
    total = base + sum(cost(it) for it in items)
    dropped_headings = cut_tail = 0
    if total > budget_tokens:                                   # 1) drop rows without digits
        keep = [it for it in items if re.search(r"\d", it[2].split("| ", 1)[-1])]
        dropped_headings = len(items) - len(keep); items = keep
        total = base + sum(cost(it) for it in items)
    while total > budget_tokens and items:                      # 2) cut from the end
        total -= cost(items.pop()); cut_tail += 1
    out, last = [], None
    for n, page, text in items:
        if page != last: out.append(f"--- page {page} ---"); last = page
        out.append(text)
    info = {"n_lines": len(lines), "n_sent": len(items), "dropped_headings": dropped_headings, "cut_tail": cut_tail,
            "truncated": bool(dropped_headings or cut_tail)}
    return head + "\n".join(out) + tail, info

def parse_llm_json(gen):
    """The assistant answer was prefilled with '{'. Returns (dict or None, status)."""
    s = "{" + (gen or "")
    try:
        obj, _ = json.JSONDecoder().raw_decode(s)
    except Exception:
        j = s.rfind("}")
        try:   obj = json.loads(s[: j + 1]) if j > 0 else None
        except Exception: obj = None
    return (obj, "ok") if isinstance(obj, dict) else (None, "invalid_json")

def norm_answer(obj, var):
    """-> ('null', None) | ('items', [{'row':..,'value':..}, ...]) | ('bad_schema', None)."""
    a = obj.get(var) if isinstance(obj, dict) else None
    if a is None: return "null", None
    if isinstance(a, dict):
        if "rows" in a: items = a["rows"]
        elif "row" in a: items = [{"row": a.get("row"), "value": a.get("value")}]
        else: return "bad_schema", None
    elif isinstance(a, list): items = a
    else: return "bad_schema", None
    if items is None or (isinstance(items, list) and not items): return "null", None
    if not isinstance(items, list) or not all(isinstance(x, dict) for x in items): return "bad_schema", None
    items = [x for x in items if not (x.get("row") is None and x.get("value") is None)]
    return ("null", None) if not items else ("items", items)

def canon(s):
    return re.sub(r"[()\s,\-\u2212\u2013]", "", str(s))

def check_value(raw_line, K, llm_value):
    """B4. The copied number must (1) be a number, (2) be a token of the cited raw line, (3) be the CURRENT-YEAR value column
    (first of the last K tokens of the first segment), not the note, page or previous-year column. The sign comes from the line's own
    brackets. Returns (ok, signed value or None, reason)."""
    if llm_value is None: return False, None, "value_missing"
    cv = canon(fix_split_numbers(str(llm_value)).strip())
    if not re.fullmatch(r"\d+(\.\d+)?", cv): return False, None, "not_a_number"
    if cv not in {canon(t) for t in _clean_tokens(raw_line) if _is_num(t)}: return False, None, "not_in_line"
    row = parse_row_fixed(raw_line, K)
    vt = row["vtoks"]
    if vt and canon(vt[0]) == cv:
        v = parse_num(vt[0])
        return (True, float(v), "ok") if isinstance(v, float) else (False, None, "dash")
    if cv in {canon(t) for t in vt[1:]}:     return False, None, "previous_year_column"
    if cv in {canon(t) for t in row["lead"]}: return False, None, "note_or_page_column"
    return False, None, "not_in_first_segment"

def validate_variable(var, status, items, lines_by_no, K):
    """-> record {status, value_printed, rows, reason}. status in valid | null | ungrounded."""
    if status == "null": return {"status": "null", "value_printed": None, "rows": [], "reason": "model answered null"}
    if status == "bad_schema": return {"status": "ungrounded", "value_printed": None, "rows": [], "reason": "bad_schema"}
    if len(items) > 1 and var not in MULTI_ROW:
        return {"status": "ungrounded", "value_printed": None, "rows": [], "reason": "several_rows_for_single_row_variable"}
    seen, total, used = set(), 0.0, []
    for it in items:
        r = it.get("row")
        if isinstance(r, str) and r.strip().isdigit(): r = int(r.strip())
        if isinstance(r, bool) or not isinstance(r, int) or r not in lines_by_no:
            return {"status": "ungrounded", "value_printed": None, "rows": used, "reason": f"bad_row({r!r})"}
        if r in seen: return {"status": "ungrounded", "value_printed": None, "rows": used, "reason": "duplicate_row"}
        seen.add(r)
        ok, v, why = check_value(lines_by_no[r], K, it.get("value"))
        if not ok: return {"status": "ungrounded", "value_printed": None, "rows": used + [r], "reason": f"row {r}: {why}"}
        total += v; used.append(r)
    return {"status": "valid", "value_printed": total, "rows": used, "reason": "ok"}

def _label(raw, K):
    return core_label(parse_row_fixed(raw, K)["label"])

def _rowno(r):
    if isinstance(r, str) and r.strip().isdigit(): r = int(r.strip())
    return r if isinstance(r, int) and not isinstance(r, bool) else None

def repair_equity(items, lines_by_no, K):
    """Key-free completeness rules for total_equity (Python, applied to the model's rows; every row is still validated afterwards):
    (1) if a 'Total equity' row (not '... and liabilities') is among the rows, keep ONLY that row (it already contains minority interest: no double count);
    (2) bank layout: if 'Reserves and surplus' is listed but the 'Capital' row printed just above it is not, add that row.
    Returns (items, note)."""
    try:
        labs = {}
        for it in items:
            n = _rowno(it.get("row"))
            if n in lines_by_no: labs[n] = _label(lines_by_no[n], K)
        if len(items) > 1:
            for it in items:
                n = _rowno(it.get("row"))
                if n in labs and labs[n].startswith("total equity") and "liabilit" not in labs[n]:
                    return [it], "kept only the Total equity row (no minority-interest double count)"
        res = [n for n, l in labs.items() if l.startswith("reserves and surplus")]
        if res and not any(l in ("capital", "share capital", "equity share capital", "capital stock") for l in labs.values()):
            top = min(res)
            for n in range(top - 1, max(0, top - 6), -1):
                if n in lines_by_no and _label(lines_by_no[n], K) in ("capital", "share capital", "equity share capital"):
                    vt = parse_row_fixed(lines_by_no[n], K)["vtoks"]
                    if vt: return items + [{"row": n, "value": vt[0]}], f"added the Capital row {n} the model left out"
                    break
    except Exception:
        pass
    return items, None

def run_statement(rid, kind, path, K, use_model=True):
    """One call per statement (checkpointed to Drive per model). Returns (checkpoint record, {var: validation record}, called_model)."""
    cp = EXT_ROOT / re.sub(r"[^A-Za-z0-9._-]", "_", LLM_LOADED) / f"{rid}_{kind}.json"
    cp.parent.mkdir(parents=True, exist_ok=True)
    lines = numbered_lines(path); by_no = {n: raw for n, _, raw in lines}
    if kind == "pl" and is_bank_text(lines): kind = "plbank"                       # bank P&L: its own prompt and variables
    cp = cp.with_name(f"{rid}_{kind}.json")
    called = False
    if cp.exists():
        rec = json.loads(cp.read_text(encoding="utf-8"))
    else:
        budget = LLM_CTX - n_tok(SYS_PROMPT[kind]) - MAX_NEW[kind] - 120
        prompt, info = build_user_prompt(rid, "pl" if kind == "plbank" else kind, lines, K, budget)
        gen = llm_chat(SYS_PROMPT[kind], prompt, "{", MAX_NEW[kind]); called = True
        obj, st = parse_llm_json(gen)
        rec = {"model": LLM_LOADED, "report_id": rid, "kind": kind, "K": K, **info, "raw_output": gen, "json_status": st, "parsed": obj}
        cp.write_text(json.dumps(rec), encoding="utf-8")
    out = {}
    for var in STMT_VARS[kind]:
        if rec["json_status"] != "ok":
            out[var] = {"status": "invalid_json", "value_printed": None, "rows": [], "reason": "invalid_json"}; continue
        st, items = norm_answer(rec["parsed"], var)
        note = None
        if var == "total_equity" and st == "items": items, note = repair_equity(items, by_no, K)
        out[var] = validate_variable(var, st, items, by_no, K)
        if note and out[var]["status"] == "valid": out[var]["reason"] = "ok; " + note
        out[var]["label"] = " + ".join(by_no[r][:60] for r in out[var]["rows"] if r in by_no)
    return rec, out, called, kind
print("Llama extractor helpers defined. Prompt tokens are measured at run time against the loaded model's context.")


In [ ]:
# ===== Part L3. Run the Llama extractor: ONE call per statement per report (checkpointed; resumes on rerun) =====
from collections import Counter
LLX, CKPT, BANK_PL, n_calls, _t0 = {}, {}, {}, 0, time.time()
for k, rid in enumerate(REPORT_IDS, 1):
    for kind in ("bs", "pl"):
        if (rid, kind) not in TXT_FILE:
            for var in STMT_VARS[kind]:
                LLX[(rid, var)] = {"status": "no_text", "value_printed": None, "rows": [], "reason": "statement .txt not saved by the page finder"}
            continue
        path = TXT_FILE[(rid, kind)]
        read_statement_file(path, fix=True)                         # fills STMT_K for this file
        K = STMT_K[str(path)][0] or 2
        rec, out, called, kind_used = run_statement(rid, kind, path, K)
        n_calls += int(called); CKPT[(rid, kind_used)] = rec
        if kind == "pl": BANK_PL[rid] = kind_used == "plbank"
        for var, v in out.items(): LLX[(rid, var)] = v
    if k % 5 == 0 or k == len(REPORT_IDS):
        print(f"[{k:2d}/{len(REPORT_IDS)}] {rid:14s} model calls this run: {n_calls} | {time.time()-_t0:.0f}s")

# ---- status counts (per statement-variable pick, like the 258 picks of v2) ----
_sc = Counter(v["status"] for v in LLX.values())
print(f"\nLlama extractor: {n_calls} new model calls, {time.time()-_t0:.0f}s | model {LLM_LOADED}")
print(f"Pick-level status: valid {_sc.get('valid',0)} | invalid JSON {_sc.get('invalid_json',0)} | ungrounded {_sc.get('ungrounded',0)} | null {_sc.get('null',0)} | no text {_sc.get('no_text',0)} | total {len(LLX)}")
_why = Counter(re.sub(r"^row \d+: |\(.*\)$", "", v["reason"]) for v in LLX.values() if v["status"] == "ungrounded")
print("Ungrounded by reason:", dict(_why))
print("Statements truncated to fit the context:", [f"{r}/{kd}" for (r, kd), c in CKPT.items() if c.get("truncated")] or "none")
LOG = pd.DataFrame([{"report_id": r, "variable": v, **{kk: (",".join(map(str, x)) if isinstance(x, list) else x) for kk, x in d.items()}} for (r, v), d in LLX.items()])
(OUT_DIR / "llm_route").mkdir(parents=True, exist_ok=True)
LOG.to_csv(OUT_DIR / "llm_route" / "llama_extraction_log.csv", index=False)

# ---- cell level: value in INR crore, EBIT = PBT + finance costs computed in Python; the unit comes from the PDF-route unit detector ----
llp_rows = []
for r in S.itertuples():
    bank = r.field == "ebit" and BANK_PL.get(r.report_id, False)
    parts = BANK_PARTS if bank else VARS[r.field]; kind = PART_KIND[parts[0]]
    unit, ustat, _ = UNIT_BY.get((r.report_id, kind), (None, "unit_unknown", None))
    recs = [LLX.get((r.report_id, p), {"status": "no_text", "reason": "none", "rows": []}) for p in parts]
    bad = next((x["status"] for x in recs if x["status"] != "valid"), None)
    st = bad or ("no_unit" if unit is None else "valid")
    if st == "valid":
        v_ = (recs[0]["value_printed"] - recs[1]["value_printed"] - recs[2]["value_printed"]) if bank else sum(x["value_printed"] for x in recs)    # bank EBIT: total income - interest expended - operating expenses
        val = v_ * UNIT_TO_CRORE[unit]
    else:
        val = None
    cls = "not_covered" if val is None else classify(float(val), float(r.key_crore), UNIT_TO_CRORE[unit])[0]     # existing frozen classify: SCORING ONLY
    llp_rows.append({"report_id": r.report_id, "field": r.field, "llp_value_crore": val if val is not None else np.nan, "llp_status": st,
                     "llp_class": cls, "llp_rows": " + ".join(x.get("label", "-") for x in recs),
                     "llp_reason": "; ".join(x.get("reason", "") for x in recs if x["status"] != "valid")})
LLP = pd.DataFrame(llp_rows)
LLP.to_csv(OUT_DIR / "llm_route" / "llama_extractor_scored.csv", index=False)
print("Status by cell:", LLP.llp_status.value_counts().to_dict())
print("Class by cell (Llama extractor alone):", LLP.llp_class.value_counts().to_dict())


# PART III - Integration (frozen v4 rule, key-free)

In [ ]:
# ===== PART III. Integration (frozen v4 rule, key-free) =====
import json, hashlib, numpy as np, pandas as pd
from IPython.display import display
HYBRID = "V4"
_frozen = json.loads(V4_RULE_PATH.read_text(encoding="utf-8"))
assert _frozen["sha256"] == hashlib.sha256(json.dumps([V4_RULE_TEXT, V4_RULE_PARAMS], sort_keys=True).encode()).hexdigest(), "rule text changed after it was frozen"
print("Frozen rule verified (file written before the results; text unchanged):", V4_RULE_PATH, "\n" + "\n".join(V4_RULE_TEXT))

_CLASSES = ("exact", "within_tolerance", "near", "mismatch")
_ov = EVAL[EVAL["method"] == "override"].copy()
assert not _ov.duplicated(["report_id", "field"]).any(), "duplicate override rows"
_f = lambda x: float(x) if x is not None and not pd.isna(x) else np.nan
_ov["ovr_value_crore"] = _ov["value_crore"].map(_f)
_ov = _ov.rename(columns={"source_used": "ovr_source", "class": "ovr_class", "no_value_reason": "ovr_reason"})[["report_id", "field", "ovr_value_crore", "ovr_source", "ovr_class", "ovr_reason"]]
_have = set(zip(_ov.report_id, _ov.field))
_missing = [(r.report_id, r.field) for r in S.itertuples() if (r.report_id, r.field) not in _have]
_oc = _ov.merge(S[["report_id", "field"]], on=["report_id", "field"])["ovr_class"].value_counts()
_now = {k: int(_oc.get(k, 0)) for k in _CLASSES}
_now["not_covered"] = len(_missing) + int(S.merge(_ov, on=["report_id", "field"])["ovr_class"].isna().sum())
print("\nINTEGRITY CHECK - override in this run vs locked v1.1:\n  this run:", _now, "\n  locked  :", LOCKED_OVERRIDE)
print("  OK: override reproduces the locked v1.1 counts." if _now == LOCKED_OVERRIDE else "  *** WARNING: override counts DIFFER from the locked v1.1 results. Continuing. ***")

INT = S[["report_id", "company", "year", "field", "key_crore", "pdf_value_crore", "class", "status", "parser_flagged", "matched_label"]].rename(columns={"class": "pdf_class", "status": "pdf_status"})
INT = INT.merge(S_OLD[["report_id", "field", "pdf_value_crore", "class", "status"]].rename(columns={"pdf_value_crore": "pdfo_value_crore", "class": "pdfo_class", "status": "pdfo_status"}), on=["report_id", "field"], how="left")
INT = INT.merge(IDENT[["report_id", "identity"]].rename(columns={"identity": "pdf_identity"}), on="report_id", how="left")
INT = INT.merge(IDENT_OLD[["report_id", "identity"]].rename(columns={"identity": "pdfo_identity"}), on="report_id", how="left")
INT["pdf_identity"] = INT["pdf_identity"].fillna("no_bs_page"); INT["pdfo_identity"] = INT["pdfo_identity"].fillna("no_bs_page")
INT = INT.merge(_ov, on=["report_id", "field"], how="left")
INT = INT.merge(LLP, on=["report_id", "field"], how="left")
_dec = FULL.set_index(["report_id", "field"])["decision"].to_dict()
INT["ovr_class"] = INT["ovr_class"].where(INT["ovr_class"].isin(_CLASSES), "not_covered")
INT["ovr_has"] = INT["ovr_class"].isin(_CLASSES)
INT["ovr_source"] = INT["ovr_source"].where(INT["ovr_has"], None)
INT["ovr_reason"] = [None if h else (rs if isinstance(rs, str) and rs else _dec.get((a, b), "no_override_row")) for h, rs, a, b in zip(INT.ovr_has, INT.ovr_reason, INT.report_id, INT.field)]
INT["pdf_found"]  = (INT.pdf_status == "found") & (INT.pdf_identity != "FAIL") & INT.pdf_value_crore.notna() & INT.pdf_class.isin(_CLASSES)
INT["pdfo_found"] = (INT.pdfo_status == "found") & (INT.pdfo_identity != "FAIL") & INT.pdfo_value_crore.notna() & INT.pdfo_class.isin(_CLASSES)
INT["llp_has"] = (INT.llp_status == "valid") & INT.llp_value_crore.notna() & INT.llp_class.isin(_CLASSES)
INT["llp_class"] = INT["llp_class"].where(INT["llp_has"], "not_covered")
INT["parser_flagged"] = INT["parser_flagged"].fillna(False).astype(bool)

def _agree(a, b, rel):
    return not (pd.isna(a) or pd.isna(b)) and abs(a - b) <= rel * max(abs(a), abs(b), 1e-9)

def _apply(r):
    """Frozen v4 rule. Returns (source, value, class, decision text, review flag)."""
    P, L, O = r.pdf_value_crore, r.llp_value_crore, r.ovr_value_crore
    ovr = (f"override: {r.ovr_source}", O, r.ovr_class)
    if r.pdf_found and r.llp_has:
        if _agree(P, L, 0.001): return "pdf", P, r.pdf_class, "R1 P and L agree (<=0.1%)", False
        if r.ovr_has:
            sp, sl = _agree(P, O, 0.01), _agree(L, O, 0.01)
            if sp and not sl: return "pdf", P, r.pdf_class, "R2a O supports P only", False
            if sl and not sp: return "llama", L, r.llp_class, "R2a O supports L only", False
        if r.parser_flagged: return "llama", L, r.llp_class, "R2b L used: O supports neither/both/absent and P came from a flagged row", False
        return "pdf", P, r.pdf_class, "R2c P kept: O supports neither/both/absent, P row not flagged (REVIEW)", True
    if r.pdf_found or r.llp_has:
        src, X, xc = ("pdf", P, r.pdf_class) if r.pdf_found else ("llama", L, r.llp_class)
        if not r.ovr_has: return src, X, xc, f"R3 only {src}; O missing", False
        if _agree(X, O, 0.01): return src, X, xc, f"R3 only {src}; O agrees", False
        if not _agree(X, O, 0.10): return src, X, xc, f"R3b only {src}; O differs by >10% (implausible): document value kept (REVIEW)", True
        return (*ovr, f"R3 only {src}; O differs by 1-10%: O taken", False)
    if r.ovr_has: return (*ovr, "R4 override only", False)
    return "none", np.nan, "not_covered", "R4 no route returned a value", False

def _hybrid_A(r, old):        # Hybrid A for comparison only: PDF first, override fallback
    found, val, cls = (r.pdfo_found, r.pdfo_value_crore, r.pdfo_class) if old else (r.pdf_found, r.pdf_value_crore, r.pdf_class)
    if found: return val, cls
    if r.ovr_has: return r.ovr_value_crore, r.ovr_class
    return np.nan, "not_covered"

_res = INT.apply(_apply, axis=1, result_type="expand")
INT["final_source"], INT["final_value_crore"], INT["final_class"], INT["decision"], INT["review"] = _res[0], _res[1], _res[2], _res[3], _res[4].astype(bool)
_hA = INT.apply(lambda r: _hybrid_A(r, True), axis=1, result_type="expand");  INT["hA_value_crore"], INT["hA_class"] = _hA[0], _hA[1]
_hF = INT.apply(lambda r: _hybrid_A(r, False), axis=1, result_type="expand"); INT["hAF_value_crore"], INT["hAF_class"] = _hF[0], _hF[1]
assert len(INT) == EXPECTED_SCORED and not INT.duplicated(["report_id", "field"]).any()
INT.to_csv(OUT_DIR / "final_v4_integration_199.csv", index=False)
print("\nIntegration table:", INT.shape)
print("P (fixed PDF route) found:", int(INT.pdf_found.sum()), "| L (Llama extractor) valid:", int(INT.llp_has.sum()), "| O override:", int(INT.ovr_has.sum()), "| P and L both:", int((INT.pdf_found & INT.llp_has).sum()))
print("Decision counts:", INT.decision.value_counts().to_dict())
print("Final source counts:", INT.final_source.value_counts().to_dict(), "| cells on the REVIEW list:", int(INT.review.sum()))
display(INT.head(10))


# PART IV - Final detailed accuracy

In [ ]:
# ===== PART IV. Final detailed accuracy (v4) =====
import numpy as np, pandas as pd
from IPython.display import display
H_DIR = OUT_DIR / "hybrid_tables"; H_DIR.mkdir(parents=True, exist_ok=True)
H_AGREE = ("exact", "within_tolerance")
H_COLS = ["Cells", "Exact", "Tolerance", "Near", "Mismatch", "Not covered", "Exact %", "With tolerance %", "Within 1%", "Coverage %", "Precision when covered %"]
H_VAR = {"total_assets": "Total assets", "total_equity": "Total equity", "current_liabilities": "Current liabilities", "inventories": "Inventories", "ebit": "EBIT"}
H_ORDER = list(H_VAR)
H_NAME = "Final v4 (P + L + override)"
H_METH = {"Override v1.1": "ovr_class", "PDF route v1.3 (old parser)": "pdfo_class", "PDF route fixed (P)": "pdf_class", "Llama extractor alone (L)": "llp_class",
          "Hybrid A (old parser)": "hA_class", "Hybrid A (fixed parser)": "hAF_class", H_NAME: "final_class"}
H_SCOPE = {"All cells": INT, "Excluding HDFC": INT[INT.company != "HDFC"]}
H_DEV = ["INFOSYS", "ZOMATO", "UTCL", "AIRTEL"]   # planned dev companies; NOT a clean held-out split (rules were designed after seeing all reports)
H_SCOPE_EXT = dict(H_SCOPE, **{"HDFC only": INT[INT.company == "HDFC"], "Dev companies (4)": INT[INT.company.isin(H_DEV)], "Test companies (other 9)": INT[~INT.company.isin(H_DEV)]})
pc = lambda x, d: round(100 * x / d, 1) if d else np.nan

def h_sum(cls):
    cls = pd.Series(list(cls), dtype=object); n = len(cls); c = cls.value_counts()
    ex, tol, ne, mm, nc = (int(c.get(k, 0)) for k in ("exact", "within_tolerance", "near", "mismatch", "not_covered"))
    cov = n - nc
    return pd.Series([n, ex, tol, ne, mm, nc, pc(ex, n), pc(ex + tol, n), pc(ex + tol + ne, n), pc(cov, n), pc(ex + tol, cov)], index=H_COLS)

TABLES = {}
# 1. headline
rows = []
for sc, d in H_SCOPE_EXT.items():
    for m, col in H_METH.items():
        rows.append(pd.concat([pd.Series({"Scope": sc, "Method": m}), h_sum(d[col])]))
TABLES["1_headline"] = pd.DataFrame(rows).reset_index(drop=True)

# ---- FIRST OUTPUT: honest verdict against the aims, against Hybrid A (v2 reference) and against the previous v3 run ----
def _head(scope, method):
    r_ = TABLES["1_headline"]; r_ = r_[(r_.Scope == scope) & (r_.Method == method)].iloc[0]
    return float(r_["Exact %"]), float(r_["With tolerance %"]), float(r_["Within 1%"])
PREV_V3 = {"all_exact": 93.5, "all_tol": 94.5, "x_tol": 98.4}          # typed from the v3 run (not recomputed here)
_f_ex, _f_tol, _f_w1 = _head("All cells", H_NAME);          _fx_ex, _fx_tol, _fx_w1 = _head("Excluding HDFC", H_NAME)
_a_ex, _a_tol, _a_w1 = _head("All cells", "Hybrid A (old parser)"); _ax_ex, _ax_tol, _ax_w1 = _head("Excluding HDFC", "Hybrid A (old parser)")
_h_ex, _h_tol, _h_w1 = _head("HDFC only", H_NAME)
BELOW_A = _f_tol < _a_tol; BELOW_V3 = _f_tol < PREV_V3["all_tol"]
RECOMMENDED = "Hybrid A (v2 reference, old parser)" if BELOW_A else ("v3 (previous notebook)" if BELOW_V3 else H_NAME)
_t = lambda ok: "PASS" if ok else "FAIL"
BANNER = [f"v4 final: all 199 cells  exact {_f_ex}%  tolerance {_f_tol}%  within 1% {_f_w1}%   |   excluding HDFC  exact {_fx_ex}%  tolerance {_fx_tol}%  within 1% {_fx_w1}%   |   HDFC only  exact {_h_ex}%  tolerance {_h_tol}%",
          f"Hybrid A (old parser, v2 reference): all cells exact {_a_ex}%  tolerance {_a_tol}%  within 1% {_a_w1}%   |   excluding HDFC tolerance {_ax_tol}%",
          f"v3 (previous run): all cells exact {PREV_V3['all_exact']}%  tolerance {PREV_V3['all_tol']}%   |   excluding HDFC tolerance {PREV_V3['x_tol']}%",
          f"Aims: all cells exact >= 95%: {_t(_f_ex >= 95)} ({_f_ex}) | all cells tolerance >= 97%: {_t(_f_tol >= 97)} ({_f_tol}) | excluding HDFC exact >= 98%: {_t(_fx_ex >= 98)} ({_fx_ex})",
          ("*** v4 IS BELOW HYBRID A on all 199 cells (tolerance %). RECOMMENDED FINAL = HYBRID A. ***" if BELOW_A else
           ("*** v4 IS BELOW THE PREVIOUS v3 RUN on all 199 cells (tolerance %). RECOMMENDED FINAL = v3. ***" if BELOW_V3 else
            f"v4 is not below Hybrid A ({_a_tol}%) or v3 ({PREV_V3['all_tol']}%) on all 199 cells (tolerance {_f_tol}%). Check the REVIEW list before using it.")),
          "Dev/test is NOT a clean held-out split: the v3 rules were designed after seeing the diagnosis of all 43 reports and the v4 changes after seeing the v3 results. True generalisation needs new reports."]
print("=" * 110); print("\n".join(BANNER)); print("=" * 110)
# 2. per variable
rows = []
for sc, d in H_SCOPE.items():
    for m, col in H_METH.items():
        for f in H_ORDER:
            rows.append(pd.concat([pd.Series({"Scope": sc, "Method": m, "Variable": H_VAR[f]}), h_sum(d[d.field == f][col])]))
TABLES["2_per_variable"] = pd.DataFrame(rows).reset_index(drop=True)
# 3. side by side
rows = []
for sc, d in H_SCOPE.items():
    for f in H_ORDER + ["ALL"]:
        dd = d if f == "ALL" else d[d.field == f]
        r = {"Scope": sc, "Variable": "All variables" if f == "ALL" else H_VAR[f], "Cells": len(dd)}
        for m, col in H_METH.items():
            s = h_sum(dd[col]); r[f"{m}: tol %"] = s["With tolerance %"]; r[f"{m}: within 1%"] = s["Within 1%"]
        rows.append(r)
TABLES["3_side_by_side"] = pd.DataFrame(rows)
# 4. company x variable, variable x year (agreeing / scored, hybrid)
INT["_ag"] = INT.final_class.isin(H_AGREE)
def h_matrix(rowkey, colkey, colorder, rowlabel):
    g = INT.groupby([rowkey, colkey])["_ag"].agg(["sum", "count"])
    out = pd.DataFrame(index=sorted(INT[rowkey].unique()), columns=colorder + ["TOTAL"], dtype=object)
    for r_ in out.index:
        for c_ in colorder:
            out.loc[r_, c_] = f"{int(g.loc[(r_, c_), 'sum'])}/{int(g.loc[(r_, c_), 'count'])}" if (r_, c_) in g.index else "-"
        sub = INT[INT[rowkey] == r_]; out.loc[r_, "TOTAL"] = f"{int(sub._ag.sum())}/{len(sub)}"
    out.loc["TOTAL"] = [f"{int(INT[INT[colkey] == c_]._ag.sum())}/{int((INT[colkey] == c_).sum())}" for c_ in colorder] + [f"{int(INT._ag.sum())}/{len(INT)}"]
    out.index.name = rowlabel; return out.reset_index()
TABLES["4a_company_x_variable"] = h_matrix("company", "field", H_ORDER, "Company").rename(columns=H_VAR)
_yrs = sorted(INT.year.unique())
TABLES["4b_variable_x_year"] = h_matrix("field", "year", _yrs, "Variable").assign(Variable=lambda d: d.Variable.map(lambda x: H_VAR.get(x, x))).rename(columns=str)
# 5. source of the final value
rows = []
for s in ["pdf", "llama", "override: yfinance", "override: screener", "override: nse", "none"]:
    d = INT[INT.final_source == s]; ag = int(d.final_class.isin(H_AGREE).sum())
    rows.append({"Final value from": s, "Cells": len(d), "Agreeing": ag, "Agreement %": pc(ag, len(d)) if s != "none" else np.nan,
                 "Within 1%": int(d.final_class.isin(H_AGREE + ("near",)).sum())})
TABLES["5_source_of_final_value"] = pd.DataFrame(rows)
# 6. both returned a value: Llama vs PDF, Llama vs override, PDF vs override
def h_cross(df, ca, cb, na, nb_, flag_a, flag_b, key):
    both = df[df[flag_a] & df[flag_b]].copy(); both["x"] = both[ca].isin(H_AGREE); both["y"] = both[cb].isin(H_AGREE); rows = []
    for f in H_ORDER + ["ALL"]:
        d = both if f == "ALL" else both[both.field == f]
        rows.append({"Variable": "All variables" if f == "ALL" else H_VAR[f], "Both returned a value": len(d),
                     f"{na} right, {nb_} right": int((d.x & d.y).sum()), f"{na} right, {nb_} wrong": int((d.x & ~d.y).sum()),
                     f"{na} wrong, {nb_} right": int((~d.x & d.y).sum()), f"{na} wrong, {nb_} wrong": int((~d.x & ~d.y).sum())})
    TABLES[key] = pd.DataFrame(rows)
h_cross(INT, "llp_class", "pdf_class", "Llama", "PDF", "llp_has", "pdf_found", "6a_llama_vs_pdf")
h_cross(INT, "llp_class", "ovr_class", "Llama", "override", "llp_has", "ovr_has", "6b_llama_vs_override")
h_cross(INT, "pdf_class", "ovr_class", "PDF", "override", "pdf_found", "ovr_has", "6c_pdf_vs_override")
# 7. outcome of every cell
def h_pdf_reason(r):
    if r.pdf_found: return ""
    if r.pdf_status != "found": return str(r.pdf_status)
    return "identity FAIL" if r.pdf_identity == "FAIL" else "no value"
def h_outcome(r):
    fc = r.final_class
    if fc == "not_covered": return "not covered"
    if fc in H_AGREE: return "ok"
    if fc == "near": return "near (<=1%)"
    k, v = r.key_crore, r.final_value_crore
    if pd.isna(k) or k == 0 or pd.isna(v): return "large gap"
    ratio = v / k
    for fac, nm in ((10, "unit-scale x10"), (0.1, "unit-scale /10"), (0.01, "unit-scale /100"), (0.001, "unit-scale /1000"), (100, "unit-scale x100"), (1000, "unit-scale x1000")):
        if abs(ratio - fac) <= 0.01 * fac: return nm
    if abs(v + k) <= 0.01 * abs(k): return "sign flipped"
    return "gap 1-5%" if abs(v - k) / abs(k) <= 0.05 else "large gap"
INT["outcome"] = INT.apply(h_outcome, axis=1)
INT["reason"] = [("pdf: " + (h_pdf_reason(r) or "n/a") + "; override: " + str(r.ovr_reason)) if r.final_class == "not_covered"
                 else ("" if r.final_class in H_AGREE else r.outcome) for r in INT.itertuples()]
_oo = ["ok", "near (<=1%)", "unit-scale x10", "unit-scale /10", "unit-scale /100", "unit-scale /1000", "unit-scale x100", "unit-scale x1000",
       "sign flipped", "gap 1-5%", "large gap", "not covered"]
t7 = pd.crosstab(INT.field.map(H_VAR), INT.outcome).reindex(index=[H_VAR[f] for f in H_ORDER]).fillna(0).astype(int)
t7 = t7.reindex(columns=[c for c in _oo if c in t7.columns] + [c for c in t7.columns if c not in _oo])
t7.loc["TOTAL"] = t7.sum(); t7.index.name = "Variable"
TABLES["7_outcome_per_variable"] = t7.reset_index()
_nc = INT[INT.final_class == "not_covered"]
TABLES["7b_not_covered_reasons"] = (_nc.groupby(["field", "reason"]).size().reset_index(name="Cells").assign(field=lambda d: d.field.map(H_VAR)).rename(columns={"field": "Variable", "reason": "Reason"})
                                    if len(_nc) else pd.DataFrame({"Variable": [], "Reason": [], "Cells": []}))
# 8. every cell still not agreeing with the key
bad = INT[~INT.final_class.isin(H_AGREE)]
TABLES["8_cells_not_agreeing"] = bad[["report_id", "field", "key_crore", "final_value_crore", "final_source", "final_class", "outcome", "reason"]].rename(
    columns={"report_id": "Report", "field": "Variable", "key_crore": "Key (crore)", "final_value_crore": "Final value (crore)", "final_source": "Source",
             "final_class": "Class", "outcome": "Outcome", "reason": "Reason"}).reset_index(drop=True)
INT.drop(columns=["_ag"]).to_csv(OUT_DIR / f"hybrid_{HYBRID}_integration_199_final.csv", index=False)

# 5b. decision reasons and 9. every EBIT cell
rows = []
for dsc, d in INT.groupby("decision"):
    ag = int(d.final_class.isin(H_AGREE).sum()); rows.append({"Decision": dsc, "Cells": len(d), "Agreeing": ag, "Agreement %": pc(ag, len(d))})
TABLES["5b_decision_reasons"] = pd.DataFrame(rows).sort_values("Cells", ascending=False).reset_index(drop=True)
_e = INT[INT.field == "ebit"].copy()
TABLES["9_ebit_cells"] = _e[["report_id", "key_crore", "pdf_value_crore", "pdf_class", "llp_value_crore", "llp_class", "llp_rows", "ovr_value_crore", "ovr_class",
                             "final_value_crore", "final_source", "final_class", "decision"]].sort_values(["final_class", "report_id"]).reset_index(drop=True)

if "8_cells_not_agreeing" in TABLES:
    TABLES["8_cells_not_agreeing"] = TABLES["8_cells_not_agreeing"].merge(S[["report_id", "field", "matched_label", "detail"]].rename(columns={"report_id": "Report", "field": "Variable", "matched_label": "P matched label", "detail": "P detail"}), on=["Report", "Variable"], how="left")
# ---- (i) cells where the final value differs from Hybrid A (old parser) ----
_RK = {"exact": 0, "within_tolerance": 1, "near": 2, "mismatch": 3, "not_covered": 4}
def _vdiff(a, b):
    if pd.isna(a) and pd.isna(b): return False
    if pd.isna(a) or pd.isna(b): return True
    return abs(a - b) > 1e-9 * max(1.0, abs(a), abs(b))
_d = INT[[ _vdiff(a, b) or ca != cb for a, b, ca, cb in zip(INT.final_value_crore, INT.hA_value_crore, INT.final_class, INT.hA_class)]].copy()
_d["verdict"] = ["better" if _RK[c] < _RK[h] else ("worse" if _RK[c] > _RK[h] else "equal (different value, same class)") for c, h in zip(_d.final_class, _d.hA_class)]
TABLES["10_final_vs_hybridA_cells"] = _d[["report_id", "field", "key_crore", "hA_value_crore", "hA_class", "final_value_crore", "final_class", "final_source", "decision", "verdict"]].rename(
    columns={"report_id": "Report", "field": "Variable", "key_crore": "Key (crore)", "hA_value_crore": "Hybrid A value", "hA_class": "Hybrid A class", "final_value_crore": "v4 value", "final_class": "v4 class", "final_source": "v4 source", "decision": "Decision", "verdict": "Verdict"}).sort_values(["Verdict", "Report"]).reset_index(drop=True)
TABLES["10b_final_vs_hybridA_summary"] = pd.DataFrame({"Verdict": ["better", "worse", "equal (different value, same class)"],
    "Cells": [int((_d.verdict == v).sum()) for v in ["better", "worse", "equal (different value, same class)"]]}).assign(**{"Cells where v4 = Hybrid A": len(INT) - len(_d)})

# ---- (ii) where Llama beats the rules (L right and P wrong) and the reverse, per variable ----
_rows = []
for f in H_ORDER + ["ALL"]:
    d = INT if f == "ALL" else INT[INT.field == f]
    lr, pr = d.llp_has & d.llp_class.isin(H_AGREE), d.pdf_found & d.pdf_class.isin(H_AGREE)
    _rows.append({"Variable": "All variables" if f == "ALL" else H_VAR[f],
                  "L right, P wrong": int((lr & d.pdf_found & ~pr).sum()), "L right, P not covered": int((lr & ~d.pdf_found).sum()),
                  "P right, L wrong": int((pr & d.llp_has & ~lr).sum()), "P right, L not covered": int((pr & ~d.llp_has).sum()),
                  "both right": int((lr & pr).sum()), "both wrong": int((d.llp_has & d.pdf_found & ~lr & ~pr).sum())})
TABLES["11_llama_vs_rules_per_variable"] = pd.DataFrame(_rows)
_x = INT[(INT.llp_has & INT.llp_class.isin(H_AGREE) & ~(INT.pdf_found & INT.pdf_class.isin(H_AGREE))) | (INT.pdf_found & INT.pdf_class.isin(H_AGREE) & INT.llp_has & ~INT.llp_class.isin(H_AGREE))].copy()
_x["who_is_right"] = np.where(_x.llp_class.isin(H_AGREE), "Llama right, rules wrong/missing", "Rules right, Llama wrong")
TABLES["11b_llama_vs_rules_cells"] = _x[["report_id", "field", "key_crore", "pdf_value_crore", "pdf_class", "pdf_status", "llp_value_crore", "llp_class", "llp_rows", "who_is_right"]].sort_values(["who_is_right", "report_id"]).reset_index(drop=True)

# ---- (iii) REVIEW list: P and L disagreed, O did not settle it, P row not flagged -> P kept, a human should look ----
_rv = INT[INT.review]
TABLES["12_REVIEW_list"] = (_rv[["report_id", "field", "pdf_value_crore", "llp_value_crore", "ovr_value_crore", "ovr_source", "final_value_crore", "final_source", "llp_rows", "matched_label" if "matched_label" in _rv.columns else "decision"]]
                            .rename(columns={"pdf_value_crore": "P (crore)", "llp_value_crore": "L (crore)", "ovr_value_crore": "O (crore)", "final_value_crore": "Used (crore)"}).reset_index(drop=True)
                            if len(_rv) else pd.DataFrame({"note": ["REVIEW list is empty"]}))

# ---- (iv) HDFC page check: is the saved .txt page the consolidated or the STANDALONE statement? ----
def hdfc_page_check():
    import ast
    rows_, eq_ = [], []
    for rid in [r for r in REPORT_IDS if COMPANY[r] == "HDFC"]:
        sel = PDFSEL[PDFSEL.report_id == rid].iloc[0]
        for kind in ("bs", "pl"):
            if (rid, kind) not in TXT_FILE:
                rows_.append({"report": rid, "stmt": kind, "page": "-", "note": "no statement saved"})
                try:                                           # why was it not saved? list pages headed as a consolidated statement of this kind and the page finder's verdict
                    doc = fitz.open(str(PDF_PATH[rid]))
                    for i_ in range(len(doc)):
                        t_ = "\n".join(page_rows(doc[i_])); hd_ = squash_spaced_caps(t_[:HEAD_CHARS])
                        if "consolidated" in hd_.lower() and (HEAD_BS if kind == "bs" else HEAD_PL).search(hd_):
                            sc_, info_ = score_page(t_, kind)
                            rows_.append({"report": rid, "stmt": kind, "page": i_ + 1, "role": "candidate by heading (NOT saved)", "heading (first rows)": " / ".join(t_.split("\n")[:4])[:150],
                                          "basis verdict": f"score {sc_:.1f}" if sc_ is not None else f"refused by page finder: {info_}"})
                    doc.close()
                except Exception as e:
                    rows_.append({"report": rid, "stmt": kind, "page": "-", "role": "could not open the PDF", "heading (first rows)": f"{type(e).__name__}"})
                continue
            cur, pgs = None, {}
            for ln in Path(TXT_FILE[(rid, kind)]).read_text(encoding="utf-8").splitlines():
                if ln.startswith("# ----- page "): cur = int(re.search(r"page (\d+)", ln).group(1)); pgs[cur] = []
                elif cur is not None and not ln.startswith("# ") and ln.strip(): pgs[cur].append(ln.strip())
            for pg, ls in pgs.items():
                head = " / ".join(ls[:4])
                nc, ns = basis_counts("\n".join(ls)[:HEAD_CHARS])
                rows_.append({"report": rid, "stmt": kind, "page": pg, "role": "SAVED page (parsed)", "heading (first rows)": head[:150], "consolidated words": nc, "standalone words": ns,
                              "basis verdict": "STANDALONE" if ns > nc else ("consolidated" if nc > ns else "unclear (equal counts)"), "page-finder top3 (score,page)": str(sel[f"{kind}_top3(score,page)"]), "via neighbour fallback": bool(sel[f"{kind}_via_neighbour_fallback"])})
            try:   top = ast.literal_eval(str(sel[f"{kind}_top3(score,page)"]))
            except Exception: top = []
            try:
                doc = fitz.open(str(PDF_PATH[rid]))
                for sc, pg in top:
                    if pg in pgs: continue
                    hs = " / ".join(page_rows(doc[pg - 1])[:4]); nc, ns = basis_counts("\n".join(page_rows(doc[pg - 1]))[:HEAD_CHARS])
                    rows_.append({"report": rid, "stmt": kind, "page": pg, "role": f"other candidate (score {sc})", "heading (first rows)": hs[:150], "consolidated words": nc, "standalone words": ns,
                                  "basis verdict": "STANDALONE" if ns > nc else ("consolidated" if nc > ns else "unclear (equal counts)")})
                doc.close()
            except Exception as e:
                rows_.append({"report": rid, "stmt": kind, "page": "-", "role": "other candidates not opened", "heading (first rows)": f"{type(e).__name__}"})
        # equity composition on the saved balance sheet (printed units; DIAGNOSTIC, the key column is scoring-only)
        if (rid, "bs") in TXT_FILE:
            rr = read_statement_file(TXT_FILE[(rid, "bs")])
            comp = {}
            for nm, cond in (("capital", lambda c: c == "capital"), ("reserves and surplus", lambda c: c.startswith("reserves and surplus")), ("minority interest", lambda c: c.startswith("minority interest") or c.startswith("non controlling")), ("employees stock options", lambda c: c.startswith("employees stock option"))):
                x = next((r_ for r_ in rr if cond(r_["core"]) and r_["vals"] and isinstance(r_["vals"][0], float)), None)
                comp[nm] = x["vals"][0] if x else None
            have = [v for k_, v in comp.items() if v is not None and k_ != "employees stock options"]
            unit = UNIT_BY[(rid, "bs")][0]; f_ = UNIT_TO_CRORE.get(unit, np.nan)
            cell = INT[(INT.report_id == rid) & (INT.field == "total_equity")].iloc[0]
            eq_.append({"report": rid, "capital": comp["capital"], "reserves and surplus": comp["reserves and surplus"], "minority interest": comp["minority interest"], "employees stock options (printed, NOT added)": comp["employees stock options"],
                        "sum printed -> crore": sum(have) * f_ if have else np.nan, "capital+reserves only (crore)": ((comp["capital"] or 0) + (comp["reserves and surplus"] or 0)) * f_,
                        "P total_equity (crore)": cell.pdf_value_crore, "L total_equity (crore)": cell.llp_value_crore, "key (crore, SCORING ONLY)": cell.key_crore, "key - sum (crore, SCORING ONLY)": cell.key_crore - (sum(have) * f_ if have else np.nan)})
    return pd.DataFrame(rows_), pd.DataFrame(eq_)
_hp, _he = hdfc_page_check()
TABLES["13_hdfc_page_check"] = _hp if len(_hp) else pd.DataFrame({"note": ["no HDFC report"]})
TABLES["13b_hdfc_equity_components"] = _he if len(_he) else pd.DataFrame({"note": ["no HDFC balance sheet"]})

# 10. CSV + Notion block
def h_fmt(x):
    if isinstance(x, float): return "" if np.isnan(x) else f"{x:,.2f}"
    return str(x)
def h_md(df):
    cols = [str(c) for c in df.columns]
    L = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for r_ in df.itertuples(index=False):
        L.append("| " + " | ".join(h_fmt(x).replace("|", "/") for x in r_) + " |")
    return "\n".join(L)
for k, t in TABLES.items():
    t.to_csv(H_DIR / f"{HYBRID}_{k}.csv", index=False)
    print("\n" + k); display(t)
print("\nCSV files written to", H_DIR)

# reproduction check against the earlier run (never tuned): Override, PDF and Hybrid A must match; Llama rows have no reference
REF = {("All cells", "Override v1.1"): (85.4, 91.0), ("All cells", "PDF route v1.3 (old parser)"): (80.4, 83.4), ("All cells", "Hybrid A (old parser)"): (90.5, 94.5),
       ("Excluding HDFC", "Override v1.1"): (90.9, 96.3), ("Excluding HDFC", "PDF route v1.3 (old parser)"): (83.4, 86.6), ("Excluding HDFC", "Hybrid A (old parser)"): (94.1, 97.9)}
t1 = TABLES["1_headline"]; chk = []
for r_ in t1.itertuples(index=False):
    ref = REF.get((r_[0], r_[1]))
    if ref: chk.append({"Scope": r_[0], "Method": r_[1], "tol % (this run)": r_[9], "tol % (reference)": ref[0], "within 1% (this run)": r_[10], "within 1% (reference)": ref[1],
                        "exact cells (this run)": r_[3], "match": abs(r_[9] - ref[0]) < 0.06 and abs(r_[10] - ref[1]) < 0.06})
CHK = pd.DataFrame(chk); TABLES["0_reference_check"] = CHK
print("\nReproduction check of the v2 numbers with the OLD parser (reference only, nothing tuned; Hybrid A all cells should show 178 exact cells):"); display(CHK)
if len(CHK) and not CHK.match.all():
    print("NOTE: some numbers differ from the reference run. Likely causes: changed website data or cache, a different PDF-route version, or the identity filter in the PDF-found test. No rule was adjusted to match.")
CHK.to_csv(H_DIR / f"{HYBRID}_0_reference_check.csv", index=False)

NOTION = "\n".join("> " + b for b in BANNER) + "\n\n" + "\n\n".join(f"### {k}\n\n{h_md(t)}" for k, t in TABLES.items())
(H_DIR / f"{HYBRID}_notion_block.md").write_text(NOTION, encoding="utf-8")
print("\nRECOMMENDED FINAL:", RECOMMENDED)
print("\n==== NOTION BLOCK (copy everything inside the box) ====")
print("```markdown\n" + NOTION + "\n```")

In [ ]:
# ===== FINAL CELL | download the 199 scored values of the final v4 run =====
import pandas as pd
from pathlib import Path

CHECK_COLS = ["report_id", "company", "year", "field",
              "key_crore",                    # ground-truth value from the annual report
              "final_value_crore",            # value the v4 pipeline chose
              "final_class",                  # exact / within_tolerance / near / mismatch / not_covered
              "final_source", "decision", "review",
              "pdf_value_crore", "pdf_class",    # PDF route
              "llp_value_crore", "llp_class",    # Llama extractor
              "ovr_value_crore", "ovr_source", "ovr_class"]   # web override

CHECK = INT[[c for c in CHECK_COLS if c in INT.columns]].copy()
CHECK["gap_crore"] = (CHECK["final_value_crore"] - CHECK["key_crore"]).round(4)
CHECK["gap_pct"]   = (100 * CHECK["gap_crore"] / CHECK["key_crore"].abs()).round(4)
CHECK = CHECK.sort_values(["company", "year", "field"]).reset_index(drop=True)

assert len(CHECK) == 199, f"expected 199 scored cells, got {len(CHECK)}"
print("rows:", len(CHECK))
print(CHECK["final_class"].value_counts().to_dict())          # expect 192 exact, 3 near, 4 mismatch
print("matched (exact + within_tolerance):", int(CHECK["final_class"].isin(["exact", "within_tolerance"]).sum()), "of", len(CHECK))

out = Path(OUT_DIR) / "accuracy_check_199.csv"
CHECK.to_csv(out, index=False)
try:
    from google.colab import files
    files.download(str(out))
except Exception:
    print("Saved to:", out)

In [ ]:
# ===== ACCURACY SUMMARY | from the 199 scored values (final v4) =====
import pandas as pd
from pathlib import Path

MATCH = ("exact", "within_tolerance")
CLASSES = ["exact", "within_tolerance", "near", "mismatch", "not_covered"]

def summarize(df):
    c = df["final_class"].value_counts()
    n = len(df)
    row = {k: int(c.get(k, 0)) for k in CLASSES}
    row["cells"] = n
    row["matched"] = row["exact"] + row["within_tolerance"]
    row["exact_%"] = round(100 * row["exact"] / n, 2) if n else None
    row["matched_%"] = round(100 * row["matched"] / n, 2) if n else None
    row["not_matched"] = n - row["matched"]          # near + mismatch + not_covered
    return pd.Series(row)

ORDER = ["cells", "exact", "within_tolerance", "matched", "not_matched", "near", "mismatch", "not_covered", "exact_%", "matched_%"]

# 1. headline: all cells and excluding HDFC
head = pd.DataFrame({
    "All 199 cells": summarize(CHECK),
    "Excluding HDFC": summarize(CHECK[CHECK.company != "HDFC"]),
    "HDFC only": summarize(CHECK[CHECK.company == "HDFC"]),
}).T[ORDER]
print("HEADLINE\n", head.to_string(), "\n")

# 2. by variable
by_var = CHECK.groupby("field").apply(summarize)[ORDER]
print("BY VARIABLE\n", by_var.to_string(), "\n")

# 3. by company
by_co = CHECK.groupby("company").apply(summarize)[ORDER]
print("BY COMPANY\n", by_co.to_string(), "\n")

# 4. every cell that is not exact / within tolerance
bad = CHECK[~CHECK.final_class.isin(MATCH)][["report_id", "company", "year", "field", "key_crore",
                                             "final_value_crore", "gap_pct", "final_class", "final_source", "decision"]]
print("CELLS NOT MATCHED (", len(bad), ")\n", bad.to_string(index=False), "\n")

# 5. checks against the numbers in ACCURACY.md (Table 1)
exp = {"cells": 199, "exact": 192, "matched": 192}
got = {k: int(head.loc["All 199 cells", k]) for k in exp}
print("Expected (ACCURACY.md Table 1):", exp)
print("This run                      :", got)
print("MATCHES ACCURACY.md" if got == exp else "*** DIFFERS from ACCURACY.md: do not publish these numbers until checked ***")

# 6. save and download as one CSV
OUT = Path(OUT_DIR)
tables = [("headline", head.reset_index().rename(columns={"index": "scope"})),
          ("by_variable", by_var.reset_index()), ("by_company", by_co.reset_index())]
parts = []
for name, t in tables:
    t = t.copy(); t.insert(0, "table", name); parts.append(t)
summary = pd.concat(parts, ignore_index=True)
f = OUT / "accuracy_summary_199.csv"
summary.to_csv(f, index=False)
try:
    from google.colab import files
    files.download(str(f))
except Exception:
    print("Saved to:", f)